# DeepVoice v4.1 — 로컬 노트북 결과 보고 수정본
**원본:** `DeepVoice_v4_HeadSpecialist_25K_OOD_Submit_Local.ipynb` (64셀).

원래 네 소스 풀(Kaggle real/fake voice, FMA real music, SONICS fake music), 25,000개 원본 선택,
Presence CNN / AASIST Aux3 / Voice WavLM / 16초 Music Transformer 구조를 유지합니다.
성능 개선을 보장하는 새 모델이 아니라 **실행·검증·패키징 연결을 바로잡은 버전**입니다.

## 먼저 읽기
- VS Code Jupyter / 로컬 Windows 기준입니다. **기존 셀과 섞지 말고 이 파일을 새 커널에서 위부터 실행**하세요.
- 설정 셀의 `PROJECT_ROOT`만 실제 기존 프로젝트 폴더로 지정하세요. 다운로드한 파일과 PANNs/MFCC 캐시는 재사용합니다.
- 기존 학습 결과를 덮어쓰지 않도록 `runs_fixed_v4_1`, `manifests_fixed_v4_1`, `build_fixed_v4_1`을 사용합니다.
- 기본값은 **데이터 준비 + 합성 입력 사전검사**, `RUN_TRAINING=False`, `RUN_OOD=False`, `BUILD_SUBMIT=False`입니다.
  준비와 모델 self-test 통과 후 설정에서 학습을 켜고 학습 단계부터 실행하세요. 네 branch 완료 후 제출을 켭니다.
- 8GB GPU 보수적 batch를 사용하지만 GPU 메모리 적합성은 실제 PC에서 self-test로 확인해야 합니다.
- 각 단계는 앞 단계가 없을 때 원인을 설명하며 중단합니다. 임의 순서 실행까지 자동 복구하는 노트북은 아닙니다.

## 검증 한계
작성 환경에서는 네트워크 다운로드, Kaggle 인증, 대규모 음원 처리, 실제 CUDA 학습/대회 서버 실행을 하지 않았습니다.
노트북 문법·셀 의존성·합성 데이터 단위검사를 실행한 범위는 함께 제공된 검증 보고서를 보세요.
외부 서버 504/권한, 실제 손상 파일, OOM, 학습 성능까지 '오류 없음'을 보장하지 않습니다.

## Windows 파일 잠금 보완 (v4.1.1)
- WinError 32/33일 때만 최대 60초 재시도합니다. 영구 잠금은 파일을 보존하고 중단합니다.
- 기존 출력/완성된 임시 파일의 크기·CRC를 확인하고 재사용합니다.
- FMA 메타데이터는 이 노트북이 실제 사용하는 `tracks.csv`만 추출합니다. 원본 ZIP과 이미 추출된 다른 CSV는 삭제하지 않습니다.
- 공통 JSON/CSV/체크포인트 저장 및 다운로드 완료 rename에도 같은 재시도를 적용했습니다.
- 프로젝트·데이터·학습 설정과 모델 코드는 이전 수정본과 같습니다. 기존 경로를 변경하지 마세요.
- 실행 전에 데이터 폴더를 대상으로 한 다른 추출 작업을 중단하고 CSV를 열어 둔 프로그램을 닫으세요. OneDrive 동기화는 필요 시 일시 중지합니다.
- Linux 임시 파일 테스트 및 WinError 32 모의 테스트만 수행했습니다. 실제 Windows/OneDrive 잠금을 여기서 재현하지는 못했습니다.

> 이 수정본은 학습/최종 검증/OOD 실행 여부를 분리하고, 이미 저장된 결과를 재학습 없이 읽는 마지막 결과 대시보드를 포함합니다.


## 0. 패키지 점검 (작동 중인 PyTorch를 자동 재설치하지 않음)
새 패키지가 필요할 때만 `INSTALL_MISSING=True`로 실행하세요. pip로 설치한 뒤에는 커널을 재시작합니다.
`torch`/`torchaudio`는 CUDA 빌드가 서로 맞는 환경을 별도로 준비하세요. 현재 정상인 CUDA 환경을 유지합니다.

In [1]:
import sys
import importlib.util
import importlib.metadata
import subprocess
INSTALL_MISSING = False
requirements_local = {
    'numpy': 'numpy', 'pandas': 'pandas>=2.0', 'scipy': 'scipy>=1.11',
    'sklearn': 'scikit-learn>=1.4', 'soundfile': 'soundfile>=0.12',
    'librosa': 'librosa>=0.10.2,<0.12', 'tqdm': 'tqdm>=4.66',
    'matplotlib': 'matplotlib', 'seaborn': 'seaborn>=0.13',
    'transformers': 'transformers==4.57.6', 'huggingface_hub': 'huggingface_hub>=0.34,<1',
    'accelerate': 'accelerate>=1.9,<2', 'panns_inference': 'panns-inference==0.1.1',
    'imageio_ffmpeg': 'imageio-ffmpeg>=0.5', 'requests': 'requests>=2.31',
    'IPython': 'ipython>=8.18', 'dotenv': 'python-dotenv>=1.0',
    'kaggle': 'kaggle>=1.7', 'einops': 'einops>=0.8',
}
missing = [value for module,value in requirements_local.items() if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING:
        raise RuntimeError('추가 패키지 필요: '+', '.join(missing)+'\nINSTALL_MISSING=True 실행 후 커널 재시작')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
    raise RuntimeError('패키지 설치 완료. 커널을 재시작하고 INSTALL_MISSING=False로 처음부터 실행하세요.')
if any(importlib.util.find_spec(name) is None for name in ('torch','torchaudio')):
    raise RuntimeError('현재 커널에 호환되는 CUDA torch/torchaudio를 설치하세요. 이 셀은 GPU 패키지를 바꾸지 않습니다.')
print('실행 Python:', sys.executable)
print('의존성 존재 확인 완료. 버전/바이너리 검사는 다음 셀에서 실행합니다.')


실행 Python: c:\Users\shj04\miniconda3\envs\new_env\python.exe
의존성 존재 확인 완료. 버전/바이너리 검사는 다음 셀에서 실행합니다.


In [2]:
from __future__ import annotations

import ast
import gc
import copy
import hashlib
import importlib.util
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import tempfile
import time
import warnings
import zipfile
from pathlib import Path
from types import SimpleNamespace

import imageio_ffmpeg
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from IPython.display import display
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm



# 기존 프로젝트를 재사용하려면 DEEPVOICE_V4_ROOT를 그 프로젝트 폴더로 지정하세요.
_configured_root = os.environ.get("DEEPVOICE_V4_ROOT", "").strip()
_legacy_root = Path(r"C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4")
if _configured_root:
    PROJECT_ROOT = Path(_configured_root).expanduser().resolve()
elif _legacy_root.is_dir():
    # 이 노트북이 원래 실행된 PC에서는 기존 checkpoint/history를 자동 재사용합니다.
    PROJECT_ROOT = _legacy_root.resolve()
else:
    PROJECT_ROOT = (Path.cwd() / "deepvoice_headspecialist_v4").resolve()
PIPELINE_VERSION = "v4.1-local-audit-20260909"
ALLOW_DOWNLOADS = True
RUN_TRAINING = True
RUN_MODEL_SELF_TEST = False
RUN_FINAL_VALIDATION = True
RUN_OOD = False
BUILD_SUBMIT = False
SHOW_SAVED_RESULTS = True   # 재학습 없이 저장된 history/metric을 마지막 셀에서 표시
BUILD_SUBMIT = False         # 네 branch 학습 + fusion 완료 후 True
REQUIRE_L4_TIME_GATE = False # False는 L4 실행 시간을 검증했다는 뜻이 아닙니다
ALLOW_UNVERIFIED_SUBMIT = True
BRANCHES_TO_TRAIN = ["presence", "aasist_aux3", "voice_wavlm", "music_long"]
KAGGLE_DATASET = "jayjoshi37/deepfake-audio-dataset-fake-vs-real-speech"
SONICS_REPO = "awsaf49/sonics"
KAGGLE_LICENSE_NOTE = "CC-BY-SA-4.0 (original notebook claim; verify original dataset terms)"

CFG = SimpleNamespace(
    seed=42,
    sample_rate=16_000,
    # 공식 AASIST 입력 길이(64,600 samples @ 16 kHz)에 맞춘다.
    clip_seconds=4.0375,
    clip_samples=64_600,
    music_clip_seconds=16.0,
    music_clip_samples=256_000,
    source_per_pool=6_250,
    source_split_counts={"train": 5_625, "validation": 625},
    recipe_counts={"train": 22_500, "validation": 2_500},
    panns_seconds=10,
    panns_batch=2,
    # 로컬 Jupyter/Windows에서 notebook 정의 Dataset의 spawn/pickle 오류를 피하는 안전 기본값.
    num_workers=0,
    max_invalid_train_fraction=0.01,
)
assert sum(CFG.source_split_counts.values()) == CFG.source_per_pool
assert sum(CFG.recipe_counts.values()) == 25_000

DACON_PROBABILITY_COLUMNS = [
    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
]
DACON_TRUTH_COLUMNS = [
    "FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT",
]
HEAD_WEIGHTS = torch.tensor([0.45, 0.18, 0.27, 0.05, 0.05], dtype=torch.float32)
AUDIO_SUFFIXES = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".opus", ".wma", ".amr"}

# 기존 셀과의 호환성을 위해 DRIVE_ROOT 이름은 유지하지만 실제 Google Drive가 아니라 로컬 프로젝트 폴더다.
DRIVE_ROOT = PROJECT_ROOT
LOCAL_ROOT = Path(os.environ.get("DEEPVOICE_DATA_ROOT", str(PROJECT_ROOT / "datasets"))).expanduser().resolve()
DATASET_ROOT = LOCAL_ROOT
VOICE_ROOT = LOCAL_ROOT / "voice_kaggle"
FMA_ROOT = LOCAL_ROOT / "fma"
SONICS_ROOT = LOCAL_ROOT / "sonics"
REPO_ROOT = PROJECT_ROOT / "repos"
RUN_ROOT = PROJECT_ROOT / "runs_fixed_v4_1"
CACHE_ROOT = PROJECT_ROOT / "manifests"
MANIFEST_ROOT = PROJECT_ROOT / "manifests_fixed_v4_1"
BUILD_ROOT = PROJECT_ROOT / "build_fixed_v4_1"
CREDENTIAL_ROOT = PROJECT_ROOT / "credentials"
for directory in (
    PROJECT_ROOT, LOCAL_ROOT, VOICE_ROOT, FMA_ROOT, SONICS_ROOT,
    REPO_ROOT, RUN_ROOT, MANIFEST_ROOT, CACHE_ROOT, BUILD_ROOT, CREDENTIAL_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

_system_ffmpeg = shutil.which("ffmpeg")
FFMPEG_EXE = str(Path(_system_ffmpeg).resolve()) if _system_ffmpeg else str(Path(imageio_ffmpeg.get_ffmpeg_exe()).resolve())
if not Path(FFMPEG_EXE).is_file():
    raise FileNotFoundError("FFmpeg 실행 파일을 찾지 못했습니다. ffmpeg를 설치하거나 imageio-ffmpeg를 다시 설치하세요.")
ffmpeg_version = subprocess.run(
    [FFMPEG_EXE, "-version"], capture_output=True, text=True, check=True,
).stdout.splitlines()[0]
ffmpeg_encoders = subprocess.run(
    [FFMPEG_EXE, "-hide_banner", "-encoders"], capture_output=True, text=True, check=True,
).stdout
required_encoders = ("libmp3lame", "aac", "libopus", "flac", "pcm_s16le")
missing_encoders = [name for name in required_encoders if not re.search(rf"\b{re.escape(name)}\b", ffmpeg_encoders)]
if missing_encoders:
    raise RuntimeError(f"현재 FFmpeg에 필요한 encoder가 없습니다: {missing_encoders}")

# torch/torchaudio binary 조합이 맞지 않으면 여기에서 빠르게 실패시킨다.
_audio_probe = torch.zeros(CFG.sample_rate, dtype=torch.float32)
_resample_probe = torchaudio.functional.resample(_audio_probe, CFG.sample_rate, 8_000)
if _resample_probe.numel() != 8_000 or not torch.isfinite(_resample_probe).all():
    raise RuntimeError("torch/torchaudio resampling self-check failed")
del _audio_probe, _resample_probe

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_write_probe = PROJECT_ROOT / ".write_test"
_write_probe.write_text("ok", encoding="utf-8")
_write_probe.unlink()
free_gib = shutil.disk_usage(PROJECT_ROOT).free / 1024**3

print("project root:", PROJECT_ROOT)
print("platform:", platform.platform())
print("python:", sys.version.split()[0])
print("device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "torch:", torch.__version__)
else:
    warnings.warn("CUDA GPU를 찾지 못했습니다. 코드 검사는 가능하지만 전체 학습은 현실적으로 매우 오래 걸립니다.")
print("ffmpeg:", ffmpeg_version)
print("DataLoader workers:", CFG.num_workers)
print("source references:", f"{4 * CFG.source_per_pool:,}")
print("dynamic recipes:", f"{sum(CFG.recipe_counts.values()):,}")
print("music long context seconds:", CFG.music_clip_seconds)
print("free disk GB:", round(free_gib, 1))
if free_gib < 110:
    warnings.warn(
        "FMA 원본·SONICS·독립 OOD 압축 및 해제본을 함께 보관하려면 공간이 부족할 수 있습니다. "
        "PROJECT_ROOT가 위치한 드라이브에 최소 110GiB 여유 공간을 권장합니다."
    )

# 공통 실행 설정 (후속 셀에서 플래그를 임의로 덮어쓰지 않음)
DEVICE_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30 if DEVICE.type == "cuda" else 0.0
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.float16
RESUME_TRAINING = True
INFERENCE_BATCHES = {"presence": 4, "aasist_aux3": 2, "voice_wavlm": 1, "music_long": 1}
OFFLOAD_INFERENCE_MODELS = True
# OOMはCPU RAMとは別。最小GPU演算の動作を確認。
if DEVICE.type == "cuda":
    with torch.no_grad():
        probe = torch.randn(32,32,device=DEVICE); _ = probe @ probe.T
        torch.cuda.synchronize()
    del probe, _
print('저장 경로(기존 runs 보존):', RUN_ROOT)
print('실행 단계:', {'train': RUN_TRAINING, 'final_validation': RUN_FINAL_VALIDATION, 'ood': RUN_OOD, 'submit': BUILD_SUBMIT, 'show_saved_results': SHOW_SAVED_RESULTS})


project root: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4
platform: Windows-11-10.0.26200-SP0
python: 3.12.14
device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU torch: 2.14.0+cu130
ffmpeg: ffmpeg version 7.1-essentials_build-www.gyan.dev Copyright (c) 2000-2024 the FFmpeg developers
DataLoader workers: 0
source references: 25,000
dynamic recipes: 25,000
music long context seconds: 16.0
free disk GB: 734.8
저장 경로(기존 runs 보존): C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1
실행 단계: {'train': True, 'final_validation': True, 'ood': False, 'submit': False, 'show_saved_results': True}


In [3]:
branch_names = [
    "presence",
    "aasist_aux3",
    "voice_wavlm",
    "music_long",
]

for candidate_root in sorted(PROJECT_ROOT.glob("runs*")):
    if not candidate_root.is_dir():
        continue

    print("\nRUN 폴더:", candidate_root)

    for name in branch_names:
        print(
            name,
            "best.pt =", (candidate_root / name / "best.pt").exists(),
            "history.csv =", (candidate_root / name / "history.csv").exists(),
        )

    print(
        "Clean/Stress metrics =",
        (candidate_root / "validation_file_metrics.csv").exists(),
    )
    print(
        "OOD metrics =",
        (candidate_root / "ood2500" / "official_metrics.json").exists(),
    )


RUN 폴더: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs
presence best.pt = False history.csv = False
aasist_aux3 best.pt = False history.csv = False
voice_wavlm best.pt = False history.csv = False
music_long best.pt = False history.csv = False
Clean/Stress metrics = False
OOD metrics = False

RUN 폴더: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1
presence best.pt = True history.csv = True
aasist_aux3 best.pt = True history.csv = True
voice_wavlm best.pt = True history.csv = True
music_long best.pt = True history.csv = True
Clean/Stress metrics = True
OOD metrics = False


## 1. 다운로드/압축 해제 공통 함수
원본에서 정의가 사라졌던 `download_http`, `safe_extract_zip`을 **사용 전에 정의**합니다.
큰 파일은 curl → 디스크 스트리밍, 부분 파일과 URL 기록 보존, 해시 검사, ZIP 파일별 CRC 검사를 사용합니다.
원본 파일/모델은 지우지 않습니다. 손상된 파일은 자동 성공 처리하지 않고 이유를 출력합니다.
Windows 잠금 재시도와 검증된 임시 파일 복구를 포함합니다. 무기한 대기하거나 잠긴 파일을 강제 삭제하지 않습니다.


In [4]:
# ZIP/Windows sharing-violation hotfix. No downloads or training on execution.
from pathlib import Path
import os
import shutil
import stat
import tempfile
import time
import zipfile
import zlib


def _retry_file_lock(action, label, max_wait=60.0):
    """Retry only Windows sharing/lock violations (32/33), not all I/O errors."""
    deadline = time.monotonic() + max_wait
    delay = 1.0
    announced = False
    while True:
        try:
            return action()
        except OSError as exc:
            if getattr(exc, 'winerror', None) not in (32, 33):
                raise
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                raise PermissionError(
                    f'파일 잠금이 {max_wait:g}초 내에 해제되지 않았습니다: {label}\n'
                    '기존 파일/임시 파일은 삭제하지 않았습니다. CSV를 연 프로그램과 '
                    '다른 실행 중인 노트북, OneDrive 동기화 상태를 확인한 뒤 다시 실행하세요.'
                ) from exc
            if not announced:
                print(f'파일 잠금 대기 (최대 {max_wait:g}초): {label}', flush=True)
                announced = True
            time.sleep(min(delay, remaining))
            delay = min(delay * 1.5, 5.0)


def _replace_with_retry(source, destination, max_wait=60.0):
    source, destination = Path(source), Path(destination)
    return _retry_file_lock(
        lambda: source.replace(destination),
        f'{source} -> {destination}',
        max_wait=max_wait,
    )


def _crc32_file(path):
    crc = 0
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            crc = zlib.crc32(block, crc)
    return crc & 0xffffffff


def safe_extract_zip(archive_path, destination, *, members=None):
    """Reuse size+CRC verified output/staging files; retry WinError 32/33.

    members=None extracts all regular entries; otherwise pass exact member names.
    Unneeded entries are not extracted. Original ZIPs are never deleted.
    Do not run simultaneous extractions into the same destination.
    """
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    if isinstance(members, str):
        raise TypeError('members에는 문자열 하나가 아닌 파일명 리스트/집합을 전달하세요.')
    wanted = None if members is None else set(members)
    seen, found, jobs = set(), set(), []
    reused = 0

    def matches(path, info):
        return _retry_file_lock(
            lambda: path.is_file() and path.stat().st_size == info.file_size
            and _crc32_file(path) == info.CRC,
            str(path),
        )

    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            normalized = info.filename.replace('\\', '/')
            parts = normalized.split('/')
            if info.is_dir() or '__MACOSX' in parts:
                continue
            if (normalized.startswith('/') or '..' in parts or ':' in normalized
                    or stat.S_ISLNK(info.external_attr >> 16)):
                raise ValueError(f'unsafe ZIP member: {info.filename}')
            target = (destination / normalized).resolve()
            if destination not in target.parents:
                raise ValueError(f'unsafe ZIP member: {info.filename}')
            key = os.path.normcase(str(target))
            if key in seen:
                raise ValueError(f'중복 ZIP 경로: {info.filename}')
            seen.add(key)
            if wanted is not None and normalized not in wanted:
                continue
            found.add(normalized)
            if matches(target, info):
                reused += 1
                continue
            # Resume a previously completed .extracting file only after CRC verification.
            legacy = target.with_name(target.name + '.extracting')
            staging = None
            candidates = [legacy]
            if target.parent.is_dir():
                candidates.extend(sorted(target.parent.glob(target.name + '.*.extracting')))
            for candidate in candidates:
                if matches(candidate, info):
                    staging = candidate
                    break
            jobs.append((info, target, staging))

        if wanted is not None and wanted - found:
            raise FileNotFoundError(f'ZIP 안에 요청 파일이 없습니다: {sorted(wanted - found)}')
        additional = sum(info.file_size for info, _, staging in jobs if staging is None)
        margin = min(1024**3, max(additional, 1024**2))
        if additional and additional + margin > shutil.disk_usage(destination).free:
            raise RuntimeError(f'압축 해제 공간 부족: 추가 {additional / 2**30:.2f} GiB + 여유 공간 필요')
        print(f'{Path(archive_path).name}: 기존 정상 파일 {reused:,}개 / 처리 {len(jobs):,}개', flush=True)
        try:
            from tqdm import tqdm
        except ImportError:
            def tqdm(values, **kwargs):
                return values
        for info, target, staging in tqdm(jobs, desc=f'Extract {Path(archive_path).name}', dynamic_ncols=True):
            target.parent.mkdir(parents=True, exist_ok=True)
            if staging is None:
                # Unique same-directory temp file: do not truncate a locked old .extracting.
                fd, tmpname = tempfile.mkstemp(prefix=target.name + '.', suffix='.extracting', dir=target.parent)
                staging = Path(tmpname)
                # Both handles close BEFORE rename. archive.open checks CRC on EOF.
                with os.fdopen(fd, 'wb') as dst:
                    with archive.open(info, 'r') as src:
                        shutil.copyfileobj(src, dst, length=1024 * 1024)
                if staging.stat().st_size != info.file_size:
                    raise RuntimeError(f'추출 크기 오류: {staging} (보존됨)')
            _replace_with_retry(staging, target)
    return len(jobs)


# 공유 유틸리티: 이 셀은 네트워크 접속/학습을 실행하지 않습니다.
import csv
import io
import stat
import zlib
from contextlib import contextmanager


def require_names(*names):
    missing = [name for name in names if name not in globals()]
    if missing:
        raise RuntimeError('선행 설정/데이터 셀을 실행하세요: ' + ', '.join(missing))


def file_hash(path, algorithm='sha256', block=1024 * 1024):
    digest = hashlib.new(algorithm)
    with Path(path).open('rb') as stream:
        for data in iter(lambda: stream.read(block), b''):
            digest.update(data)
    return digest.hexdigest()


def atomic_json(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.writing')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    _replace_with_retry(tmp, path)


def atomic_csv(frame, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.writing')
    frame.to_csv(tmp, index=False, encoding='utf-8')
    _replace_with_retry(tmp, path)


def atomic_torch_save(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.writing')
    torch.save(value, tmp); _replace_with_retry(tmp, path)


def tail_text(path, length=5000):
    path = Path(path)
    if not path.exists(): return ''
    with path.open('rb') as stream:
        stream.seek(max(0, path.stat().st_size - length))
        return stream.read(length).decode('utf-8', errors='replace')


def run_to_log(command, log_path, watched_file=None):
    """Python에 응답 본문을 쌓지 않고 실행. 중단 시 자식 프로세스도 종료."""
    log_path = Path(log_path); log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('wb') as log:
        process = subprocess.Popen(list(map(str, command)), stdin=subprocess.DEVNULL,
                                   stdout=log, stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    returncode = process.wait(timeout=20); break
                except subprocess.TimeoutExpired:
                    if watched_file is not None and Path(watched_file).exists():
                        print(f'  저장됨: {Path(watched_file).stat().st_size / 2**20:.1f} MiB', flush=True)
                    else:
                        print(f'  실행 중. 로그: {log_path.name}', flush=True)
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if returncode:
        raise RuntimeError(f'외부 명령 실패 (exit={returncode}).\n{tail_text(log_path)}\n로그: {log_path}')
    return log_path


def download_http(url, destination, expected_hash=None, algorithm='sha256', mirrors=()):
    """curl -> .part -> 해시 확인 -> 최종 이름. URL별 부분 파일 분리."""
    destination = Path(destination); destination.parent.mkdir(parents=True, exist_ok=True)
    def verified(path):
        return path.is_file() and path.stat().st_size > 0 and (
            expected_hash is None or file_hash(path, algorithm) == expected_hash.lower())
    if destination.exists():
        if not verified(destination):
            raise RuntimeError(f'기존 파일이 비어 있거나 해시 불일치: {destination}\n자동 삭제하지 않습니다. 별도 보관 후 재시도하세요.')
        return destination
    if not ALLOW_DOWNLOADS:
        raise FileNotFoundError(f'다운로드 비활성 상태입니다. 필요한 파일: {destination}')
    curl = shutil.which('curl.exe') or shutil.which('curl')
    if not curl: raise FileNotFoundError('curl 실행 파일이 필요합니다. 터미널에서 curl.exe --version 확인')
    errors = []
    for source_index, source_url in enumerate((url, *mirrors)):
        suffix = '.part' if source_index == 0 else '.' + hashlib.sha256(source_url.encode()).hexdigest()[:10] + '.part'
        partial = destination.with_name(destination.name + suffix)
        source_marker = partial.with_name(partial.name + '.url.json')
        if source_marker.exists():
            stored_url = json.loads(source_marker.read_text(encoding='utf-8')).get('url')
            if stored_url != source_url:
                errors.append(f'다른 URL의 부분 파일입니다: {partial}'); continue
        elif partial.exists() and expected_hash is None:
            errors.append(f'출처를 확인할 수 없는 부분 파일: {partial} (보존됨)'); continue
        atomic_json({'url': source_url, 'expected_hash': expected_hash, 'algorithm': algorithm}, source_marker)
        if expected_hash and verified(partial):
            _replace_with_retry(partial, destination); return destination
        log = destination.with_name(destination.name + f'.curl{source_index}.log')
        print('Download:', destination.name, f'({source_index + 1}/{1 + len(mirrors)})', flush=True)
        try:
            run_to_log([
                curl, '--disable', '--location', '--fail', '--silent', '--show-error',
                '--continue-at', '-', '--header', 'Accept-Encoding: identity',
                '--connect-timeout', '30', '--speed-limit', '1024', '--speed-time', '120',
                '--retry', '3', '--retry-delay', '10', '--output', str(partial), source_url,
            ], log, partial)
            if not verified(partial):
                raise RuntimeError(f'다운로드 해시 불일치 또는 빈 파일: {partial}')
            _replace_with_retry(partial, destination)
            return destination
        except RuntimeError as exc:
            errors.append(str(exc))
            print('해당 경로 실패. 부분 파일 보존.', flush=True)
    raise RuntimeError('다운로드를 완료하지 못했습니다.\n' + '\n\n'.join(errors))






def audio_files(root):
    return sorted(p for p in Path(root).rglob('*') if p.is_file()
                  and p.suffix.lower() in AUDIO_SUFFIXES and '__MACOSX' not in p.parts)


def as_boolean(series):
    if series.dtype == bool: return series
    normalized = series.astype(str).str.strip().str.lower()
    known = normalized.isin({'true','false','1','0','1.0','0.0','yes','no'})
    if not known.all(): raise ValueError(f'알 수 없는 boolean 값: {normalized[~known].unique()[:5]}')
    return normalized.isin({'true','1','1.0','yes'})


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed % 2**32); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


@contextmanager
def numpy_seed(seed):
    state = np.random.get_state()
    np.random.seed(seed % 2**32)
    try: yield
    finally: np.random.set_state(state)


def decode_audio_rate(path, sample_rate=16000, seconds=None):
    """SF -> FFmpeg. 빈 오디오/NaN을 묵묵히 유효 샘플로 바꾸지 않음. SF 채널 평균 후 resample."""
    path = Path(path)
    if not path.is_file(): raise FileNotFoundError(path)
    try:
        info = sf.info(path)
        frames = -1 if seconds is None else int(math.ceil(seconds * info.samplerate))
        values, sr = sf.read(path, frames=frames, dtype='float32', always_2d=True)
        waveform = torch.from_numpy(values.mean(axis=1).copy())
        if sr != sample_rate:
            waveform = torchaudio.functional.resample(waveform, int(sr), int(sample_rate))
    except (RuntimeError, OSError, ValueError):
        command = [FFMPEG_EXE, '-hide_banner', '-loglevel', 'error', '-i', str(path)]
        if seconds is not None: command += ['-t', str(float(seconds))]
        command += ['-ac', '1', '-ar', str(sample_rate), '-f', 'f32le', 'pipe:1']
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, timeout=120)
        waveform = torch.from_numpy(np.frombuffer(result.stdout, dtype='<f4').copy())
    if waveform.numel() == 0: raise ValueError(f'empty audio: {path}')
    if not torch.isfinite(waveform).all(): raise ValueError(f'NaN/Inf audio: {path}')
    return waveform.float().clamp(-1, 1)


## 2. Kaggle Voice 재사용/준비 + FMA 다운로드·압축 해제
`voice_audio`/`fma_audio`를 같은 셀에서 새로 생성합니다. FMA ZIP과 metadata ZIP은 서로 다릅니다.
이미 받은 `fma_medium.zip`은 재다운로드하지 않으며 `tracks.csv`까지 실제 존재하는지 확인합니다.
Kaggle 인증은 파일/환경변수가 없다는 이유만으로 OAuth를 차단하지 않고 CLI에 맡깁니다.
FMA metadata에서는 `fma_metadata/tracks.csv`만 추출합니다. `raw_albums.csv`는 이후 학습 코드에서 사용하지 않으므로 새로 쓰지 않습니다.


In [5]:
from dotenv import load_dotenv
# 기본은 프로젝트 .env만 확인. 별도 .env가 있으면 명시하세요. 토큰을 코드에 넣지 마세요.
EXTRA_ENV_FILE = ""
for env_path in [PROJECT_ROOT / '.env', CREDENTIAL_ROOT / '.env'] + ([Path(EXTRA_ENV_FILE)] if EXTRA_ENV_FILE else []):
    if env_path.is_file(): load_dotenv(env_path, override=False)


def configure_kaggle_auth():
    if os.environ.get('KAGGLE_API_TOKEN') or (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
        return
    config_dir = Path(os.environ.get('KAGGLE_CONFIG_DIR', str(Path.home()/'.kaggle'))).expanduser()
    for folder in (config_dir, CREDENTIAL_ROOT):
        token = folder / 'access_token'
        if token.is_file():
            value = token.read_text(encoding='utf-8-sig').strip()
            if value:
                os.environ['KAGGLE_API_TOKEN'] = value
                return
        legacy = folder / 'kaggle.json'
        if legacy.is_file():
            value = json.loads(legacy.read_text(encoding='utf-8-sig'))
            if value.get('username') and value.get('key'):
                os.environ['KAGGLE_USERNAME'] = str(value['username'])
                os.environ['KAGGLE_KEY'] = str(value['key'])
                if os.name != 'nt': legacy.chmod(0o600)
                return
    print('명시적 Kaggle 키 없음: 기존 CLI/OAuth 인증으로 시도합니다. (키 값은 출력하지 않음)')


def kaggle_command(*args):
    # 현재 커널 Python의 console entrypoint를 사용; 다른 conda 환경의 kaggle.exe 방지
    entry = ("import sys; from importlib.metadata import distribution; "
             "sys.argv=['kaggle', *sys.argv[1:]]; "
             "next(e for e in distribution('kaggle').entry_points "
             "if e.group=='console_scripts' and e.name=='kaggle').load()()")
    return [sys.executable, '-c', entry, *args]


def split_voice_paths(paths):
    real, fake, unknown = [], [], []
    for path in paths:
        path = Path(path)
        # Project 부모 폴더 이름이 real/fake인 경우를 피하고 데이터 폴더 안만 검사
        rel = path.resolve().relative_to(VOICE_ROOT.resolve())
        parts = {p.lower() for p in rel.parts[:-1]}
        if 'real' in parts and 'fake' not in parts: real.append(path)
        elif 'fake' in parts and 'real' not in parts: fake.append(path)
        else: unknown.append(path)
    return real, fake, unknown


voice_audio = audio_files(VOICE_ROOT)
rv, fv, _ = split_voice_paths(voice_audio)
if min(len(rv),len(fv)) < CFG.source_per_pool:
    voice_zip = VOICE_ROOT / (KAGGLE_DATASET.split('/')[-1] + '.zip')
    if voice_zip.is_file():
        safe_extract_zip(voice_zip, VOICE_ROOT)
    else:
        if not ALLOW_DOWNLOADS:
            raise RuntimeError('Kaggle 음성 부족. 다운로드 허용 또는 기존 데이터 경로 지정이 필요합니다.')
        configure_kaggle_auth()
        print('Kaggle 다운로드 시작. 로그인 필요 시 같은 커널 환경의 kaggle auth login을 먼저 실행하세요.')
        run_to_log(kaggle_command('datasets','download','-d',KAGGLE_DATASET,'-p',str(VOICE_ROOT)),
                   VOICE_ROOT / 'kaggle_download.log')
        if not voice_zip.is_file():
            raise FileNotFoundError(f'Kaggle ZIP 결과 확인 필요: {voice_zip}')
        safe_extract_zip(voice_zip, VOICE_ROOT)
voice_audio = audio_files(VOICE_ROOT)
real_voice_all, fake_voice_all, unresolved_voice = split_voice_paths(voice_audio)
if min(len(real_voice_all),len(fake_voice_all)) < CFG.source_per_pool:
    raise RuntimeError(f'Voice pool 부족: real={len(real_voice_all)}, fake={len(fake_voice_all)}, unresolved={len(unresolved_voice)}')

FMA_FILES = {
    'fma_medium.zip': ('https://os.unil.cloud.switch.ch/fma/fma_medium.zip',
                       'c67b69ea232021025fca9231fc1c7c1a063ab50b'),
    'fma_metadata.zip': ('https://os.unil.cloud.switch.ch/fma/fma_metadata.zip',
                         'f0df49ffe5f2a6008d7dc83c6915b31835dfe733'),
}
for filename,(url,sha1) in FMA_FILES.items():
    archive_path = download_http(url, FMA_ROOT / filename, sha1, 'sha1')
    # 이 노트북에서 사용하는 메타데이터 tracks.csv만 추출. 음원 ZIP은 전체 추출.
    needed = {'fma_metadata/tracks.csv'} if filename == 'fma_metadata.zip' else None
    safe_extract_zip(archive_path, FMA_ROOT, members=needed)
fma_audio = audio_files(FMA_ROOT / 'fma_medium')
tracks_path = FMA_ROOT / 'fma_metadata' / 'tracks.csv'
if not tracks_path.is_file(): raise FileNotFoundError(tracks_path)
if len(fma_audio) < CFG.source_per_pool: raise RuntimeError(f'FMA MP3 부족: {len(fma_audio)}. 음원 압축 해제/경로 확인')
print({'real_voice':len(real_voice_all), 'fake_voice':len(fake_voice_all), 'FMA_audio':len(fma_audio)})


fma_medium.zip: 기존 정상 파일 25,002개 / 처리 0개


Extract fma_medium.zip: 0it [00:00, ?it/s]


fma_metadata.zip: 기존 정상 파일 1개 / 처리 0개


Extract fma_metadata.zip: 0it [00:00, ?it/s]


{'real_voice': 9609, 'fake_voice': 7183, 'FMA_audio': 25000}


## 3. SONICS 준비 (원본 revision 유지)
이미 추출된 파일이 충분하면 네트워크를 사용하지 않습니다. 부족한 경우 원본의 두 ZIP을 내려받아 안전하게 추출합니다.
SONICS `no_vocal`은 메타데이터상의 성분 정보이며 실제 구간의 보컬 존재를 확정하는 정답은 아닙니다.

In [6]:
from huggingface_hub import hf_hub_download

SONICS_REPO = "awsaf49/sonics"
SONICS_REVISION = "3788dca9f9f11ad92e9097ef4b58eee247661e7f"
SONICS_PARTS = ["fake_songs/part_01.zip", "fake_songs/part_02.zip"]
DOWNLOAD_SONICS = ALLOW_DOWNLOADS
DELETE_SONICS_ARCHIVES_AFTER_EXTRACT = False
SONICS_METADATA_PATH = SONICS_ROOT / "fake_songs.csv"
SONICS_AUDIO_ROOT = SONICS_ROOT / "fake_songs"
SONICS_ROOT.mkdir(parents=True, exist_ok=True)


def download_sonics_file(filename):
    return Path(hf_hub_download(
        repo_id=SONICS_REPO,
        repo_type="dataset",
        filename=filename,
        revision=SONICS_REVISION,
        local_dir=str(SONICS_ROOT),
    ))



if not SONICS_METADATA_PATH.is_file():
    if not ALLOW_DOWNLOADS: raise FileNotFoundError(SONICS_METADATA_PATH)
    download_sonics_file("fake_songs.csv")
if len(audio_files(SONICS_AUDIO_ROOT)) < CFG.source_per_pool:
    if not ALLOW_DOWNLOADS: raise RuntimeError("SONICS 파일 부족. 기존 경로 또는 ALLOW_DOWNLOADS 확인")
    for part_name in SONICS_PARTS:
        archive_path = SONICS_ROOT / part_name
        if not archive_path.is_file(): archive_path = download_sonics_file(part_name)
        safe_extract_zip(archive_path, SONICS_ROOT)
# 아카이브를 자동 삭제하지 않습니다.

sonics_audio = sorted(
    path for path in SONICS_AUDIO_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in AUDIO_SUFFIXES and path.stat().st_size > 0
)
sonics_metadata = pd.read_csv(
    SONICS_METADATA_PATH,
    usecols=["filename", "algorithm", "source", "label", "target", "skip_time", "no_vocal", "split"],
    low_memory=False,
)
sonics_metadata = sonics_metadata[sonics_metadata["target"].eq(1)].copy()
sonics_metadata["file_stem"] = sonics_metadata["filename"].astype(str).map(lambda value: Path(value).stem)
if sonics_metadata["no_vocal"].isna().any():
    raise ValueError("SONICS no_vocal 결측치: 자동 보컬 유무 판정 금지")
if sonics_metadata["no_vocal"].dtype != bool:
    sonics_metadata["no_vocal"] = (
        sonics_metadata["no_vocal"].astype(str).str.strip().str.lower().isin({"true", "1", "yes"})
    )
sonics_metadata = sonics_metadata.drop_duplicates("file_stem", keep="last")
sonics_metadata_lookup = sonics_metadata.set_index("file_stem").to_dict("index")
sonics_candidates = [
    str(path.resolve()) for path in sonics_audio if path.stem in sonics_metadata_lookup
]
print({
    "SONICS extracted audio": len(sonics_audio),
    "metadata fake rows": len(sonics_metadata),
    "matched candidates": len(sonics_candidates),
    "sources": sonics_metadata.loc[
        sonics_metadata["file_stem"].isin({Path(path).stem for path in sonics_candidates}), "source"
    ].value_counts().to_dict(),
})
if len(sonics_candidates) < CFG.source_per_pool:
    raise RuntimeError(f"SONICS fake song이 {CFG.source_per_pool:,}개 필요합니다.")


c:\Users\shj04\miniconda3\envs\new_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'SONICS extracted audio': 10000, 'metadata fake rows': 49074, 'matched candidates': 10000, 'sources': {'udio': 5035, 'suno': 4965}}


## 4. 네 pool 후보 선택 / 라이선스·장르·경로 확인
기존 NoDerivatives 제외 정책을 유지합니다. 필터를 통과했다는 사실만으로 대회 사용 권한 전체를 보증하지 않습니다.
후보 수가 부족하면 복제/임의 라벨 변환 없이 중단합니다. 기존 source 후보 선택 방법은 그대로 유지합니다.

In [7]:
def stable_int(text):
    return int(hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:16], 16)


def select_exact(paths, count, namespace):
    unique_paths = sorted({str(Path(path).resolve()) for path in paths})
    ordered = sorted(unique_paths, key=lambda value: stable_int(f"{CFG.seed}|{namespace}|{value}"))
    if len(ordered) < count:
        raise RuntimeError(f"{namespace}: {count:,}개 필요, {len(ordered):,}개 발견")
    return ordered[:count]


def select_balanced_exact(paths, count, namespace, group_lookup):
    """그룹별 round-robin으로 선택해 한 장르/생성기가 풀을 지배하지 않게 한다."""
    unique_paths = sorted({str(Path(path).resolve()) for path in paths})
    if len(unique_paths) < count:
        raise RuntimeError(f"{namespace}: {count:,}개 필요, {len(unique_paths):,}개 발견")
    queues = {}
    for path in unique_paths:
        group = str(group_lookup.get(path, "unknown") or "unknown")
        queues.setdefault(group, []).append(path)
    for group, values in queues.items():
        queues[group] = sorted(
            values, key=lambda value: stable_int(f"{CFG.seed}|{namespace}|{group}|{value}")
        )
    groups = sorted(queues, key=lambda group: stable_int(f"group|{CFG.seed}|{namespace}|{group}"))
    selected_paths = []
    cursor = 0
    while len(selected_paths) < count:
        progressed = False
        for group in groups:
            if cursor < len(queues[group]):
                selected_paths.append(queues[group][cursor])
                progressed = True
                if len(selected_paths) == count:
                    break
        if not progressed:
            break
        cursor += 1
    if len(selected_paths) != count:
        raise RuntimeError(f"{namespace}: balanced selection failed ({len(selected_paths):,}/{count:,})")
    return selected_paths


real_voice_all, fake_voice_all, unresolved_voice = [], [], []
for path in voice_audio:
    path = Path(path)
    parts = {part.lower() for part in path.resolve().relative_to(VOICE_ROOT.resolve()).parts[:-1]}
    if "real" in parts and "fake" not in parts:
        real_voice_all.append(path)
    elif "fake" in parts and "real" not in parts:
        fake_voice_all.append(path)
    else:
        unresolved_voice.append(path)
print({"real_voice": len(real_voice_all), "fake_voice": len(fake_voice_all), "unresolved": len(unresolved_voice)})

tracks_path = FMA_ROOT / "fma_metadata" / "tracks.csv"
tracks = pd.read_csv(tracks_path, index_col=0, header=[0, 1], low_memory=False)
small_medium_tracks = tracks[
    tracks[("set", "subset")].astype(str).str.strip().str.lower().isin({"small", "medium"})
].copy()

def fma_track_path(track_id):
    track_id = int(track_id)
    return FMA_ROOT / "fma_medium" / f"{track_id:06d}"[:3] / f"{track_id:06d}.mp3"

def fma_license_allowed(value):
    value = str(value).lower()
    if not value or value == "nan":
        return False
    compact = re.sub(r"[^a-z0-9]+", "", value)
    normalized = re.sub(r"[^a-z0-9]+", " ", value)
    no_derivatives = (
        "noderivative" in compact
        or "noderivs" in compact
        or "musicsharing" in compact
        or re.search(r"\bby\s+(?:nc\s+)?nd\b", normalized) is not None
    )
    return not no_derivatives

fma_candidates = []
fma_license_lookup = {}
fma_genre_lookup = {}
fma_artist_lookup = {}
for track_id, row in small_medium_tracks.iterrows():
    path = fma_track_path(track_id)
    license_value = row.get(("track", "license"), "")
    if path.is_file() and path.stat().st_size > 0 and fma_license_allowed(license_value):
        resolved_path = str(path.resolve())
        fma_candidates.append(path)
        fma_license_lookup[resolved_path] = str(license_value)
        genre_value = str(row.get(("track", "genre_top"), "unknown")).strip()
        fma_genre_lookup[resolved_path] = genre_value if genre_value and genre_value != "nan" else "unknown"
        artist_value = str(row.get(("artist", "id"), "unknown")).strip()
        fma_artist_lookup[resolved_path] = artist_value if artist_value and artist_value != "nan" else "unknown"

sonics_balance_lookup = {}
for path in sonics_candidates:
    record = sonics_metadata_lookup.get(Path(path).stem, {})
    source_name = str(record.get("source", "unknown") or "unknown")
    algorithm_name = str(record.get("algorithm", "unknown") or "unknown")
    sonics_balance_lookup[str(Path(path).resolve())] = f"{source_name}|{algorithm_name}"

print({'real_voice':len(real_voice_all),'fake_voice':len(fake_voice_all),
       'FMA_after_license':len(fma_candidates),'SONICS':len(sonics_candidates)})
if not fma_candidates:
    raise RuntimeError('FMA 후보 0: 음원 디렉터리/메타데이터 ID/라이선스별 값 확인. 필터를 제거하지 마세요.')
selected = {
    "real_voice": select_exact(real_voice_all, CFG.source_per_pool, "real_voice"),
    "fake_voice": select_exact(fake_voice_all, CFG.source_per_pool, "fake_voice"),
    "real_music": select_balanced_exact(
        fma_candidates, CFG.source_per_pool, "real_music", fma_genre_lookup
    ),
    "fake_music": select_balanced_exact(
        sonics_candidates, CFG.source_per_pool, "fake_music", sonics_balance_lookup
    ),
}
assert sum(map(len, selected.values())) == 25_000
if set(selected["real_voice"]) & set(selected["fake_voice"]):
    raise RuntimeError("real/fake voice pool overlap")
display(pd.DataFrame({name: [len(paths)] for name, paths in selected.items()}))
print("FMA eligible after license filter:", f"{len(fma_candidates):,}")
print("balanced FMA genres:", pd.Series([fma_genre_lookup[path] for path in selected["real_music"]]).value_counts().head(20).to_dict())
print("balanced SONICS groups:", pd.Series([sonics_balance_lookup[path] for path in selected["fake_music"]]).value_counts().to_dict())


{'real_voice': 9609, 'fake_voice': 7183, 'unresolved': 0}
{'real_voice': 9609, 'fake_voice': 7183, 'FMA_after_license': 14639, 'SONICS': 10000}


,real_voice,fake_voice,real_music,fake_music
0,6250,6250,6250,6250


FMA eligible after license filter: 14,639
balanced FMA genres: {'Electronic': 633, 'Experimental': 632, 'Rock': 632, 'Instrumental': 632, 'Folk': 632, 'Pop': 632, 'Hip-Hop': 632, 'Old-Time / Historic': 504, 'Classical': 427, 'International': 362, 'Jazz': 223, 'Soul-RnB': 121, 'Spoken': 80, 'Country': 51, 'Blues': 42, 'Easy Listening': 15}
balanced SONICS groups: {'udio|udio-120s': 2882, 'suno|chirp-v3.5': 2881, 'udio|udio-30s': 197, 'suno|chirp-v3': 196, 'suno|chirp-v2-xxl-alpha': 94}


## 5. Group-disjoint source 분할
원본의 파일명 추정 / MFCC pseudo-speaker / FMA artist / SONICS song ID 방식을 유지합니다.
**MFCC cluster는 실제 speaker ID를 보증하지 않고, SONICS 곡 분리는 generator holdout이 아닙니다.**
캐시에 예전 파일이나 NaN 특징이 섞여 있으면 현재 대상에 해당하는 유효한 96차원 특징만 사용합니다.

In [8]:
require_names("real_voice_all", "fake_voice_all", "fma_candidates", "sonics_candidates")
def infer_voice_speaker(path):
    stem = Path(path).stem.lower()
    stem = re.sub(r"(?:[_-](?:chunk|segment|part|clip))?[_-]?\d+$", "", stem)
    converted = re.split(r"(?:-to-|_to_|\s+to\s+)", stem)
    identity = converted[-1]
    identity = re.sub(r"(?:[_-](?:real|fake|original|converted))+$", "", identity)
    identity = re.sub(r"[^a-z0-9가-힣]+", "_", identity).strip("_")
    return identity or Path(path).stem.lower()


def make_group_lookup(paths, resolver, namespace):
    lookup = {str(Path(path).resolve()): str(resolver(str(Path(path).resolve()))) for path in paths}
    counts = pd.Series(lookup.values()).value_counts()
    if len(counts) < 2:
        warnings.warn(f"{namespace}: 화자 group을 파일명에서 복원하지 못해 파일 단위 group으로 대체합니다.")
        lookup = {path: f"file:{Path(path).stem}" for path in lookup}
    return lookup


VOICE_FEATURE_CACHE = CACHE_ROOT / "voice_pseudo_speaker_features.csv"
VOICE_GROUP_CACHE = CACHE_ROOT / "voice_pseudo_speaker_groups.csv"


def speaker_feature(path):
    audio = decode_audio_rate(path, CFG.sample_rate, seconds=4.0).numpy()
    if len(audio) < 2_048:
        raise ValueError("too short")
    audio = librosa.util.normalize(np.asarray(audio, np.float32))
    mfcc = librosa.feature.mfcc(y=audio, sr=CFG.sample_rate, n_mfcc=24, n_fft=512, hop_length=160)
    delta = librosa.feature.delta(mfcc, width=min(9, mfcc.shape[1] // 2 * 2 - 1)) if mfcc.shape[1] >= 5 else np.zeros_like(mfcc)
    stacked = np.concatenate([mfcc, delta], axis=0)
    return np.concatenate([stacked.mean(1), stacked.std(1)]).astype(np.float32)


def pseudo_speaker_group_lookups(real_paths, fake_paths, clusters=64):
    from sklearn.cluster import MiniBatchKMeans
    from sklearn.preprocessing import StandardScaler

    all_paths = sorted({str(Path(path).resolve()) for path in [*real_paths, *fake_paths]})
    if VOICE_FEATURE_CACHE.exists():
        feature_frame = pd.read_csv(VOICE_FEATURE_CACHE)
    else:
        feature_frame = pd.DataFrame(columns=["path", *[f"f{i:03d}" for i in range(96)]])
    feature_frame = feature_frame.drop_duplicates("path", keep="last")
    cols = [f"f{i:03d}" for i in range(96)]
    feature_frame = feature_frame.reindex(columns=["path", *cols])
    feature_frame = feature_frame[feature_frame.path.astype(str).isin(all_paths)].copy()
    feature_frame[cols] = feature_frame[cols].apply(pd.to_numeric, errors="coerce")
    feature_frame = feature_frame[np.isfinite(feature_frame[cols].to_numpy(float)).all(axis=1)].copy()
    completed = set(feature_frame.path.astype(str))
    rows = feature_frame.to_dict("records")
    for index, path in enumerate(tqdm([p for p in all_paths if p not in completed], desc="voice pseudo-speaker features"), 1):
        try:
            feature = speaker_feature(path)
            if feature.shape != (96,) or not np.isfinite(feature).all(): raise ValueError("invalid MFCC feature")
            rows.append({"path": path, **{f"f{i:03d}": float(value) for i, value in enumerate(feature)}})
        except Exception as exc:
            print("speaker feature skip:", path, repr(exc))
        if index % 128 == 0:
            atomic_csv(pd.DataFrame(rows), VOICE_FEATURE_CACHE)
    feature_frame = pd.DataFrame(rows).drop_duplicates("path", keep="last")
    atomic_csv(feature_frame, VOICE_FEATURE_CACHE)
    feature_columns = [column for column in feature_frame if column.startswith("f")]
    if len(feature_frame) < 2 * CFG.source_per_pool or len(feature_columns) != 96:
        raise RuntimeError(f"pseudo-speaker feature shortage: {len(feature_frame):,}, dims={len(feature_columns)}")
    matrix = StandardScaler().fit_transform(feature_frame[feature_columns].to_numpy(np.float32))
    labels = MiniBatchKMeans(
        n_clusters=max(2, min(clusters, len(feature_frame) // 50)), random_state=CFG.seed,
        batch_size=1024, n_init=10,
    ).fit_predict(matrix)
    grouped = pd.DataFrame({"path": feature_frame.path.astype(str), "split_group": [f"pseudo_{x:03d}" for x in labels]})
    atomic_csv(grouped, VOICE_GROUP_CACHE)
    lookup = dict(zip(grouped.path, grouped.split_group))
    return lookup


def build_voice_group_lookups(real_paths, fake_paths):
    parsed_real = make_group_lookup(real_paths, infer_voice_speaker, "real_voice")
    parsed_fake = make_group_lookup(fake_paths, infer_voice_speaker, "fake_voice")
    real_counts = pd.Series(parsed_real.values()).value_counts()
    fake_counts = pd.Series(parsed_fake.values()).value_counts()
    common = set(real_counts.index) & set(fake_counts.index)
    metadata_is_usable = (
        len(common) >= 4 and real_counts.median() >= 3 and fake_counts.median() >= 3
    )
    if metadata_is_usable:
        print("voice split groups: filename speaker metadata")
        return parsed_real, parsed_fake
    print("voice filenames have no usable speaker ID; building cached MFCC pseudo-speaker groups")
    combined = pseudo_speaker_group_lookups(real_paths, fake_paths)
    return (
        {str(Path(path).resolve()): combined[str(Path(path).resolve())] for path in real_paths if str(Path(path).resolve()) in combined},
        {str(Path(path).resolve()): combined[str(Path(path).resolve())] for path in fake_paths if str(Path(path).resolve()) in combined},
    )


def select_group_disjoint(paths, counts, namespace, group_lookup, balance_lookup=None):
    paths = sorted({str(Path(path).resolve()) for path in paths})
    train_count, validation_count = counts["train"], counts["validation"]
    groups = {}
    for path in paths:
        groups.setdefault(str(group_lookup[path]), []).append(path)
    ordered_groups = sorted(groups, key=lambda group: stable_int(f"holdout|{CFG.seed}|{namespace}|{group}"))
    validation_groups, heldout_size = [], 0
    for group in ordered_groups:
        if len(paths) - heldout_size - len(groups[group]) < train_count:
            continue
        validation_groups.append(group); heldout_size += len(groups[group])
        if heldout_size >= validation_count:
            break
    if heldout_size < validation_count:
        raise RuntimeError(f"{namespace}: group-disjoint validation {validation_count}개를 구성할 수 없습니다.")
    heldout = [path for group in validation_groups for path in groups[group]]
    train_candidates = [path for group, values in groups.items() if group not in validation_groups for path in values]
    selector = select_balanced_exact if balance_lookup is not None else select_exact
    if balance_lookup is None:
        validation = selector(heldout, validation_count, namespace + "_validation")
        train = selector(train_candidates, train_count, namespace + "_train")
    else:
        validation = selector(heldout, validation_count, namespace + "_validation", balance_lookup)
        train = selector(train_candidates, train_count, namespace + "_train", balance_lookup)
    split = {path: "train" for path in train}; split.update({path: "validation" for path in validation})
    selected_paths = train + validation
    selected_groups = {path: group_lookup[path] for path in selected_paths}
    train_groups = {selected_groups[path] for path in train}; val_groups = {selected_groups[path] for path in validation}
    if train_groups & val_groups:
        raise RuntimeError(f"{namespace}: group leakage")
    return selected_paths, split, selected_groups


def select_joint_voice_group_disjoint(pool_paths, pool_group_lookup, counts):
    normalized = {
        pool: sorted({str(Path(path).resolve()) for path in paths})
        for pool, paths in pool_paths.items()
    }
    grouped = {}
    for pool, paths in normalized.items():
        by_group = {}
        for path in paths:
            by_group.setdefault(pool_group_lookup[pool][path], []).append(path)
        grouped[pool] = by_group
    common_groups = set.intersection(*(set(value) for value in grouped.values()))
    train_count, validation_count = counts["train"], counts["validation"]
    validation_groups = []
    # 고정 hash 순서 한 번만으로 실패할 수 있어 128개 재현 가능한 그룹 순서 시도
    for attempt in range(128):
        trial, heldout = [], {pool: 0 for pool in normalized}
        ordered = sorted(common_groups, key=lambda value: stable_int(f"voice-holdout|{CFG.seed}|{attempt}|{value}"))
        for group in ordered:
            proposed = {pool: heldout[pool] + len(grouped[pool][group]) for pool in normalized}
            if any(len(normalized[pool])-proposed[pool] < train_count for pool in normalized): continue
            trial.append(group); heldout=proposed
            if all(value >= validation_count for value in heldout.values()):
                validation_groups = trial; break
        if validation_groups: break
    if not validation_groups:
        raise RuntimeError("공통 voice group holdout 구성 실패. 그룹별 수량 확인/실제 speaker metadata 필요. 파일 분할로 자동 우회하지 않음")
    selected, splits, chosen_groups = {}, {}, {}
    for pool, paths in normalized.items():
        validation_candidates = [path for group in validation_groups for path in grouped[pool][group]]
        train_candidates = [path for group, values in grouped[pool].items() if group not in validation_groups for path in values]
        validation = select_exact(validation_candidates, validation_count, pool + "_validation")
        train = select_exact(train_candidates, train_count, pool + "_train")
        selected[pool] = train + validation
        splits[pool] = {path: "train" for path in train} | {path: "validation" for path in validation}
        chosen_groups[pool] = {path: pool_group_lookup[pool][path] for path in selected[pool]}
    return selected, splits, chosen_groups


voice_real_groups, voice_fake_groups = build_voice_group_lookups(real_voice_all, fake_voice_all)
real_voice_all = [path for path in real_voice_all if str(Path(path).resolve()) in voice_real_groups]
fake_voice_all = [path for path in fake_voice_all if str(Path(path).resolve()) in voice_fake_groups]
fma_groups = {str(Path(path).resolve()): fma_artist_lookup[str(Path(path).resolve())] for path in fma_candidates}
sonics_song_groups = {str(Path(path).resolve()): Path(path).stem for path in sonics_candidates}

selected, preassigned_split, selection_group_lookup = select_joint_voice_group_disjoint(
    {"real_voice": real_voice_all, "fake_voice": fake_voice_all},
    {"real_voice": voice_real_groups, "fake_voice": voice_fake_groups},
    CFG.source_split_counts,
)
for pool, paths, group_lookup, balance_lookup in (
    ("real_music", fma_candidates, fma_groups, fma_genre_lookup),
    ("fake_music", sonics_candidates, sonics_song_groups, sonics_balance_lookup),
):
    chosen, split, chosen_groups = select_group_disjoint(
        paths, CFG.source_split_counts, pool, group_lookup, balance_lookup
    )
    selected[pool] = chosen
    preassigned_split[pool] = split
    selection_group_lookup[pool] = chosen_groups
assert sum(map(len, selected.values())) == 25_000
display(pd.DataFrame({pool: pd.Series(split).value_counts() for pool, split in preassigned_split.items()}).fillna(0).astype(int))


voice filenames have no usable speaker ID; building cached MFCC pseudo-speaker groups


voice pseudo-speaker features: 0it [00:00, ?it/s]


,real_voice,fake_voice,real_music,fake_music
train,5625,5625,5625,5625
validation,625,625,625,625


## 6. PANNs 라벨/가중치 준비 및 보컬 스크리닝
`wget`에 의존하지 않고 라벨 CSV를 import 전에 준비합니다. PANNs 자동 다운로드 wrapper 대신
공식 Cnn14 구조 + 검증된 가중치를 명시적으로 읽습니다. MD5는 전송 무결성 검사이며 완전한 보안 인증 수단은 아닙니다.
기존 스크리닝 캐시는 재사용하되 실패한 파일을 '보컬 없음'으로 간주하지 않습니다.

In [9]:
PANNS_LABEL_PATH = Path.home() / 'panns_data' / 'class_labels_indices.csv'
PANNS_WEIGHT_PATH = PROJECT_ROOT / 'panns' / 'Cnn14_mAP=0.431.pth'
PANNS_MD5 = '541141fa2ee191a88f24a3219fff024e'
# 첫 요청은 공식 저자 배포처. 미러는 같은 공식 MD5를 통과할 때만 허용합니다.
PANNS_WEIGHT_URL = 'https://zenodo.org/records/3987831/files/Cnn14_mAP%3D0.431.pth?download=1'
PANNS_MIRROR_URL = ('https://huggingface.co/thelou1s/panns-inference/resolve/'
    'f98a09ed0dca73b284786393cfc2b2bd6f46f878/Cnn14_mAP%3D0.431.pth?download=true')
FMA_VOCAL_THRESHOLD = .20
FMA_SCREEN_PATH = CACHE_ROOT / 'fma_panns_vocal_screen.csv'
RUN_FMA_VOCAL_SCREEN = True


def prepare_panns_assets():
    labels_url = 'https://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv'
    labels_mirror = 'https://raw.githubusercontent.com/qiuqiangkong/audioset_tagging_cnn/master/metadata/class_labels_indices.csv'
    download_http(labels_url, PANNS_LABEL_PATH, mirrors=(labels_mirror,))
    table = pd.read_csv(PANNS_LABEL_PATH)
    if list(table.columns) != ['index','mid','display_name'] or len(table)!=527:
        raise ValueError('PANNs 라벨 CSV 형식 오류. 잘못된 파일을 별도 보관 후 재시도하세요.')
    if table['index'].tolist()!=list(range(527)): raise ValueError('PANNs label index 오류')
    download_http(PANNS_WEIGHT_URL, PANNS_WEIGHT_PATH, PANNS_MD5, 'md5', mirrors=(PANNS_MIRROR_URL,))
    return table.display_name.astype(str).tolist()


def screen_fma_sources():
    names = ['Speech','Male speech, man speaking','Female speech, woman speaking',
             'Conversation','Narration, monologue','Singing','Choir','Vocal music']
    columns = ['path','vocal_score','music_score','contains_voice','screen_ok']
    frame = pd.read_csv(FMA_SCREEN_PATH) if FMA_SCREEN_PATH.exists() else pd.DataFrame(columns=columns)
    if not set(columns).issubset(frame.columns):
        raise ValueError(f'기존 PANNs 캐시 schema 오류: {FMA_SCREEN_PATH}')
    frame = frame.drop_duplicates('path', keep='last')
    if len(frame):
        frame['screen_ok'] = as_boolean(frame['screen_ok'])
        for name in ('vocal_score','music_score'): frame[name] = pd.to_numeric(frame[name],errors='coerce')
        valid = np.isfinite(frame[['vocal_score','music_score']].to_numpy(float)).all(1)
        frame['screen_ok'] &= valid
    complete = set(frame.loc[frame.screen_ok.eq(True), 'path'].astype(str))
    todo = [path for path in selected['real_music'] if path not in complete]
    if todo and not RUN_FMA_VOCAL_SCREEN:
        raise RuntimeError('PANNs 미검사 음악이 있습니다. 라벨을 False로 채우지 않습니다. 스크리닝을 활성화하세요.')
    if todo:
        panns_labels = prepare_panns_assets()   # import 전에 CSV 준비
        from panns_inference.models import Cnn14
        tagger_model = Cnn14(sample_rate=32000,window_size=1024,hop_size=320,mel_bins=64,
                             fmin=50,fmax=14000,classes_num=527)
        state = torch.load(PANNS_WEIGHT_PATH,map_location='cpu',weights_only=True)
        tagger_model.load_state_dict(state['model'],strict=True)
        del state
        tagger_model.to(DEVICE).eval()
        vocal_indices = [panns_labels.index(name) for name in names if name in panns_labels]
        if not vocal_indices: raise RuntimeError('PANNs 보컬 label 누락')
        music_index = panns_labels.index('Music')
        pending, errors = [], []
        try:
            for offset in tqdm(range(0,len(todo),CFG.panns_batch),desc='FMA PANNs vocal screen'):
                batch_audio, valid_paths = [], []
                for path in todo[offset:offset+CFG.panns_batch]:
                    try:
                        wave = decode_audio_rate(path,32000,seconds=CFG.panns_seconds)
                        target = 32000*CFG.panns_seconds
                        wave = F.pad(wave[:target], (0,max(0,target-wave.numel())))
                        batch_audio.append(wave);valid_paths.append(path)
                    except Exception as exc:
                        errors.append({'path':path,'error':repr(exc)})
                if batch_audio:
                    with torch.inference_mode():
                        # PANNs 스크리닝 FP32. 학습 AMP와 별개.
                        scores = tagger_model(torch.stack(batch_audio).to(DEVICE))['clipwise_output'].cpu().numpy()
                    if not np.isfinite(scores).all(): raise RuntimeError('PANNs NaN/Inf')
                    for path,score in zip(valid_paths,scores):
                        vocal=float(score[vocal_indices].max())
                        pending.append(dict(path=path,vocal_score=vocal,music_score=float(score[music_index]),
                                            contains_voice=vocal>=FMA_VOCAL_THRESHOLD,screen_ok=True))
                if len(pending)>=128 or offset+CFG.panns_batch>=len(todo):
                    frame=pd.concat([frame,pd.DataFrame(pending,columns=columns)],ignore_index=True).drop_duplicates('path',keep='last')
                    atomic_csv(frame,FMA_SCREEN_PATH);pending=[]
        finally:
            del tagger_model;gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            if errors: atomic_csv(pd.DataFrame(errors), CACHE_ROOT/'fma_panns_errors.csv')
        if errors: raise RuntimeError(f'{len(errors)}개 PANNs 읽기 실패. 오류 CSV 확인; 유효 source에서 제외/재준비 후 source 선택부터 재실행')
    result=frame[frame.path.astype(str).isin(selected['real_music'])].copy()
    if set(result.path.astype(str))!=set(selected['real_music']) or not as_boolean(result.screen_ok).all():
        raise RuntimeError('선택된 FMA source의 PANNs 스크리닝이 완성되지 않았습니다.')
    result['contains_voice']=pd.to_numeric(result.vocal_score).ge(FMA_VOCAL_THRESHOLD)
    result['screen_ok']=as_boolean(result.screen_ok)
    print('FMA screening:',len(result),'vocal-like:',int(result.contains_voice.sum()))
    return result


def _normalized_path_key(path):
    """Windows 대소문자 차이까지 흡수한 비교용 키."""
    import os
    return os.path.normcase(str(Path(path).expanduser().resolve()))


def _load_panns_decode_blacklist():
    """이전에 확인한 디코딩 불가 FMA 경로를 읽습니다."""
    blacklist_path = CACHE_ROOT / "fma_panns_decode_blacklist.csv"
    if not blacklist_path.is_file():
        return set()

    table = pd.read_csv(blacklist_path, dtype=str, keep_default_na=False)
    if "path" not in table.columns:
        raise ValueError(f"PANNs blacklist schema 오류: {blacklist_path}")

    return {
        _normalized_path_key(path)
        for path in table["path"].astype(str)
        if str(path).strip()
    }


def _save_panns_decode_blacklist(paths, error_table=None):
    """디코딩 불가 경로를 영구 캐시. 원본 음원은 삭제하지 않습니다."""
    blacklist_path = CACHE_ROOT / "fma_panns_decode_blacklist.csv"

    old_rows = []
    if blacklist_path.is_file():
        old = pd.read_csv(blacklist_path, dtype=str, keep_default_na=False)
        if "path" in old.columns:
            old_rows = old.to_dict("records")

    error_lookup = {}
    if error_table is not None and {"path", "error"}.issubset(error_table.columns):
        error_lookup = {
            _normalized_path_key(row.path): str(row.error)
            for row in error_table.itertuples(index=False)
        }

    rows = list(old_rows)
    existing = {
        _normalized_path_key(row.get("path", ""))
        for row in rows
        if row.get("path", "")
    }

    for path in sorted(paths):
        key = _normalized_path_key(path)
        if key in existing:
            continue
        rows.append({
            "path": str(Path(path).expanduser().resolve()),
            "reason": error_lookup.get(key, "audio decode failed during PANNs screening"),
        })
        existing.add(key)

    out = pd.DataFrame(rows, columns=["path", "reason"]).drop_duplicates("path", keep="last")
    atomic_csv(out, blacklist_path)
    return blacklist_path


def _current_panns_decode_failures():
    """
    방금 screen_fma_sources()가 기록한 오류 중
    현재 selected['real_music']에 실제 포함된 경로만 반환합니다.
    """
    error_path = CACHE_ROOT / "fma_panns_errors.csv"
    if not error_path.is_file():
        return [], None

    table = pd.read_csv(error_path, dtype=str, keep_default_na=False)
    if not {"path", "error"}.issubset(table.columns):
        raise ValueError(f"PANNs 오류 CSV schema 오류: {error_path}")

    current = {
        _normalized_path_key(path)
        for path in selected["real_music"]
    }

    table = table[
        table["path"].astype(str).map(_normalized_path_key).isin(current)
    ].drop_duplicates("path", keep="last")

    paths = [
        str(Path(path).expanduser().resolve())
        for path in table["path"].astype(str)
    ]
    return paths, table


def _reselect_real_music_excluding(excluded_keys):
    """
    디코딩 불가 FMA를 후보에서 제외하고
    기존과 동일한 artist group-disjoint + genre balance 조건으로
    Real Music 6,250개를 다시 선택합니다.
    """
    global fma_candidates, fma_groups

    filtered = []
    for path in fma_candidates:
        resolved = str(Path(path).expanduser().resolve())
        if _normalized_path_key(resolved) in excluded_keys:
            continue
        filtered.append(Path(resolved))

    if len(filtered) < CFG.source_per_pool:
        raise RuntimeError(
            f"디코딩 불가 FMA 제외 후 후보 부족: "
            f"{len(filtered):,} < {CFG.source_per_pool:,}"
        )

    normalized = [str(path.resolve()) for path in filtered]

    missing_artist = [path for path in normalized if path not in fma_artist_lookup]
    missing_genre = [path for path in normalized if path not in fma_genre_lookup]
    if missing_artist or missing_genre:
        raise RuntimeError(
            "FMA lookup 누락: "
            f"artist={len(missing_artist)}, genre={len(missing_genre)}"
        )

    fma_candidates = filtered
    fma_groups = {
        path: fma_artist_lookup[path]
        for path in normalized
    }

    chosen, split, chosen_groups = select_group_disjoint(
        fma_candidates,
        CFG.source_split_counts,
        "real_music",
        fma_groups,
        fma_genre_lookup,
    )

    expected = sum(CFG.source_split_counts.values())
    if len(chosen) != expected:
        raise RuntimeError(
            f"Real Music 재선택 수량 오류: {len(chosen)} != {expected}"
        )

    if any(_normalized_path_key(path) in excluded_keys for path in chosen):
        raise RuntimeError("격리한 FMA 파일이 다시 선택되었습니다.")

    selected["real_music"] = chosen
    preassigned_split["real_music"] = split
    selection_group_lookup["real_music"] = chosen_groups

    print(
        "Real Music 재선택:",
        len(chosen),
        pd.Series(split).value_counts().to_dict(),
    )


def screen_fma_sources_with_repair(max_repair_rounds=8):
    """
    PANNs 읽기 실패만 자동 복구:
      1) 실패 경로 격리
      2) FMA 후보에서 제외
      3) real_music group-disjoint 재선택
      4) 기존 성공 PANNs 캐시 재사용
      5) 새로 필요한 파일만 다시 검사

    PANNs 모델 오류/NaN/가중치 오류 등 다른 RuntimeError는 그대로 중단합니다.
    """
    excluded = _load_panns_decode_blacklist()

    # 이전 실행에서 이미 격리한 파일이 있으면 첫 PANNs 실행 전에 제외합니다.
    if excluded:
        current_keys = {
            _normalized_path_key(path)
            for path in selected["real_music"]
        }
        if current_keys & excluded:
            print(
                f"기존 PANNs 디코딩 blacklist {len(current_keys & excluded)}개가 "
                "현재 선택에 포함되어 있어 먼저 재선택합니다."
            )
            _reselect_real_music_excluding(excluded)

    for attempt in range(max_repair_rounds + 1):
        try:
            result = screen_fma_sources()
            print(
                f"PANNs screening 완료: {len(result):,}개 "
                f"(자동 복구 round={attempt})"
            )
            return result

        except RuntimeError as exc:
            message = str(exc)

            # 이 wrapper가 해결하도록 설계된 오류가 아니면 숨기지 않습니다.
            if "PANNs 읽기 실패" not in message:
                raise

            failed_paths, error_table = _current_panns_decode_failures()

            if not failed_paths:
                raise RuntimeError(
                    "PANNs 읽기 실패가 발생했지만 현재 선택 파일에 해당하는 "
                    "오류 경로를 CSV에서 찾지 못했습니다."
                ) from exc

            new_keys = {
                _normalized_path_key(path)
                for path in failed_paths
            } - excluded

            if not new_keys:
                raise RuntimeError(
                    "같은 PANNs 디코딩 실패 경로가 재발했습니다. "
                    "blacklist/선택 상태를 확인하세요."
                ) from exc

            print("\nPANNs 디코딩 실패 FMA 격리:")
            for path in failed_paths:
                if _normalized_path_key(path) in new_keys:
                    print(" -", path)

            excluded |= new_keys
            blacklist_path = _save_panns_decode_blacklist(
                failed_paths,
                error_table,
            )
            print("blacklist 저장:", blacklist_path)

            _reselect_real_music_excluding(excluded)

    raise RuntimeError(
        f"PANNs 자동 복구가 {max_repair_rounds}회를 초과했습니다."
    )


# 직접 screen_fma_sources()를 호출하지 않고 자동 복구 wrapper를 사용합니다.
fma_screen = screen_fma_sources_with_repair()


기존 PANNs 디코딩 blacklist 4개가 현재 선택에 포함되어 있어 먼저 재선택합니다.
Real Music 재선택: 6250 {'train': 5625, 'validation': 625}
FMA screening: 6250 vocal-like: 1236
PANNs screening 완료: 6,250개 (자동 복구 round=0)


In [10]:
# ============================================================
# 실패한 FMA 파일 확인 + 읽기 재시도
#
# 기존 PANNs 스크리닝 셀 아래에 새 셀로 추가하세요.
# 파일 삭제 / 재다운로드 / 임의 라벨 생성 없음
# ============================================================

from pathlib import Path

import pandas as pd
import torch.nn.functional as F


# ------------------------------------------------------------
# 1. 현재 노트북 상태 확인
# ------------------------------------------------------------

required = [
    "CACHE_ROOT",
    "CFG",
    "selected",
    "decode_audio_rate",
    "screen_fma_sources",
]

missing_variables = [
    name for name in required if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "필요한 이전 셀의 변수가 없습니다:\n"
        + ", ".join(missing_variables)
        + "\n커널을 재시작했다면 설정 및 데이터 준비 셀을 "
          "먼저 실행하세요."
    )

error_path = Path(CACHE_ROOT) / "fma_panns_errors.csv"

if not error_path.is_file():
    raise FileNotFoundError(
        f"오류 기록을 찾지 못했습니다:\n{error_path}"
    )

error_df = pd.read_csv(
    error_path,
    dtype=str,
    keep_default_na=False,
)

if not {"path", "error"}.issubset(error_df.columns):
    raise ValueError(
        f"예상한 오류 CSV 형식이 아닙니다: "
        f"{error_df.columns.tolist()}"
    )

# 이전 실행의 오류가 남아 있을 수 있으므로
# 현재 선택된 Real Music에 해당하는 파일만 확인
current_paths = set(map(str, selected["real_music"]))

error_df = (
    error_df[error_df["path"].isin(current_paths)]
    .drop_duplicates("path", keep="last")
)

print("오류 CSV:", error_path)
print(f"현재 선택 파일 중 오류 기록: {len(error_df)}개")


# ------------------------------------------------------------
# 2. 실패 파일만 다시 읽어 입력 준비 확인
# ------------------------------------------------------------

failed_again = []

for row in error_df.itertuples(index=False):
    path = Path(row.path)

    print("\n" + "=" * 70)
    print("파일:", path)
    print("기록된 오류:", row.error)

    try:
        if not path.is_file():
            raise FileNotFoundError(
                f"실제 파일이 없습니다: {path}"
            )

        print(f"파일 크기: {path.stat().st_size:,} bytes")

        # 기존 스크리닝과 동일한 디코딩 조건
        wave = decode_audio_rate(
            path,
            32000,
            seconds=CFG.panns_seconds,
        )

        target_samples = 32000 * CFG.panns_seconds

        # 기존 스크리닝과 동일한 padding
        wave = F.pad(
            wave[:target_samples],
            (0, max(0, target_samples - wave.numel())),
        )

        print("읽기 및 입력 준비 재시도 성공:", tuple(wave.shape))

    except Exception as exc:
        failed_again.append(str(path))

        print("현재 재시도 오류:", repr(exc))

        # FFmpeg 오류는 repr()에 상세 stderr가 안 나올 수 있음
        stderr = getattr(exc, "stderr", None)

        if stderr:
            if isinstance(stderr, bytes):
                stderr = stderr.decode("utf-8", errors="replace")

            print("\nFFmpeg 상세 오류:")
            print(str(stderr)[-6000:])


# ------------------------------------------------------------
# 3. 읽기가 해결된 경우에만 스크리닝 재개
# ------------------------------------------------------------

if failed_again:
    print("\n" + "=" * 70)
    print(f"아직 읽지 못한 파일: {len(failed_again)}개")
    print("원본 파일과 기존 스크리닝 캐시는 그대로 유지했습니다.")
    print("다음 manifest 생성 셀은 아직 실행하지 마세요.")
    print("위 '현재 재시도 오류'와 FFmpeg 상세 오류를 확인하세요.")

else:
    print("\n읽기 재시도에서 남은 오류가 없습니다.")
    print("기존 성공 캐시를 재사용해 미완료 파일을 스크리닝합니다.")

    fma_screen = screen_fma_sources()

    print("\n선택된 FMA 파일의 스크리닝 완료")
    print("이제 다음 Source manifest 생성 셀로 진행하세요.")

오류 CSV: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\manifests\fma_panns_errors.csv
현재 선택 파일 중 오류 기록: 0개

읽기 재시도에서 남은 오류가 없습니다.
기존 성공 캐시를 재사용해 미완료 파일을 스크리닝합니다.
FMA screening: 6250 vocal-like: 1236

선택된 FMA 파일의 스크리닝 완료
이제 다음 Source manifest 생성 셀로 진행하세요.


## 7. Source manifest / 성분 라벨 / 누수 감사
원본의 5개 target 순서를 유지합니다. 없는 성분의 fake 라벨은 0 placeholder지만 해당 loss mask는 0입니다.
FMA 첫 10초 PANNs 검사와 SONICS `no_vocal`은 **파일 수준 약한 라벨**입니다. 임의 4초/16초 crop의 정답과 항상 일치한다고 보증할 수 없습니다.
5개 출력 컬럼의 일치만으로 라벨 품질·실제 생성 여부·미지 생성기 일반화를 보증하지 않습니다.

In [11]:
fma_voice_lookup = dict(zip(fma_screen["path"].astype(str), fma_screen["contains_voice"].astype(bool)))
rows = []
for pool, paths in selected.items():
    boundaries = {
        split_name: [path for path in paths if preassigned_split[pool][path] == split_name]
        for split_name in CFG.source_split_counts
    }
    assert {name: len(values) for name, values in boundaries.items()} == CFG.source_split_counts
    for split_name, split_paths in boundaries.items():
        for path in split_paths:
            sonics_record = sonics_metadata_lookup.get(Path(path).stem, {}) if pool == "fake_music" else {}
            if pool == "real_music":
                contains_voice = bool(fma_voice_lookup.get(path, False))
            elif pool == "fake_music":
                contains_voice = not bool(sonics_record.get("no_vocal", False))
            else:
                contains_voice = pool.endswith("voice")
            rows.append({
                "source_id": hashlib.sha256(f"{pool}|{path}".encode()).hexdigest()[:20],
                "pool": pool, "split": split_name, "path": path,
                "split_group": selection_group_lookup[pool][path],
                "contains_voice": contains_voice,
                "voice_fake": int(pool == "fake_voice" or (pool == "fake_music" and contains_voice)),
                "contains_music": pool.endswith("music"),
                "music_fake": int(pool == "fake_music"),
                "license": (
                    KAGGLE_LICENSE_NOTE if pool.endswith("voice") else
                    fma_license_lookup.get(path, "FMA artist-selected") if pool == "real_music" else
                    "CC-BY-NC-4.0"
                ),
                "origin": (
                    KAGGLE_DATASET if pool.endswith("voice") else
                    "FMA medium" if pool == "real_music" else SONICS_REPO
                ),
                "source_detail": str(sonics_record.get("source", "")),
                "generation_algorithm": str(sonics_record.get("algorithm", "")),
                "sonics_label": str(sonics_record.get("label", "")),
                "sonics_original_split": str(sonics_record.get("split", "")),
                "genre_or_style": (
                    fma_genre_lookup.get(path, "unknown") if pool == "real_music" else
                    sonics_balance_lookup.get(path, "unknown") if pool == "fake_music" else
                    "speech"
                ),
                "balance_group": (
                    fma_genre_lookup.get(path, "unknown") if pool == "real_music" else
                    sonics_balance_lookup.get(path, "unknown") if pool == "fake_music" else
                    "speech"
                ),
                "artist_group": fma_artist_lookup.get(path, "") if pool == "real_music" else "",
                "original_suffix": Path(path).suffix.lower(),
            })
source_manifest = pd.DataFrame(rows)
assert len(source_manifest) == 25_000
assert source_manifest["source_id"].is_unique
assert source_manifest.groupby("pool").size().eq(CFG.source_per_pool).all()
assert source_manifest.groupby(["pool", "split"]).size().to_dict() == {
    (pool, split_name): count
    for pool in selected for split_name, count in CFG.source_split_counts.items()
}
if source_manifest.groupby("path")["split"].nunique().max() != 1:
    raise RuntimeError("source path가 여러 split에 존재합니다.")
SOURCE_MANIFEST_PATH = MANIFEST_ROOT / "source_manifest_25000.csv"
source_manifest.to_csv(SOURCE_MANIFEST_PATH, index=False, encoding="utf-8")
display(pd.crosstab(source_manifest["pool"], source_manifest["split"], margins=True))
display(source_manifest.groupby(["pool", "license"]).size().rename("count").reset_index())
display(source_manifest[source_manifest["pool"].eq("fake_music")][
    ["source_detail", "generation_algorithm", "sonics_label", "contains_voice"]
].value_counts().rename("count").reset_index().head(30))
display(source_manifest.groupby(["pool", "balance_group"]).size().rename("count").reset_index())
display(pd.crosstab(source_manifest["pool"], source_manifest["original_suffix"], margins=True))
print("saved:", SOURCE_MANIFEST_PATH)


split,train,validation,All
pool,,,
fake_music,5625,625,6250
fake_voice,5625,625,6250
real_music,5625,625,6250
real_voice,5625,625,6250
All,22500,2500,25000


,pool,license,count
0,fake_music,CC-BY-NC-4.0,6250
1,fake_voice,CC-BY-SA-4.0 (original notebook claim; verify ...,6250
2,real_music,Art Libre,6
3,real_music,Attribution,429
4,real_music,Attribution 2.0 UK: England,1
5,real_music,Attribution 3.0 International,57
6,real_music,Attribution 3.0 United States,68
7,real_music,Attribution NonCommercial 2.5,1
8,real_music,Attribution-NonCommercial,404
9,real_music,Attribution-NonCommercial 2.5,13


,source_detail,generation_algorithm,sonics_label,contains_voice,count
0,udio,udio-120s,mostly fake,True,2897
1,suno,chirp-v3.5,half fake,True,1488
2,suno,chirp-v3.5,mostly fake,True,1378
3,udio,udio-30s,mostly fake,True,197
4,suno,chirp-v3,mostly fake,True,165
5,suno,chirp-v2-xxl-alpha,mostly fake,True,76
6,suno,chirp-v3,half fake,True,31
7,suno,chirp-v2-xxl-alpha,half fake,True,18


,pool,balance_group,count
0,fake_music,suno|chirp-v2-xxl-alpha,94
1,fake_music,suno|chirp-v3,196
2,fake_music,suno|chirp-v3.5,2866
3,fake_music,udio|udio-120s,2897
4,fake_music,udio|udio-30s,197
5,fake_voice,speech,6250
6,real_music,Blues,42
7,real_music,Classical,427
8,real_music,Country,51
9,real_music,Easy Listening,15


original_suffix,.mp3,.wav,All
pool,,,
fake_music,6250,0,6250
fake_voice,0,6250,6250
real_music,6250,0,6250
real_voice,0,6250,6250
All,12500,12500,25000


saved: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\manifests_fixed_v4_1\source_manifest_25000.csv


In [12]:
component_manifest = source_manifest.copy()
component_manifest["VOICE_PRESENT"] = component_manifest["contains_voice"].astype(int)
component_manifest["MUSIC_PRESENT"] = component_manifest["contains_music"].astype(int)
component_manifest["VOICE_FAKE"] = component_manifest["voice_fake"].astype(int)
component_manifest["MUSIC_FAKE"] = component_manifest["music_fake"].astype(int)
component_manifest["FILE_FAKE"] = component_manifest[["VOICE_FAKE", "MUSIC_FAKE"]].max(axis=1)

if (component_manifest["VOICE_FAKE"] > component_manifest["VOICE_PRESENT"]).any():
    raise RuntimeError("VOICE_FAKE=1인데 Voice가 존재하지 않는 source가 있습니다.")
if (component_manifest["MUSIC_FAKE"] > component_manifest["MUSIC_PRESENT"]).any():
    raise RuntimeError("MUSIC_FAKE=1인데 Music이 존재하지 않는 source가 있습니다.")

fma_failed = fma_screen.loc[~fma_screen["screen_ok"], "path"].astype(str).tolist()
if fma_failed:
    warnings.warn(
        f"FMA 보컬 스크리닝 실패 {len(fma_failed)}개는 보수적으로 REAL music-only로 유지합니다. "
        f"DynamicMix 디코딩이 실패하면 다른 source를 재시도합니다: {fma_failed[:3]}"
    )

label_columns = ["VOICE_PRESENT", "MUSIC_PRESENT", "VOICE_FAKE", "MUSIC_FAKE", "FILE_FAKE"]
display(component_manifest.groupby(["pool", *label_columns]).size().rename("count").reset_index())
display(pd.crosstab(component_manifest["pool"], component_manifest["split"], margins=True))
component_manifest["VOICE_FAKE_VALID"] = component_manifest["VOICE_PRESENT"]
component_manifest["MUSIC_FAKE_VALID"] = component_manifest["MUSIC_PRESENT"]
COMPONENT_MANIFEST_PATH = MANIFEST_ROOT / "component_manifest_25000.csv"
component_manifest.to_csv(COMPONENT_MANIFEST_PATH, index=False, encoding="utf-8")
print("component label audit: OK", COMPONENT_MANIFEST_PATH)
print("주의: 학습 데이터 원본 파일을 2차 평가 때 제출할 수 있는 라이선스인지 최종 확인하세요.")

group_leakage = (
    source_manifest.groupby(["pool", "split_group"])["split"].nunique().gt(1)
)
if group_leakage.any():
    raise RuntimeError("speaker/artist/song group leakage detected")
voice_group_leakage = (
    source_manifest[source_manifest.pool.str.endswith("voice")]
    .groupby("split_group")["split"].nunique().gt(1)
)
if voice_group_leakage.any():
    raise RuntimeError("real/fake voice 사이의 speaker group leakage detected")
display(source_manifest.groupby(["pool", "split"])["split_group"].nunique().unstack(fill_value=0))
sonics_split = source_manifest[source_manifest.pool.eq("fake_music")].copy()
display(pd.crosstab(sonics_split["source_detail"], sonics_split["split"], margins=True))
print("speaker/artist/song group leakage: 0")


,pool,VOICE_PRESENT,MUSIC_PRESENT,VOICE_FAKE,MUSIC_FAKE,FILE_FAKE,count
0,fake_music,1,1,1,1,1,6250
1,fake_voice,1,0,1,0,1,6250
2,real_music,0,1,0,0,0,5014
3,real_music,1,1,0,0,0,1236
4,real_voice,1,0,0,0,0,6250


split,train,validation,All
pool,,,
fake_music,5625,625,6250
fake_voice,5625,625,6250
real_music,5625,625,6250
real_voice,5625,625,6250
All,22500,2500,25000


component label audit: OK C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\manifests_fixed_v4_1\component_manifest_25000.csv
주의: 학습 데이터 원본 파일을 2차 평가 때 제출할 수 있는 라이선스인지 최종 확인하세요.


split,train,validation
pool,,
fake_music,5625,625
fake_voice,56,6
real_music,1921,125
real_voice,51,6


split,train,validation,All
source_detail,,,
suno,2855,301,3156
udio,2770,324,3094
All,5625,625,6250


speaker/artist/song group leakage: 0


## 8. 원본 DynamicMix recipe 25,000개
원본 선택 수(25,000)와 recipe 수(25,000)는 다른 개념입니다. Train 22,500 / validation 2,500 recipe 설정을 유지합니다.

In [13]:
RECIPE_TYPES = ["rv", "fv", "rm", "fm", "rv_rm", "fv_rm", "rv_fm", "fv_fm"]


def build_recipe_manifest():
    recipe_rows = []
    for split_name, count in CFG.recipe_counts.items():
        recipe_types = [RECIPE_TYPES[index % len(RECIPE_TYPES)] for index in range(count)]
        random.Random(CFG.seed + stable_int(split_name)).shuffle(recipe_types)
        for index, recipe_type in enumerate(recipe_types):
            recipe_rows.append({
                "recipe_id": f"{split_name}_{index:05d}",
                "split": split_name, "recipe_type": recipe_type,
                "seed": stable_int(f"recipe|{CFG.seed}|{split_name}|{index}|{recipe_type}") % (2**31 - 1),
            })
    return pd.DataFrame(recipe_rows)


recipe_manifest = build_recipe_manifest()
assert len(recipe_manifest) == 25_000
assert recipe_manifest["recipe_id"].is_unique
assert recipe_manifest.groupby("split").size().to_dict() == CFG.recipe_counts
RECIPE_MANIFEST_PATH = MANIFEST_ROOT / "train_validation_recipes_25000.csv"
recipe_manifest.to_csv(RECIPE_MANIFEST_PATH, index=False, encoding="utf-8")
display(pd.crosstab(recipe_manifest["recipe_type"], recipe_manifest["split"], margins=True))
print("saved:", RECIPE_MANIFEST_PATH)


split,train,validation,All
recipe_type,,,
fm,2813,313,3126
fv,2813,313,3126
fv_fm,2812,312,3124
fv_rm,2812,312,3124
rm,2813,313,3126
rv,2813,313,3126
rv_fm,2812,312,3124
rv_rm,2812,312,3124
All,22500,2500,25000


saved: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\manifests_fixed_v4_1\train_validation_recipes_25000.csv


## 9. 공식 AASIST/RawBoost 소스 준비
공유 다운로드·ZIP helper를 이 단계 전에 이미 정의했습니다. 저장된 원본의 실제 실패 지점(`download_http` NameError)을 수정했습니다.
기존 pinned 소스는 재사용하고, 필요한 commit이 다르면 기존 폴더를 백업합니다.

In [14]:
repositories = {
    "aasist": ("https://github.com/clovaai/aasist", "a04c9863f63d44471dde8a6abcb3b082b07cd1d1"),
    "rawboost": ("https://github.com/TakHemlata/RawBoost-antispoofing", "4f161a8b4d0d9f4a8431509ddd86c645869ef6c4"),
}
repository_requirements = {
    "aasist": [Path("models/AASIST.py"), Path("config/AASIST.conf")],
    "rawboost": [Path("RawBoost.py")],
}


def prepare_repository_snapshot(name, url, commit):
    destination = REPO_ROOT / name
    marker = destination / ".pinned_commit"
    required = repository_requirements[name]
    if marker.is_file() and marker.read_text(encoding="utf-8").strip() == commit and all(
        (destination / relative).is_file() for relative in required
    ):
        print("reuse repository:", name, commit[:12])
        return

    # 기존 git clone이 정확한 commit이면 재다운로드하지 않고 marker만 추가한다.
    if destination.is_dir() and shutil.which("git") and (destination / ".git").exists():
        try:
            current = subprocess.check_output(
                ["git", "-C", str(destination), "rev-parse", "HEAD"], text=True,
            ).strip()
        except Exception:
            current = ""
        if current == commit and all((destination / relative).is_file() for relative in required):
            marker.write_text(commit, encoding="utf-8")
            print("reuse existing git repository:", name, commit[:12])
            return

    if destination.exists():
        backup = REPO_ROOT / f"{name}_backup_{int(time.time())}"
        destination.rename(backup)
        print("기존 repository를 보존했습니다:", backup)

    archive_dir = REPO_ROOT / ".archives"
    archive_path = archive_dir / f"{name}-{commit}.zip"
    download_http(f"{url}/archive/{commit}.zip", archive_path)
    with tempfile.TemporaryDirectory(prefix=f"{name}_", dir=REPO_ROOT) as temporary:
        temporary_root = Path(temporary)
        safe_extract_zip(archive_path, temporary_root)
        extracted = [path for path in temporary_root.iterdir() if path.is_dir()]
        if len(extracted) != 1:
            raise RuntimeError(f"{name}: repository archive root가 하나가 아닙니다: {extracted}")
        shutil.copytree(extracted[0], destination)
    if not all((destination / relative).is_file() for relative in required):
        raise RuntimeError(f"{name}: 필요한 소스 파일이 없습니다.")
    marker.write_text(commit, encoding="utf-8")


repo_commits = {}
for name, (url, commit) in repositories.items():
    prepare_repository_snapshot(name, url, commit)
    repo_commits[name] = commit
print(repo_commits)

rawboost_path = REPO_ROOT / "rawboost" / "RawBoost.py"
rawboost_spec = importlib.util.spec_from_file_location("official_rawboost", rawboost_path)
RAWBOOST = importlib.util.module_from_spec(rawboost_spec)
sys.modules[rawboost_spec.name] = RAWBOOST
assert rawboost_spec.loader is not None
rawboost_spec.loader.exec_module(RAWBOOST)


def decode_audio(path):
    return decode_audio_rate(path, CFG.sample_rate)


def crop_or_repeat(waveform, rng, training, target_samples=None):
    target_samples = int(target_samples or CFG.clip_samples)
    waveform = waveform.float().reshape(-1)
    if target_samples<=0 or waveform.numel()==0: raise ValueError("empty waveform / invalid target length")
    if not torch.isfinite(waveform).all(): raise ValueError("non-finite waveform")
    if waveform.numel()<target_samples:
        waveform=waveform.repeat(math.ceil(target_samples/waveform.numel()))
    maximum_start=waveform.numel()-target_samples
    start=rng.randint(0,maximum_start) if training and maximum_start else maximum_start//2
    return waveform[start:start+target_samples].contiguous().clone()


reuse repository: aasist a04c9863f63d
reuse repository: rawboost 4f161a8b4d0d
{'aasist': 'a04c9863f63d44471dde8a6abcb3b082b07cd1d1', 'rawboost': '4f161a8b4d0d9f4a8431509ddd86c645869ef6c4'}


In [15]:
# Windows DataLoader가 RawBoost 모듈을 다시 불러오는 것을 방지
CFG.num_workers = 0


def _rawboost_scalar_rand_range(x1, x2, integer):
    value = float(np.random.uniform(low=x1, high=x2))
    return int(value) if bool(integer) else value


# RawBoost 모듈 내부 전역 함수를 직접 교체
RAWBOOST.__dict__["randRange"] = _rawboost_scalar_rand_range

# 패치 확인
integer_probe = RAWBOOST.randRange(10, 100, True)
float_probe = RAWBOOST.randRange(0.0, 1.0, False)

print("적용된 함수:", RAWBOOST.randRange.__name__)
print("integer probe:", integer_probe, type(integer_probe))
print("float probe:", float_probe, type(float_probe))

assert RAWBOOST.randRange.__name__ == "_rawboost_scalar_rand_range"
assert isinstance(integer_probe, int)
assert np.isscalar(float_probe)

# 실제로 오류가 발생했던 필터 생성까지 검사
numpy_state = np.random.get_state()

try:
    coefficient_probe = RAWBOOST.genNotchCoeffs(
        nBands=1,
        minF=20,
        maxF=3000,
        minBW=100,
        maxBW=500,
        minCoeff=10,
        maxCoeff=20,
        minG=0,
        maxG=0,
        fs=16000,
    )
finally:
    np.random.set_state(numpy_state)

assert np.asarray(coefficient_probe).ndim == 1
assert np.isfinite(coefficient_probe).all()

print("RawBoost NumPy 2.x compatibility: OK")

적용된 함수: _rawboost_scalar_rand_range
integer probe: 43 <class 'int'>
float probe: 0.9507143064099162 <class 'float'>
RawBoost NumPy 2.x compatibility: OK


## 10. 동적 믹싱 / 출처 편향 감사
원본의 augmentation 정책을 유지합니다. source 선택 RNG를 오디오 처리 RNG와 분리해 branch 간 입력 길이가 달라도 같은 recipe가 같은 source를 선택하게 합니다.
빈 파일/NaN을 정상 label로 학습하지 않습니다. 검증 읽기 오류는 대체 source로 조용히 바꾸지 않고 중단합니다.

In [16]:
def component_targets(entries):
    if not entries: raise ValueError("empty components")
    vp = int(any(bool(e["contains_voice"]) for e in entries))
    mp = int(any(bool(e["contains_music"]) for e in entries))
    vf = int(any(bool(e["contains_voice"]) and int(e["voice_fake"]) == 1 for e in entries))
    mf = int(any(bool(e["contains_music"]) and int(e["music_fake"]) == 1 for e in entries))
    target = torch.tensor([max(vf,mf),vf,mf,vp,mp],dtype=torch.float32)
    mask = torch.tensor([1,vp,mp,1,1],dtype=torch.float32)
    return target, mask


BIAS_MITIGATION = SimpleNamespace(
    enabled=True,
    codec_p=0.45,
    eq_p=0.35,
    noise_p=0.08,
    target_db_min=-26.0,
    target_db_max=-18.0,
)
RUN_SOURCE_BIAS_AUDIT = False
SOURCE_BIAS_AUDIT_PER_POOL = 64
SOURCE_BIAS_AUDIT_PATH = MANIFEST_ROOT / "source_bias_audit.csv"


RECIPE_COMPONENTS = {
    "rv": ["real_voice"], "fv": ["fake_voice"],
    "rm": ["real_music"], "fm": ["fake_music"],
    "rv_rm": ["real_voice", "real_music"],
    "fv_rm": ["fake_voice", "real_music"],
    "rv_fm": ["real_voice", "fake_music"],
    "fv_fm": ["fake_voice", "fake_music"],
}


def force_exact_audio_length(waveform, target_samples):
    target_samples = int(target_samples)
    waveform = waveform.reshape(-1)
    if waveform.numel() < target_samples:
        waveform = F.pad(waveform, (0, target_samples - waveform.numel()))
    return waveform[:target_samples].contiguous()


def rms_normalize(waveform, target_db):
    rms = waveform.square().mean().clamp_min(1e-8).sqrt()
    target = 10 ** (target_db / 20)
    return waveform * (target / rms)


def common_codec_augment(waveform, rng):
    """원본 데이터셋과 무관한 공통 전송/코덱 프로필을 한 번 덧씌운다."""
    original_samples = waveform.numel()
    mode = rng.choices(
        ["clean", "telephone", "resample", "mulaw", "pcm", "bandlimit"],
        weights=[15, 15, 20, 15, 15, 20],
        k=1,
    )[0]
    if mode == "telephone":
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, 8000)
        encoded = torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), 256)
        waveform = torchaudio.functional.mu_law_decoding(encoded, 256)
        waveform = torchaudio.functional.resample(waveform, 8000, CFG.sample_rate)
    elif mode == "resample":
        intermediate_rate = rng.choice([10_000, 12_000, 22_050, 24_000])
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, intermediate_rate)
        waveform = torchaudio.functional.resample(waveform, intermediate_rate, CFG.sample_rate)
    elif mode == "mulaw":
        channels = rng.choice([128, 256, 512])
        encoded = torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), channels)
        waveform = torchaudio.functional.mu_law_decoding(encoded, channels)
    elif mode == "pcm":
        bits = rng.choice([8, 10, 12, 14])
        levels = float(2 ** (bits - 1) - 1)
        waveform = torch.round(waveform.clamp(-1, 1) * levels) / levels
    elif mode == "bandlimit":
        cutoff = rng.uniform(3400.0, 7600.0)
        waveform = torchaudio.functional.lowpass_biquad(waveform, CFG.sample_rate, cutoff)
        if rng.random() < 0.5:
            waveform = torchaudio.functional.highpass_biquad(
                waveform, CFG.sample_rate, rng.uniform(25.0, 120.0)
            )
    return force_exact_audio_length(waveform, original_samples), mode


def pre_mix_harmonize(waveform, rng, training):
    """길이 통일 후, 믹싱 전에 모든 source class에 같은 분포의 변형을 적용한다."""
    original_samples = waveform.numel()
    waveform = waveform.float().nan_to_num()
    waveform = waveform - waveform.mean()
    if BIAS_MITIGATION.enabled and training:
        if rng.random() < BIAS_MITIGATION.codec_p:
            waveform, _ = common_codec_augment(waveform, rng)
        if rng.random() < BIAS_MITIGATION.eq_p:
            center = rng.choice([125.0, 250.0, 500.0, 1000.0, 2000.0, 4000.0, 6500.0])
            waveform = torchaudio.functional.equalizer_biquad(
                waveform, CFG.sample_rate, center, rng.uniform(-6.0, 6.0), rng.uniform(0.5, 1.5)
            )
        if rng.random() < BIAS_MITIGATION.noise_p:
            power = waveform.square().mean().clamp_min(1e-8)
            snr_db = rng.uniform(22.0, 42.0)
            generator = torch.Generator().manual_seed(rng.randrange(2**31 - 1))
            noise = torch.randn(waveform.shape, generator=generator, dtype=waveform.dtype)
            waveform = waveform + noise * (power / 10 ** (snr_db / 10)).sqrt()
        target_db = rng.uniform(BIAS_MITIGATION.target_db_min, BIAS_MITIGATION.target_db_max)
    else:
        target_db = -22.0
    waveform = rms_normalize(waveform, target_db)
    waveform = waveform.nan_to_num().clamp(-1, 1)
    return force_exact_audio_length(waveform, original_samples)


def inspect_source_bias_row(record):
    path = str(record["path"])
    try:
        info = sf.info(path)
        original_rate = int(info.samplerate)
        channels = int(info.channels)
        codec = f"{info.format}|{info.subtype}"
    except Exception:
        original_rate = -1
        channels = -1
        codec = Path(path).suffix.lower() or "unknown"
    waveform = decode_audio(path)
    duration_seconds = waveform.numel() / CFG.sample_rate
    segment = crop_or_repeat(
        waveform, random.Random(stable_int(f"bias-audit|{record['source_id']}")), training=False
    )
    rms_db = float(20 * torch.log10(segment.square().mean().clamp_min(1e-12).sqrt()))
    return {
        "source_id": record["source_id"], "pool": record["pool"],
        "codec": codec, "original_sample_rate": original_rate, "channels": channels,
        "duration_seconds": duration_seconds, "center_rms_db": rms_db,
        "balance_group": record.get("balance_group", "unknown"),
    }


if RUN_SOURCE_BIAS_AUDIT:
    audit_rows = []
    for pool_name, pool_frame in source_manifest.groupby("pool", sort=False):
        records = pool_frame.to_dict("records")
        records = sorted(
            records, key=lambda row: stable_int(f"bias-audit-select|{CFG.seed}|{row['source_id']}")
        )[:SOURCE_BIAS_AUDIT_PER_POOL]
        for record in tqdm(records, desc=f"source bias audit: {pool_name}"):
            try:
                audit_rows.append(inspect_source_bias_row(record))
            except Exception as exc:
                print("bias audit skip:", record["path"], repr(exc))
    source_bias_audit = pd.DataFrame(audit_rows)
    if not source_bias_audit.empty:
        source_bias_audit.to_csv(SOURCE_BIAS_AUDIT_PATH, index=False, encoding="utf-8")
        display(source_bias_audit.groupby("pool")[[
            "original_sample_rate", "channels", "duration_seconds", "center_rms_db"
        ]].agg(["count", "mean", "std", "median"]).round(3))
        display(pd.crosstab(source_bias_audit["pool"], source_bias_audit["codec"], margins=True))
        print("saved:", SOURCE_BIAS_AUDIT_PATH)


def apply_temporal_layout(waveforms, rng, target_samples):
    waveforms = [force_exact_audio_length(waveform, target_samples) for waveform in waveforms]
    if len(waveforms) < 2:
        return waveforms, "single"
    choice = rng.random()
    if choice < 0.65:
        return waveforms, "overlap"
    length = int(target_samples)
    if choice < 0.85:
        result = []
        for waveform in waveforms:
            active = rng.randint(CFG.sample_rate, length)
            start = rng.randint(0, length - active)
            mask = torch.zeros(length)
            mask[start:start + active] = 1
            result.append(waveform * mask)
        return result, "partial"
    boundary = rng.randint(int(0.35 * length), int(0.65 * length))
    first_mask = torch.zeros(length); first_mask[:boundary] = 1
    second_mask = 1 - first_mask
    return [waveforms[0] * first_mask, waveforms[1] * second_mask], "sequential"


def communication_augment(waveform, rng):
    original_samples = waveform.numel()
    mode = rng.choice(["telephone", "mulaw", "noise", "gain_clip"])
    if mode == "telephone":
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, 8000)
        waveform = torchaudio.functional.resample(waveform, 8000, CFG.sample_rate)
    elif mode == "mulaw":
        encoded = torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), 256)
        waveform = torchaudio.functional.mu_law_decoding(encoded, 256)
    elif mode == "noise":
        power = waveform.square().mean().clamp_min(1e-8)
        snr_db = rng.uniform(12, 35)
        generator = torch.Generator().manual_seed(rng.randrange(2**31 - 1))
        noise = torch.randn(waveform.shape, generator=generator, dtype=waveform.dtype)
        waveform = waveform + noise * (power / 10 ** (snr_db / 10)).sqrt()
    else:
        waveform = waveform * 10 ** (rng.uniform(-8, 5) / 20)
        limit = rng.uniform(0.35, 0.95)
        waveform = waveform.clamp(-limit, limit) / limit
    return force_exact_audio_length(waveform, original_samples)


class DynamicMixDataset(Dataset):
    def __init__(self, recipes, sources, training=False, rawboost_p=0.0, communication_p=0.0, target_samples=None):
        self.recipes = recipes.reset_index(drop=True)
        self.training = training
        self.rawboost_p = rawboost_p
        self.communication_p = communication_p
        self.target_samples = int(target_samples or CFG.clip_samples)
        self.epoch = 0
        self.pools = {
            pool: frame.to_dict("records")
            for pool, frame in sources.groupby("pool", sort=False)
        }
        self.pool_groups = {}
        for pool, records in self.pools.items():
            groups = {}
            for record in records:
                group = str(record.get("balance_group", "unknown") or "unknown")
                groups.setdefault(group, []).append(record)
            self.pool_groups[pool] = [groups[name] for name in sorted(groups)]
        for pool in RECIPE_COMPONENTS.values():
            for name in pool:
                if not self.pools.get(name):
                    raise RuntimeError(f"empty source pool: {name}")

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.recipes)

    def _load_from_pool(self, pool_name, rng, source_rng):
        errors = []
        for _ in range(5 if self.training else 1):
            groups = self.pool_groups[pool_name]
            group_records = groups[source_rng.randrange(len(groups))]
            entry = group_records[source_rng.randrange(len(group_records))]
            try:
                waveform = crop_or_repeat(decode_audio(entry["path"]), rng, self.training, self.target_samples)
                waveform = pre_mix_harmonize(waveform, rng, self.training)
                waveform = force_exact_audio_length(waveform, self.target_samples)
                return waveform, entry
            except Exception as exc:
                errors.append(f"{entry['path']}: {repr(exc)}")
        raise RuntimeError(" | ".join(errors))

    def __getitem__(self, index):
        recipe = self.recipes.iloc[index]
        epoch = self.epoch if self.training else 0
        rng = random.Random(stable_int(f"mix|{recipe.seed}|{epoch}|{index}"))
        components = RECIPE_COMPONENTS[recipe.recipe_type]
        waveforms, entries = [], []
        try:
            for pool_name in components:
                source_rng = random.Random(stable_int(f"source|{recipe.seed}|{epoch}|{pool_name}"))
                waveform, entry = self._load_from_pool(pool_name, rng, source_rng)
                waveforms.append(waveform)
                entries.append((pool_name, entry))
        except Exception as exc:
            if not self.training: raise RuntimeError(f"validation source read failed: {recipe.recipe_id}") from exc
            return {
                "audio": torch.zeros(self.target_samples), "target": torch.zeros(5),
                "mask": torch.ones(5), "id": recipe.recipe_id,
                "recipe_type": recipe.recipe_type, "valid": torch.tensor(False),
                "error": repr(exc), "layout": "error",
            }

        if len(waveforms) == 2:
            snr_db = rng.uniform(-12, 12)
            waveforms[0] = rms_normalize(waveforms[0], -22 + snr_db / 2)
            waveforms[1] = rms_normalize(waveforms[1], -22 - snr_db / 2)
        else:
            waveforms[0] = rms_normalize(waveforms[0], rng.uniform(-27, -17))
        waveforms, layout = apply_temporal_layout(waveforms, rng, self.target_samples)
        waveforms = [force_exact_audio_length(waveform, self.target_samples) for waveform in waveforms]
        mixed = torch.stack(waveforms).sum(0)

        target, mask = component_targets([entry for _,entry in entries])

        if self.training and rng.random() < self.rawboost_p:
            with numpy_seed(stable_int(f"rawboost|{recipe.seed}|{epoch}")):
                values = mixed.numpy()
                values = RAWBOOST.LnL_convolutive_noise(
                    values, N_f=5, nBands=5, minF=20, maxF=8000,
                    minBW=100, maxBW=1000, minCoeff=10, maxCoeff=100,
                    minG=0, maxG=0, minBiasLinNonLin=5, maxBiasLinNonLin=20,
                    fs=CFG.sample_rate,
                )
                values = RAWBOOST.ISD_additive_noise(values, P=10, g_sd=2)
                mixed = torch.from_numpy(np.asarray(RAWBOOST.normWav(values, 0), dtype=np.float32))
        if self.training and rng.random() < self.communication_p:
            mixed = communication_augment(mixed, rng)

        mixed = force_exact_audio_length(mixed, self.target_samples)
        mixed = mixed - mixed.mean()
        peak = mixed.abs().max().clamp_min(1e-8)
        mixed = mixed / peak * (rng.uniform(0.65, 0.98) if self.training else 0.82)
        if not torch.isfinite(mixed).all():
            if not self.training: raise ValueError(f"non-finite validation sample: {recipe.recipe_id}")
            return {
                "audio": torch.zeros(self.target_samples), "target": target, "mask": mask,
                "id": recipe.recipe_id, "recipe_type": recipe.recipe_type,
                "valid": torch.tensor(False), "error": "NaN/Inf after mixing", "layout": layout,
            }
        return {
            "audio": force_exact_audio_length(mixed.float().clamp(-1, 1), self.target_samples), "target": target, "mask": mask,
            "id": recipe.recipe_id, "recipe_type": recipe.recipe_type,
            "valid": torch.tensor(True), "error": "", "layout": layout,
        }


source_by_split = {
    split_name: source_manifest[source_manifest["split"] == split_name].copy()
    for split_name in CFG.source_split_counts
}
recipe_by_split = {
    split_name: recipe_manifest[recipe_manifest["split"] == split_name].copy()
    for split_name in CFG.recipe_counts
}
preview_dataset = DynamicMixDataset(recipe_by_split["validation"].head(8), source_by_split["validation"])
preview_rows = []
for index in range(len(preview_dataset)):
    item = preview_dataset[index]
    preview_rows.append({"id": item["id"], "type": item["recipe_type"], **dict(zip(DACON_TRUTH_COLUMNS, item["target"].tolist()))})
display(pd.DataFrame(preview_rows))


,id,type,FILE_FAKE,VOICE_FAKE,MUSIC_FAKE,VOICE_PRESENT,MUSIC_PRESENT
0,validation_00000,rv_rm,0.0,0.0,0.0,1.0,1.0
1,validation_00001,fm,1.0,1.0,1.0,1.0,1.0
2,validation_00002,fv,1.0,1.0,0.0,1.0,0.0
3,validation_00003,rv_fm,1.0,1.0,1.0,1.0,1.0
4,validation_00004,fv_fm,1.0,1.0,1.0,1.0,1.0
5,validation_00005,rv_fm,1.0,1.0,1.0,1.0,1.0
6,validation_00006,rv,0.0,0.0,0.0,1.0,0.0
7,validation_00007,rv_fm,1.0,1.0,1.0,1.0,1.0


## 11. DACON score / 성분 mask
공식 평가식의 FAKE=1, EER `drop_intermediate=False`, voice/music 존재 mask를 유지합니다.
참고: https://dacon.io/competitions/official/236749/overview/evaluation (2026-09-09 확인)

In [17]:
OFFICIAL_METRIC_WEIGHTS = {
    "score_ads": 0.9,
    "score_cps": 0.1,
    "ads_file": 0.5,
    "ads_voice": 0.2,
    "ads_music": 0.3,
    "cps_voice_presence": 0.5,
    "cps_music_presence": 0.5,
}


def _official_binary_inputs(y_true, y_score, metric_name):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score, dtype=float)
    if y_true.shape != y_score.shape or y_true.size == 0:
        raise ValueError(f"{metric_name}: shape/empty input error")
    if not np.isfinite(y_true).all() or not np.isfinite(y_score).all():
        raise ValueError(f"{metric_name}: NaN/Inf input")
    if not np.isin(y_true,[0,1]).all(): raise ValueError(f"{metric_name}: label must be exactly 0/1")
    y_true = y_true.astype(int)
    if np.unique(y_true).size < 2:
        raise ValueError(f"{metric_name}: positive/negative classes are both required")
    return y_true, y_score


def equal_error_rate(y_true, y_score):
    y_true, y_score = _official_binary_inputs(y_true, y_score, "EER")
    # DACON 평가 페이지에 공개된 EER 구현과 동일합니다.
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1, drop_intermediate=False)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2
    return float(eer)


def official_roc_auc(y_true, y_score):
    y_true, y_score = _official_binary_inputs(y_true, y_score, "ROC-AUC")
    return float(roc_auc_score(y_true, y_score))


def dacon_official_score(y_true, y_pred):
    file_eer = equal_error_rate(y_true["FILE_FAKE"], y_pred["FILE_FAKE_PROB"])
    voice_mask = y_true["VOICE_PRESENT"].eq(1)
    music_mask = y_true["MUSIC_PRESENT"].eq(1)
    voice_eer = equal_error_rate(y_true.loc[voice_mask, "VOICE_FAKE"], y_pred.loc[voice_mask, "VOICE_FAKE_PROB"])
    music_eer = equal_error_rate(y_true.loc[music_mask, "MUSIC_FAKE"], y_pred.loc[music_mask, "MUSIC_FAKE_PROB"])
    voice_auc = official_roc_auc(y_true["VOICE_PRESENT"], y_pred["VOICE_PRESENT_PROB"])
    music_auc = official_roc_auc(y_true["MUSIC_PRESENT"], y_pred["MUSIC_PRESENT_PROB"])
    ads = (
        OFFICIAL_METRIC_WEIGHTS["ads_file"] * (1 - file_eer)
        + OFFICIAL_METRIC_WEIGHTS["ads_voice"] * (1 - voice_eer)
        + OFFICIAL_METRIC_WEIGHTS["ads_music"] * (1 - music_eer)
    )
    cps = (
        OFFICIAL_METRIC_WEIGHTS["cps_voice_presence"] * voice_auc
        + OFFICIAL_METRIC_WEIGHTS["cps_music_presence"] * music_auc
    )
    score = (
        OFFICIAL_METRIC_WEIGHTS["score_ads"] * ads
        + OFFICIAL_METRIC_WEIGHTS["score_cps"] * cps
    )
    return {
        "file_eer": file_eer, "voice_eer": voice_eer, "music_eer": music_eer,
        "voice_presence_auc": voice_auc, "music_presence_auc": music_auc,
        "ads": ads, "cps": cps, "score": score,
    }


def masked_multitask_loss(logits, targets, masks):
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    weights = HEAD_WEIGHTS.to(logits.device).unsqueeze(0) * masks
    return (loss * weights).sum() / weights.sum().clamp_min(1e-8)


def report_from_arrays(targets, predictions):
    y_true = pd.DataFrame(targets, columns=DACON_TRUTH_COLUMNS)
    y_pred = pd.DataFrame(predictions, columns=DACON_PROBABILITY_COLUMNS)
    return dacon_official_score(y_true, y_pred)


## 12. 실제 codec roundtrip·reverb 및 stress view
함수 이름의 재정의는 원본의 최종 구현을 유지합니다. 짧은 신호의 reverb/빈 codec 결과를 방어합니다.

In [18]:
BIAS_MITIGATION.reverb_p = 0.08
BIAS_MITIGATION.clipping_p = 0.08
_FFMPEG_CODEC_FAILURES = 0


def ffmpeg_codec_roundtrip(waveform, rng):
    global _FFMPEG_CODEC_FAILURES
    choices = [
        ("mp3", "libmp3lame", bitrate) for bitrate in ("32k", "48k", "64k", "96k")
    ] + [
        ("adts", "aac", bitrate) for bitrate in ("32k", "48k", "64k", "96k")
    ] + [
        ("ogg", "libopus", bitrate) for bitrate in ("24k", "32k", "48k", "64k")
    ]
    container, encoder, bitrate = rng.choice(choices)
    raw = np.asarray(waveform.detach().cpu(), dtype="<f4").tobytes()
    try:
        encoded = subprocess.run(
            [FFMPEG_EXE, "-hide_banner", "-loglevel", "error", "-f", "f32le", "-ar", "16000",
             "-ac", "1", "-i", "pipe:0", "-c:a", encoder, "-b:a", bitrate, "-f", container, "pipe:1"],
            input=raw, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, timeout=30,
        ).stdout
        decoded = subprocess.run(
            [FFMPEG_EXE, "-hide_banner", "-loglevel", "error", "-i", "pipe:0", "-f", "f32le",
             "-ar", "16000", "-ac", "1", "pipe:1"],
            input=encoded, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, timeout=30,
        ).stdout
        result = torch.from_numpy(np.frombuffer(decoded, dtype="<f4").copy())
        if result.numel() == 0: raise ValueError("codec returned empty audio")
        if result.numel() < waveform.numel():
            result = result.repeat(math.ceil(waveform.numel() / max(1, result.numel())))
        result = result[:waveform.numel()]
        if result.numel() != waveform.numel() or not torch.isfinite(result).all():
            raise ValueError("invalid codec round-trip")
        return result, f"{encoder}:{bitrate}"
    except Exception as exc:
        _FFMPEG_CODEC_FAILURES += 1
        if _FFMPEG_CODEC_FAILURES <= 3:
            print("codec round-trip fallback:", repr(exc))
        return waveform, "lossy_fallback"


def common_codec_augment(waveform, rng):
    original_samples = waveform.numel()
    mode = rng.choices(
        ["clean", "telephone", "resample", "mulaw", "pcm", "bandlimit", "lossy"],
        weights=[25, 15, 15, 10, 10, 10, 15], k=1,
    )[0]
    if mode == "telephone":
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, 8000)
        encoded = torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), 256)
        waveform = torchaudio.functional.mu_law_decoding(encoded, 256)
        waveform = torchaudio.functional.resample(waveform, 8000, CFG.sample_rate)
    elif mode == "resample":
        intermediate_rate = rng.choice([10_000, 12_000, 22_050, 24_000])
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, intermediate_rate)
        waveform = torchaudio.functional.resample(waveform, intermediate_rate, CFG.sample_rate)
    elif mode == "mulaw":
        channels = rng.choice([128, 256, 512])
        waveform = torchaudio.functional.mu_law_decoding(
            torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), channels), channels
        )
    elif mode == "pcm":
        levels = float(2 ** (rng.choice([8, 10, 12, 14]) - 1) - 1)
        waveform = torch.round(waveform.clamp(-1, 1) * levels) / levels
    elif mode == "bandlimit":
        waveform = torchaudio.functional.lowpass_biquad(
            waveform, CFG.sample_rate, rng.uniform(3400.0, 7600.0)
        )
        if rng.random() < .5:
            waveform = torchaudio.functional.highpass_biquad(
                waveform, CFG.sample_rate, rng.uniform(25.0, 120.0)
            )
    elif mode == "lossy":
        waveform, mode = ffmpeg_codec_roundtrip(waveform, rng)
    return force_exact_audio_length(waveform, original_samples), mode


def light_reverb(waveform, rng):
    result = waveform.clone()
    for delay_ms in rng.sample([18, 31, 47, 71, 103, 137], k=3):
        delay = int(delay_ms * CFG.sample_rate / 1000)
        gain = rng.uniform(.06, .22) * rng.choice([-1, 1])
        if 0 < delay < waveform.numel(): result[delay:] += gain * waveform[:-delay]
    return result / result.abs().max().clamp_min(1.0)


def pre_mix_harmonize(waveform, rng, training):
    original_samples = waveform.numel()
    waveform = waveform.float().nan_to_num() - waveform.float().nan_to_num().mean()
    if BIAS_MITIGATION.enabled and training:
        if rng.random() < BIAS_MITIGATION.codec_p:
            waveform, _ = common_codec_augment(waveform, rng)
        if rng.random() < BIAS_MITIGATION.eq_p:
            waveform = torchaudio.functional.equalizer_biquad(
                waveform, CFG.sample_rate,
                rng.choice([125.0, 250.0, 500.0, 1000.0, 2000.0, 4000.0, 6500.0]),
                rng.uniform(-6.0, 6.0), rng.uniform(0.5, 1.5),
            )
        if rng.random() < BIAS_MITIGATION.noise_p:
            power = waveform.square().mean().clamp_min(1e-8)
            generator = torch.Generator().manual_seed(rng.randrange(2**31 - 1))
            noise = torch.randn(waveform.shape, generator=generator, dtype=waveform.dtype)
            waveform += noise * (power / 10 ** (rng.uniform(22.0, 42.0) / 10)).sqrt()
        if rng.random() < BIAS_MITIGATION.reverb_p:
            waveform = light_reverb(waveform, rng)
        if rng.random() < BIAS_MITIGATION.clipping_p:
            limit = rng.uniform(.45, .95)
            waveform = waveform.clamp(-limit, limit) / limit
        target_db = rng.uniform(BIAS_MITIGATION.target_db_min, BIAS_MITIGATION.target_db_max)
    else:
        target_db = -22.0
    waveform = rms_normalize(waveform, target_db).nan_to_num().clamp(-1, 1)
    return force_exact_audio_length(waveform, original_samples)

class DomainStressView(Dataset):
    def __init__(self, base_dataset, enabled):
        self.base = base_dataset
        self.enabled = bool(enabled)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, index):
        item = self.base[index]
        if not self.enabled or not bool(item["valid"]):
            return item
        rng = random.Random(stable_int(f"domain-stress|{CFG.seed}|{item['id']}"))
        audio = item["audio"].clone()
        # 학습 augmentation과 다른 조합 강도. 결과는 index별로 항상 동일하다.
        audio, codec_mode = common_codec_augment(audio, rng)
        if rng.random() < 0.70:
            center = rng.choice([180.0, 350.0, 750.0, 1500.0, 3000.0, 6000.0])
            audio = torchaudio.functional.equalizer_biquad(
                audio, CFG.sample_rate, center, rng.uniform(-8.0, 8.0), rng.uniform(0.4, 1.8)
            )
        if rng.random() < 0.45:
            audio = communication_augment(audio, rng)
        if rng.random() < 0.35:
            power = audio.square().mean().clamp_min(1e-8)
            snr_db = rng.uniform(10.0, 28.0)
            generator = torch.Generator().manual_seed(rng.randrange(2**31 - 1))
            noise = torch.randn(audio.shape, generator=generator, dtype=audio.dtype)
            audio = audio + noise * (power / 10 ** (snr_db / 10)).sqrt()
        audio = rms_normalize(audio - audio.mean(), rng.uniform(-27.0, -17.0))
        limit = rng.uniform(0.55, 0.98)
        audio = audio.clamp(-limit, limit) / limit
        result = dict(item)
        result["audio"] = force_exact_audio_length(
            audio.nan_to_num().float().clamp(-1, 1), self.base.target_samples
        )
        result["layout"] = f"stress:{codec_mode}"
        return result


validation_base = DynamicMixDataset(
    recipe_by_split["validation"], source_by_split["validation"], training=False
)
validation_clean = DomainStressView(validation_base, enabled=False)
validation_stress = DomainStressView(validation_base, enabled=True)
print("validation views:", len(validation_clean), len(validation_stress))


validation views: 2500 2500


## 13. 원본 Head-Specialist 모델 / 8GB 보수적 설정
구조는 유지합니다. 작은 batch + accumulation, WavLM 동결 구간 eval 모드, 추론은 CPU 보관 후 모델별 GPU 이동이 가능합니다.
`MODEL_SOURCE`를 제출에도 그대로 포함해 학습 모델 정의와 제출 모델 정의의 수동 복사 차이를 없앴습니다.

In [19]:
MODEL_SOURCE = 'from pathlib import Path\nimport hashlib, importlib.util, json, sys\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torchaudio\n\nclass PresenceLogMel(nn.Module):\n    def __init__(self, dropout=0.20):\n        super().__init__()\n        self.mel = torchaudio.transforms.MelSpectrogram(\n            sample_rate=16000, n_fft=1024, win_length=400,\n            hop_length=160, n_mels=96, f_min=20, f_max=7600,\n        )\n        self.encoder = nn.Sequential(\n            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.SiLU(), nn.MaxPool2d(2),\n            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.SiLU(), nn.MaxPool2d(2),\n            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.SiLU(), nn.MaxPool2d(2),\n            nn.Conv2d(128, 192, 3, padding=1), nn.BatchNorm2d(192), nn.SiLU(),\n            nn.AdaptiveAvgPool2d(1), nn.Flatten(),\n        )\n        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(192, 2))\n\n    def forward(self, audio):\n        feature = torch.log(self.mel(audio).clamp_min(1e-6))\n        feature = (feature - feature.mean((-2, -1), keepdim=True)) / (\n            feature.std((-2, -1), keepdim=True) + 1e-5\n        )\n        return self.head(self.encoder(feature.unsqueeze(1)))\n\n\ndef build_aasist_aux3(repo):\n    repo = Path(repo)\n    with (repo / "config" / "AASIST.conf").open(encoding="utf-8") as file:\n        model_config = json.load(file)["model_config"]\n    module_path = repo / "models" / "AASIST.py"\n    module_name = "_aasist_" + hashlib.sha256(str(module_path.resolve()).encode()).hexdigest()[:12]\n    spec = importlib.util.spec_from_file_location(module_name,module_path)\n    module = importlib.util.module_from_spec(spec);sys.modules[module_name]=module;spec.loader.exec_module(module)\n    OfficialAASIST = module.Model\n\n    class AASISTAux3(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.net = OfficialAASIST(model_config)\n            self.net.out_layer = nn.Linear(self.net.out_layer.in_features, 3)\n\n        def forward(self, audio):\n            return self.net(audio, Freq_aug=False)[1]\n\n    return AASISTAux3()\n\n\nWAVLM_MODEL = "microsoft/wavlm-base-plus"\nWAVLM_REVISION = "4c66d4806a428f2e922ccfa1a962776e232d487b"\n\n\nclass VoiceWavLM(nn.Module):\n    def __init__(self, freeze=True, dropout=0.20, config_path=None, pretrained=True):\n        super().__init__()\n        from transformers import AutoModel, AutoConfig\n        if pretrained:\n            self.ssl = AutoModel.from_pretrained(WAVLM_MODEL, revision=WAVLM_REVISION)\n        else:\n            if config_path is None: raise ValueError("offline WavLM requires local config_path")\n            config = AutoConfig.from_pretrained(str(config_path),local_files_only=True)\n            self.ssl = AutoModel.from_config(config)\n        self.ssl.config.layerdrop = 0.0\n        self.ssl.config.apply_spec_augment = False\n        hidden = self.ssl.config.hidden_size\n        self.attention = nn.Sequential(nn.Linear(hidden, 128), nn.Tanh(), nn.Linear(128, 1))\n        self.head = nn.Sequential(\n            nn.LayerNorm(hidden * 2), nn.Dropout(dropout),\n            nn.Linear(hidden * 2, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1),\n        )\n        if freeze:\n            for parameter in self.ssl.parameters():\n                parameter.requires_grad = False\n\n    def train(self, mode=True):\n        super().train(mode)\n        if not any(p.requires_grad for p in self.ssl.parameters()): self.ssl.eval()\n        return self\n\n    def unfreeze_last(self, count=4):\n        for parameter in self.ssl.parameters():\n            parameter.requires_grad = False\n        for layer in self.ssl.encoder.layers[-count:]:\n            for parameter in layer.parameters():\n                parameter.requires_grad = True\n\n    def forward(self, audio):\n        audio = (audio - audio.mean(1, keepdim=True)) / (audio.std(1, keepdim=True) + 1e-5)\n        if any(parameter.requires_grad for parameter in self.ssl.parameters()):\n            hidden = self.ssl(audio).last_hidden_state\n        else:\n            with torch.no_grad():\n                hidden = self.ssl(audio).last_hidden_state\n        hidden = hidden.float()\n        weight = torch.softmax(self.attention(hidden).float(), dim=1)\n        mean = (hidden * weight).sum(1)\n        variance = ((hidden - mean[:, None]) ** 2 * weight).sum(1).clamp_min(1e-6)\n        return self.head(torch.cat([mean, variance.sqrt()], dim=-1))\n\n\nclass MusicSegmentTransformer(nn.Module):\n    """16초 log-mel을 짧은 time token으로 압축한 뒤 전역 구조를 모델링한다."""\n    def __init__(self, dropout=0.20, dim=192, layers=3):\n        super().__init__()\n        self.mel = torchaudio.transforms.MelSpectrogram(\n            sample_rate=16000, n_fft=1024, win_length=1024,\n            hop_length=320, n_mels=128, f_min=20, f_max=7800,\n        )\n        self.stem = nn.Sequential(\n            nn.Conv2d(1, 48, 5, stride=(2, 2), padding=2), nn.BatchNorm2d(48), nn.GELU(),\n            nn.Conv2d(48, 96, 3, stride=(2, 2), padding=1), nn.BatchNorm2d(96), nn.GELU(),\n            nn.Conv2d(96, dim, 3, stride=(2, 2), padding=1), nn.BatchNorm2d(dim), nn.GELU(),\n        )\n        block = nn.TransformerEncoderLayer(\n            d_model=dim, nhead=6, dim_feedforward=dim * 4, dropout=dropout,\n            activation="gelu", batch_first=True, norm_first=True,\n        )\n        self.transformer = nn.TransformerEncoder(block, num_layers=layers, enable_nested_tensor=False)\n        self.position = nn.Parameter(torch.zeros(1, 160, dim))\n        nn.init.trunc_normal_(self.position, std=0.02)\n        self.head = nn.Sequential(\n            nn.LayerNorm(dim * 2), nn.Dropout(dropout),\n            nn.Linear(dim * 2, dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim, 1),\n        )\n\n    def forward(self, audio):\n        feature = torch.log(self.mel(audio).clamp_min(1e-6))\n        feature = (feature - feature.mean((-2, -1), keepdim=True)) / (\n            feature.std((-2, -1), keepdim=True) + 1e-5\n        )\n        tokens = self.stem(feature.unsqueeze(1)).mean(2).transpose(1, 2)\n        if tokens.shape[1] > self.position.shape[1]:\n            tokens = F.adaptive_avg_pool1d(tokens.transpose(1, 2), self.position.shape[1]).transpose(1, 2)\n        tokens = self.transformer(tokens + self.position[:, :tokens.shape[1]])\n        return self.head(torch.cat([tokens.mean(1), tokens.amax(1)], dim=-1))\n\n\n'
exec(compile(MODEL_SOURCE,"runtime_models.py","exec"), globals())

MODEL_CONFIGS = {
    "presence": dict(task="presence", batch=8, eval_batch=8, grad_accum=4, epochs=10,
                     lr=3e-4, backbone_lr=3e-4, patience=4, rawboost_p=0.03,
                     communication_p=0.08, dropout=0.20, clip_samples=CFG.clip_samples),
    "aasist_aux3": dict(task="aasist", batch=4, eval_batch=4, grad_accum=4, epochs=14,
                        lr=1e-4, backbone_lr=1e-4, patience=5, rawboost_p=0.20,
                        communication_p=0.10, dropout=0.20, ranking_weight=0.08,
                        clip_samples=CFG.clip_samples),
    "voice_wavlm": dict(task="voice", batch=1, eval_batch=1, grad_accum=16, epochs=8,
                        lr=8e-5, backbone_lr=2e-6, patience=3, freeze_epochs=2,
                        unfreeze_last=4, rawboost_p=0.18, communication_p=0.12,
                        dropout=0.20, ranking_weight=0.08, clip_samples=CFG.clip_samples),
    "music_long": dict(task="music", batch=2, eval_batch=2, grad_accum=8, epochs=12,
                       lr=2e-4, backbone_lr=2e-4, patience=4, rawboost_p=0.05,
                       communication_p=0.05, dropout=0.20, ranking_weight=0.08,
                       clip_samples=CFG.music_clip_samples),
}
# microbatch=1이면 batch 안에 양/음 예제가 함께 없어 pairwise loss가 0일 수 있습니다.
# accumulation은 여러 microbatch의 ranking pair를 합쳐주지 않습니다.
print('8GB 보수적 설정. OOM이면 batch를 더 줄이고 실제 GPU 테스트가 필요합니다.')
display(pd.DataFrame(MODEL_CONFIGS).T)


8GB 보수적 설정. OOM이면 batch를 더 줄이고 실제 GPU 테스트가 필요합니다.


,task,batch,eval_batch,grad_accum,epochs,lr,backbone_lr,patience,rawboost_p,communication_p,dropout,clip_samples,ranking_weight,freeze_epochs,unfreeze_last
presence,presence,8,8,4,10,0.0003,0.0003,4,0.03,0.08,0.2,64600,NaN,NaN,NaN
aasist_aux3,aasist,4,4,4,14,0.0001,0.0001,5,0.2,0.1,0.2,64600,0.08,NaN,NaN
voice_wavlm,voice,1,1,16,8,0.00008,0.000002,3,0.18,0.12,0.2,64600,0.08,2,4
music_long,music,2,2,8,12,0.0002,0.0002,4,0.05,0.05,0.2,256000,0.08,NaN,NaN


## 14. Branch별 loss / epoch 실행
NaN/Inf logits·loss를 검출하고 AMP gradient overflow는 해당 update를 건너뛰며 scale을 낮춥니다. 검증 실패 파일은 건너뛰지 않습니다. 유효 label이 없는 batch는 optimizer를 진행하지 않습니다.
마지막 불완전 accumulation도 실제 누적 횟수로 나눠 처리합니다. 예측 확률은 FP32 sigmoid로 기록합니다.

In [20]:
def pairwise_ranking_loss(logits, targets, masks, margin=0.25):
    losses = []
    for head in range(logits.shape[1]):
        valid = masks[:, head].bool()
        positive = logits[valid & targets[:, head].eq(1), head]
        negative = logits[valid & targets[:, head].eq(0), head]
        if positive.numel() and negative.numel():
            losses.append(F.softplus(margin - (positive[:, None] - negative[None, :])).mean())
    return torch.stack(losses).mean() if losses else logits.sum() * 0


def task_targets(batch, task):
    if task == "presence":
        return batch["target"][:, 3:5], torch.ones_like(batch["target"][:, 3:5])
    if task == "aasist":
        return batch["target"][:, :3], batch["mask"][:, :3]
    index = {"voice": 1, "music": 2}[task]
    return batch["target"][:, index:index + 1], batch["mask"][:, index:index + 1]


def task_loss(logits, targets, masks, task, ranking_weight=0.0):
    logits = logits.float()
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    if task == "aasist":
        weights = torch.tensor([.5,.2,.3],device=logits.device)
        den = masks.sum(0)
        head_loss = (bce*masks).sum(0)/den.clamp_min(1)
        weights = weights * (den>0)
        value = (head_loss*weights).sum()/weights.sum().clamp_min(1e-8)
    else:
        value = (bce * masks).sum() / masks.sum().clamp_min(1)
    if task != "presence" and ranking_weight:
        value = value + ranking_weight * pairwise_ranking_loss(logits, targets, masks)
    return value


def fake_report(targets, predictions, masks):
    report = {}
    for head, name in enumerate(("file_eer", "voice_eer", "music_eer")):
        valid = masks[:, head].astype(bool)
        report[name] = equal_error_rate(targets[valid, head], predictions[valid, head])
    report["ads"] = .5 * (1 - report["file_eer"]) + .2 * (1 - report["voice_eer"]) + .3 * (1 - report["music_eer"])
    return report


def specialist_report(targets, predictions, masks, task):
    valid = masks[:, 0].astype(bool)
    eer = equal_error_rate(targets[valid, 0], predictions[valid, 0])
    auc = official_roc_auc(targets[valid, 0], predictions[valid, 0])
    return {f"{task}_eer": eer, f"{task}_auc": auc, f"{task}_quality": 1 - eer}


def presence_report(targets, predictions):
    voice_auc = official_roc_auc(targets[:, 0], predictions[:, 0])
    music_auc = official_roc_auc(targets[:, 1], predictions[:, 1])
    return {"voice_presence_auc": voice_auc, "music_presence_auc": music_auc,
            "cps": .5 * voice_auc + .5 * music_auc}


def build_branch(name, pretrained=True):
    if name == "presence": return PresenceLogMel(MODEL_CONFIGS[name]["dropout"])
    if name == "aasist_aux3": return build_aasist_aux3(REPO_ROOT / "aasist")
    if name == "voice_wavlm": return VoiceWavLM(dropout=MODEL_CONFIGS[name]["dropout"], pretrained=pretrained, config_path=RUN_ROOT / "wavlm_config")
    if name == "music_long": return MusicSegmentTransformer(dropout=MODEL_CONFIGS[name]["dropout"])
    raise KeyError(name)


def make_branch_loaders(config):
    target_samples = int(config["clip_samples"])
    train_dataset = DynamicMixDataset(
        recipe_by_split["train"], source_by_split["train"], training=True,
        rawboost_p=config["rawboost_p"], communication_p=config["communication_p"],
        target_samples=target_samples,
    )
    validation_base_for_branch = DynamicMixDataset(
        recipe_by_split["validation"], source_by_split["validation"], training=False,
        target_samples=target_samples,
    )
    clean = DomainStressView(validation_base_for_branch, enabled=False)
    stress = DomainStressView(validation_base_for_branch, enabled=True)
    common = dict(num_workers=CFG.num_workers, pin_memory=DEVICE.type == "cuda", persistent_workers=False)
    return (
        train_dataset,
        DataLoader(train_dataset, batch_size=config["batch"], shuffle=True, drop_last=True, **common),
        DataLoader(clean, batch_size=config["eval_batch"], shuffle=False, **common),
        DataLoader(stress, batch_size=config["eval_batch"], shuffle=False, **common),
    )



def run_branch_loader(model, loader, config, optimizer=None, scheduler=None, scaler=None, description="eval"):
    training = optimizer is not None
    model.train(training)
    if training and scaler is None: raise ValueError('training requires GradScaler(enabled=False on CPU)')
    if training: optimizer.zero_grad(set_to_none=True)
    total_loss=count=skipped=supervised_updates=pending=0
    all_targets,all_predictions,all_masks,all_ids=[],[],[],[]

    def step_optimizer(number):
        scaler.unscale_(optimizer)
        for parameter in model.parameters():
            if parameter.grad is not None: parameter.grad.div_(number)
        # AMP overflow는 GradScaler가 해당 update를 건너뛰고 scale을 낮추게 합니다.
        finite = all(torch.isfinite(p.grad).all().item() for p in model.parameters() if p.grad is not None)
        if not finite and not scaler.is_enabled():
            raise FloatingPointError('FP32 gradient NaN/Inf. 데이터/손실을 확인하세요.')
        if finite:
            nn.utils.clip_grad_norm_(model.parameters(),5.0,error_if_nonfinite=True)
        previous=scaler.get_scale();scaler.step(optimizer);scaler.update()
        optimizer.zero_grad(set_to_none=True)
        updated=finite and scaler.get_scale()>=previous
        if scheduler is not None and updated: scheduler.step()
        return int(updated)

    for batch in tqdm(loader,desc=description,dynamic_ncols=True):
        valid=batch['valid'].bool(); skipped+=int((~valid).sum())
        if not training and not valid.all():
            raise RuntimeError(f'{description}: 검증 샘플 읽기 실패: {batch.get("error")}')
        if not valid.any(): continue
        audio=batch['audio'][valid].to(DEVICE)
        raw_targets,raw_masks=task_targets(batch,config['task'])
        targets=raw_targets[valid].to(DEVICE);masks=raw_masks[valid].to(DEVICE)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=DEVICE.type,dtype=AMP_DTYPE,enabled=USE_AMP):
                logits=model(audio)
                unscaled=task_loss(logits,targets,masks,config['task'],config.get('ranking_weight',0.0))
            if not torch.isfinite(logits).all() or not torch.isfinite(unscaled):
                raise FloatingPointError(f'{description}: NaN/Inf. AMP off 비교 및 해당 음원 확인')
            if training and masks.sum()>0:
                scaler.scale(unscaled).backward();pending+=1
                if pending==config['grad_accum']:
                    supervised_updates+=step_optimizer(pending);pending=0
        batch_count=int(targets.shape[0]);count+=batch_count
        total_loss+=float(unscaled.detach().cpu())*batch_count
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(torch.sigmoid(logits.detach().float()).cpu().numpy())
        all_masks.append(masks.detach().cpu().numpy())
        all_ids.extend([v for v,keep in zip(batch['id'],valid.tolist()) if keep])
    if training and pending: supervised_updates+=step_optimizer(pending)
    if not all_targets: raise RuntimeError(f'{description}: 유효 데이터 없음')
    if skipped/max(1,skipped+count)>CFG.max_invalid_train_fraction:
        raise RuntimeError(f'{description}: 실패 샘플 {skipped}/{count+skipped}. 데이터 감사 필요')
    targets=np.concatenate(all_targets);predictions=np.concatenate(all_predictions);masks=np.concatenate(all_masks)
    if config['task']=='presence': report=presence_report(targets,predictions)
    elif config['task']=='aasist': report=fake_report(targets,predictions,masks)
    else: report=specialist_report(targets,predictions,masks,config['task'])
    report.update(loss=total_loss/count,samples=count,skipped=skipped,optimizer_updates=supervised_updates)
    return report,targets,predictions,masks,all_ids


## 15. 실제 장치에서 1-batch 모델 검사 (전체 학습 아님)
학습 전 네 모델을 순서대로 구성/실행하고 바로 해제합니다. WavLM은 마지막 학습 대상 layer를 해제해 검사하며, 원본 다운로드가 이 시점에 필요합니다.
8GB GPU에 맞지 않거나 CUDA 빌드가 지원되지 않으면 여기에서 실제 오류를 확인하세요.
이 테스트 통과도 전체 학습이나 L4 제출 시간 성공을 보장하지 않습니다.

In [21]:
if RUN_MODEL_SELF_TEST:
    for name in BRANCHES_TO_TRAIN:
        model=build_branch(name).to(DEVICE)
        if name=='voice_wavlm':
            (RUN_ROOT/'wavlm_config').mkdir(parents=True,exist_ok=True)
            model.ssl.config.save_pretrained(RUN_ROOT/'wavlm_config')
            model.unfreeze_last(MODEL_CONFIGS[name].get('unfreeze_last',4))
        config=MODEL_CONFIGS[name]
        try:
            model.train()
            dummy=torch.randn(config['batch'],config['clip_samples'],device=DEVICE)*.05
            with torch.autocast(device_type=DEVICE.type,dtype=AMP_DTYPE,enabled=USE_AMP):
                result=model(dummy)
                loss=result.float().square().mean()
            assert result.shape==(config['batch'],{'presence':2,'aasist':3,'voice':1,'music':1}[config['task']])
            if not torch.isfinite(result).all(): raise FloatingPointError(name)
            loss.backward()
            print(name,'shape',tuple(result.shape),'forward/backward OK')
            del dummy,result,loss
        finally:
            del model;gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
else:
    print('RUN_MODEL_SELF_TEST=False: actual model forward/backward 미검사')


RUN_MODEL_SELF_TEST=False: actual model forward/backward 미검사


## 16. 학습 및 안전한 epoch 재개
기존 runs는 재사용하지 않습니다. 수정본 안에서도 설정/manifest/pipeline signature가 다르면 재개를 중단합니다.
Epoch마다 RNG를 고정하여 단순 재개시 샘플 순서의 변화를 줄입니다. 완벽한 bitwise 재현성을 보증하지 않습니다.
기존 warm-start는 기본 비활성화이며 켜더라도 `out_layer`는 제외합니다.

In [22]:
USE_LEGACY_WARMSTART = False
LEGACY_AASIST_BEST = PROJECT_ROOT/'legacy'/'deepvoice_domainrobust_v3'/'runs'/'aasist_fake3'/'best.pt'

def training_signature(config):
    payload = dict(version=PIPELINE_VERSION,config=config,source=file_hash(SOURCE_MANIFEST_PATH),
                   component=file_hash(COMPONENT_MANIFEST_PATH),recipe=file_hash(RECIPE_MANIFEST_PATH),
                   model_source=hashlib.sha256(MODEL_SOURCE.encode()).hexdigest(),repo=repo_commits)
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()

def warm_start_legacy_aasist(model):
    if not USE_LEGACY_WARMSTART or not LEGACY_AASIST_BEST.exists():
        print("legacy AASIST checkpoint 없음: 새로 학습")
        return
    state = torch.load(LEGACY_AASIST_BEST, map_location="cpu", weights_only=False)["model_state"]
    current = model.state_dict()
    compatible = {key: value for key, value in state.items() if key in current and current[key].shape == value.shape and "out_layer" not in key}
    result = model.load_state_dict(compatible, strict=False)
    print("legacy AASIST warm-start:", len(compatible), "keys; new/missing:", len(result.missing_keys))


def save_validation_frame(name, view, result, task):
    report, targets, predictions, masks, identifiers = result
    frame = pd.DataFrame({"ID": identifiers})
    if task == "presence": prefixes = ["VOICE_PRESENT", "MUSIC_PRESENT"]
    elif task == "aasist": prefixes = ["FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE"]
    else: prefixes = [task.upper() + "_FAKE"]
    for index, column in enumerate(prefixes):
        frame[column] = targets[:, index]
        frame[column + "_PROB"] = predictions[:, index]
        frame[column + "_MASK"] = masks[:, index]
    atomic_csv(frame, RUN_ROOT / name / f"validation_{view}_predictions.csv")


def fit_branch(name):
    global USE_AMP

    # Presence/AASIST/Voice WavLM은 AMP,
    # NaN이 발생한 Music Long만 FP32로 학습
    USE_AMP = DEVICE.type == "cuda" and name != "music_long"

    print(
        f"{name} precision:",
        "AMP FP16" if USE_AMP else "FP32",
    )

    from transformers import get_cosine_schedule_with_warmup

    config = dict(MODEL_CONFIGS[name]); run_dir = RUN_ROOT / name; run_dir.mkdir(parents=True, exist_ok=True)
    train_dataset, train_loader, clean_loader, stress_loader = make_branch_loaders(config)
    signature = training_signature(config)
    seed_everything(stable_int(f"init|{CFG.seed}|{name}") % 2**31)
    model = build_branch(name).to(DEVICE)
    if name == "voice_wavlm": model.ssl.config.save_pretrained(RUN_ROOT / "wavlm_config")
    if name == "aasist_aux3" and not (run_dir / "best.pt").exists(): warm_start_legacy_aasist(model)
    backbone, head = [], []
    for parameter_name, parameter in model.named_parameters():
        (backbone if parameter_name.startswith("ssl.") else head).append(parameter)
    groups = []
    if backbone: groups.append({"params": backbone, "lr": config["backbone_lr"]})
    if head: groups.append({"params": head, "lr": config["lr"]})
    optimizer = torch.optim.AdamW(groups, weight_decay=1e-4)
    updates = math.ceil(len(train_loader) / config["grad_accum"]) * config["epochs"]
    scheduler = get_cosine_schedule_with_warmup(optimizer, max(10, int(updates * .08)), updates)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best_score, stale, history, start_epoch = -float("inf"), 0, [], 1
    last_path = run_dir / "last.pt"
    if RESUME_TRAINING and last_path.exists():
        state = torch.load(last_path, map_location="cpu", weights_only=False)
        if state.get("signature") != signature:
            raise RuntimeError(f"{name}: checkpoint/config/manifest 불일치. 새 RUN_ROOT 사용 또는 원래 설정 복원")
        if state.get("config") == config:
            model.load_state_dict(state["model_state"], strict=True); optimizer.load_state_dict(state["optimizer_state"])
            scheduler.load_state_dict(state["scheduler_state"]); scaler.load_state_dict(state["scaler_state"])
            best_score, stale = float(state["best_score"]), int(state["stale"])
            history, start_epoch = list(state["history"]), int(state["epoch"]) + 1
            if state.get("finished"):
                if not (run_dir/"best.pt").is_file(): raise FileNotFoundError(run_dir/"best.pt")
                del state, model, optimizer, scheduler, scaler;gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                return pd.DataFrame(history)
    if hasattr(model, "unfreeze_last") and start_epoch > config.get("freeze_epochs", float("inf")):
        model.unfreeze_last(config.get("unfreeze_last", 4))
    for epoch in range(start_epoch, config["epochs"] + 1):
        seed_everything(stable_int(f"epoch|{CFG.seed}|{name}|{epoch}") % 2**31)
        started = time.time(); train_dataset.set_epoch(epoch)
        if epoch == config.get("freeze_epochs", -1) + 1 and hasattr(model, "unfreeze_last"):
            model.unfreeze_last(config.get("unfreeze_last", 4))
        train_report, *_ = run_branch_loader(model, train_loader, config, optimizer, scheduler, scaler, f"{name} train {epoch}")
        with torch.no_grad():
            clean = run_branch_loader(model, clean_loader, config, description=f"{name} clean val")
            stress = run_branch_loader(model, stress_loader, config, description=f"{name} stress val")
        metric_key = {"presence": "cps", "aasist": "ads", "voice": "voice_quality", "music": "music_quality"}[config["task"]]
        robust_score = .5 * clean[0][metric_key] + .5 * stress[0][metric_key]
        record = {
            "epoch": epoch,
            "train_loss": train_report["loss"],
            "train_samples": train_report["samples"],
            "train_skipped": train_report["skipped"],
            "train_optimizer_updates": train_report["optimizer_updates"],
            "robust_score": robust_score,
            **{f"clean_{key}": value for key, value in clean[0].items()},
            **{f"stress_{key}": value for key, value in stress[0].items()},
            "minutes": (time.time() - started) / 60,
        }
        history.append(record); pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False); print(record)
        if robust_score > best_score:
            best_score, stale = robust_score, 0
            atomic_torch_save({"model_state": model.state_dict(), "name": name, "config": config, "signature": signature,
                        "epoch": epoch, "robust_score": robust_score, "repo_commits": repo_commits,
                        "source_manifest_sha256": hashlib.sha256(SOURCE_MANIFEST_PATH.read_bytes()).hexdigest(),
                        "component_manifest_sha256": hashlib.sha256(COMPONENT_MANIFEST_PATH.read_bytes()).hexdigest(),
                        "recipe_manifest_sha256": hashlib.sha256(RECIPE_MANIFEST_PATH.read_bytes()).hexdigest()}, run_dir / "best.pt")
            save_validation_frame(name, "clean", clean, config["task"])
            save_validation_frame(name, "stress", stress, config["task"])
        else: stale += 1
        finished = stale >= config["patience"] or epoch == config["epochs"]
        atomic_torch_save({"signature": signature, "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(), "scaler_state": scaler.state_dict(),
                    "config": config, "epoch": epoch, "best_score": best_score,
                    "stale": stale, "history": history, "finished": finished}, last_path)
        if stale >= config["patience"]: break
    del model, optimizer, scheduler, scaler; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(history)


if RUN_TRAINING:
    summaries = []
    for branch_name in BRANCHES_TO_TRAIN:
        print("\n=== recommended branch:", branch_name, "===")
        history = fit_branch(branch_name)
        summaries.append({"branch": branch_name, **history.loc[history.robust_score.idxmax()].to_dict()})
    branch_training_summary = pd.DataFrame(summaries).sort_values("robust_score", ascending=False)
    atomic_csv(branch_training_summary, RUN_ROOT / "branch_training_best_metrics.csv")
    display(branch_training_summary)

else:
    print("RUN_TRAINING=False: 학습 생략. 준비와 self-test 성공 후 이 플래그를 True로 실행하세요.")



=== recommended branch: presence ===
presence precision: AMP FP16

=== recommended branch: aasist_aux3 ===
aasist_aux3 precision: AMP FP16

=== recommended branch: voice_wavlm ===
voice_wavlm precision: AMP FP16

=== recommended branch: music_long ===
music_long precision: FP32


,branch,epoch,train_loss,train_samples,train_skipped,train_optimizer_updates,robust_score,clean_voice_presence_auc,clean_music_presence_auc,clean_cps,...,stress_music_eer,stress_ads,clean_voice_auc,clean_voice_quality,stress_voice_auc,stress_voice_quality,clean_music_auc,clean_music_quality,stress_music_auc,stress_music_quality
0,presence,9.0,0.100123,22496.0,0.0,702.0,0.982025,0.983693,0.999463,0.991578,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,music_long,12.0,0.090915,22500.0,0.0,1318.0,0.952508,NaN,NaN,NaN,...,0.085379,NaN,NaN,NaN,NaN,NaN,0.999673,0.990395,0.976172,0.914621
1,aasist_aux3,13.0,0.333352,22500.0,0.0,1406.0,0.911752,NaN,NaN,NaN,...,0.155816,0.871393,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,voice_wavlm,8.0,0.321657,22500.0,0.0,1274.0,0.789992,NaN,NaN,NaN,...,NaN,NaN,0.904606,0.816241,0.858679,0.763743,NaN,NaN,NaN,NaN


In [23]:
# 학습 없이 최종 Validation과 fusion 생성
RUN_TRAINING = False
RUN_MODEL_SELF_TEST = False
RUN_FINAL_VALIDATION = True
RUN_OOD = False
BUILD_SUBMIT = False

NEED_TRAINED_MODELS = True

print("학습 재실행:", RUN_TRAINING)
print("최종 Validation:", RUN_FINAL_VALIDATION)

학습 재실행: False
최종 Validation: True


## 17. 검증·OOD·제출이 공유하는 파일 단위 추론
원본의 Top-K/logit fusion 수식과 voice 상대 presence gate를 유지합니다. **music에는 원본처럼 Top-K만 적용**하며 '양쪽 모두 gated'라고 부르지 않습니다.
최대 segment 제한 때문에 아주 짧은 생성 구간을 놓칠 수 있습니다. 이 정책 변경은 validation으로 별도 실험해야 합니다.
빈 오디오, 중복 ID, NaN, 파일 미존재는 중립 확률로 숨기지 않고 중단합니다.

In [24]:
INFERENCE_CORE_SOURCE = "from pathlib import Path\nimport math\nimport shutil\nimport subprocess\nimport numpy as np\nimport soundfile as sf\nimport torch\nimport torch.nn.functional as F\nimport torchaudio\n\nCOLUMNS = ['FILE_FAKE_PROB','VOICE_FAKE_PROB','MUSIC_FAKE_PROB','VOICE_PRESENT_PROB','MUSIC_PRESENT_PROB']\nSR, SHORT, LONG = 16000, 64600, 256000\n\n\ndef read_audio(path, ffmpeg=None):\n    path = Path(path)\n    if not path.is_file(): raise FileNotFoundError(path)\n    try:\n        data,sr = sf.read(path,dtype='float32',always_2d=True)\n        wave = torch.from_numpy(data.mean(1).copy())\n    except (RuntimeError,OSError,ValueError):\n        exe=ffmpeg or shutil.which('ffmpeg')\n        if not exe: raise RuntimeError(f'SoundFile cannot decode; FFmpeg is required: {path}')\n        result=subprocess.run([str(exe),'-hide_banner','-loglevel','error','-i',str(path),\n                               '-ac','1','-ar',str(SR),'-f','f32le','pipe:1'],\n                              stdout=subprocess.PIPE,stderr=subprocess.PIPE,check=True,timeout=120)\n        wave=torch.from_numpy(np.frombuffer(result.stdout,dtype='<f4').copy());sr=SR\n    if wave.numel()==0 or not torch.isfinite(wave).all(): raise ValueError(f'empty/nonfinite audio: {path}')\n    if sr!=SR: wave=torchaudio.functional.resample(wave,int(sr),SR)\n    return wave.float().clamp(-1,1)\n\n\ndef evaluation_segments(audio, clip_samples, maximum):\n    clip_samples=int(clip_samples);maximum=int(maximum)\n    if clip_samples<1 or maximum<1: raise ValueError('clip/max must be positive')\n    audio=audio.detach().cpu().float().reshape(-1)\n    if not audio.numel() or not torch.isfinite(audio).all(): raise ValueError('invalid waveform')\n    if audio.numel()<clip_samples:\n        audio=audio.repeat(math.ceil(clip_samples/audio.numel()))\n    starts=list(range(0,audio.numel()-clip_samples+1,clip_samples))\n    last=audio.numel()-clip_samples\n    if starts[-1]!=last: starts.append(last)\n    if len(starts)>maximum:\n        indices=np.linspace(0,len(starts)-1,maximum).round().astype(int)\n        starts=[starts[i] for i in sorted(set(indices.tolist()))]\n    result=[]\n    for start in starts:\n        segment=audio[start:start+clip_samples];segment=segment-segment.mean()\n        result.append((segment/segment.abs().max().clamp_min(1e-8)*.82).clamp(-1,1))\n    return torch.stack(result)\n\n\ndef topk_mean(values,fraction=.30):\n    values=np.asarray(values,float).reshape(-1)\n    if not len(values) or not np.isfinite(values).all() or not 0<fraction<=1:\n        raise ValueError('invalid top-k input')\n    k=max(1,math.ceil(len(values)*fraction))\n    return float(np.sort(values)[-k:].mean())\n\n\ndef presence_gated_fake(fake_scores,presence_scores,fraction=.30):\n    fake=np.asarray(fake_scores,float).reshape(-1);presence=np.asarray(presence_scores,float).reshape(-1)\n    if fake.shape!=presence.shape or not len(fake) or not np.isfinite(presence).all():\n        raise ValueError('fake/presence shape mismatch')\n    # 원본의 상대 순위 gate 유지. 낮은 presence에서도 최소 1개를 선택하므로 절대 threshold gate는 아님.\n    k=max(1,math.ceil(len(fake)*max(.5,fraction)))\n    return topk_mean(fake[np.argsort(presence,kind='stable')[-k:]],fraction)\n\n\ndef blend_probabilities(first,second,weight_first):\n    if not 0<=weight_first<=1: raise ValueError('fusion weight outside [0,1]')\n    a=np.asarray(first,float);b=np.asarray(second,float)\n    if a.shape!=b.shape or not np.isfinite(a).all() or not np.isfinite(b).all():\n        raise ValueError('invalid fusion scores')\n    a=np.clip(a,1e-5,1-1e-5);b=np.clip(b,1e-5,1-1e-5)\n    z=weight_first*(np.log(a)-np.log1p(-a))+(1-weight_first)*(np.log(b)-np.log1p(-b))\n    return 1/(1+np.exp(-z))\n\n\ndef predict_segments(model, segments, batch_size, device, use_amp=True, offload=True):\n    device=torch.device(device)\n    if int(batch_size)<1: raise ValueError('batch must be positive')\n    out=[];model.to(device).eval()\n    try:\n        with torch.inference_mode():\n            for start in range(0,len(segments),int(batch_size)):\n                x=segments[start:start+int(batch_size)].to(device)\n                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=use_amp and device.type=='cuda'):\n                    logits=model(x)\n                p=torch.sigmoid(logits.float()).cpu().numpy()\n                if not np.isfinite(p).all(): raise FloatingPointError('nonfinite inference logits')\n                out.append(p)\n    finally:\n        if offload: model.to('cpu')\n    if not out: raise ValueError('empty segments')\n    return np.concatenate(out)\n\n\ndef collect_file_evidence(audio,models,fusion,device='cpu',offload=True):\n    agg=fusion['aggregation']\n    short=evaluation_segments(audio,SHORT,agg['short_maximum_segments'])\n    long=evaluation_segments(audio,LONG,agg['music_maximum_segments'])\n    batches=fusion.get('inference_batches',{'presence':4,'aasist_aux3':2,'voice_wavlm':1,'music_long':1})\n    def predict(name,x):\n        return predict_segments(models[name],x,batches[name],device,use_amp=fusion.get('use_amp',True),offload=offload)\n    p=predict('presence',short)\n    a=predict('aasist_aux3',short)\n    v=predict('voice_wavlm',short)[:,0]\n    m=predict('music_long',long)[:,0]\n    if p.shape!=(len(short),2) or a.shape!=(len(short),3): raise ValueError('branch output shape')\n    return {'presence':p,'aux':a,'voice':v,'music':m}\n\n\ndef aggregate_evidence(evidence,fusion):\n    p,a,v,m=(evidence[k] for k in ('presence','aux','voice','music'))\n    f=float(fusion['aggregation']['top_fraction']);w=fusion['weights']\n    vs=blend_probabilities(v,a[:,1],float(w['voice_specialist']))\n    vp=topk_mean(p[:,0],f);mp=topk_mean(p[:,1],f)\n    vf=presence_gated_fake(vs,p[:,0],f)\n    # 원본 music aggregation을 유지: long/aux 각각 top-k. presence gate는 voice에만 적용.\n    mf=float(blend_probabilities(topk_mean(m,f),topk_mean(a[:,2],f),float(w['music_specialist'])))\n    coherent=1-(1-vp*vf)*(1-mp*mf)\n    ff=float(blend_probabilities(topk_mean(a[:,0],f),coherent,float(w['file_direct'])))\n    result=np.clip([ff,vf,mf,vp,mp],0,1)\n    if result.shape!=(5,) or not np.isfinite(result).all(): raise ValueError('invalid five outputs')\n    return result\n\n\ndef predict_file_ensemble(audio,models,fusion,device='cpu',offload=True):\n    return aggregate_evidence(collect_file_evidence(audio,models,fusion,device,offload),fusion)\n\n\ndef resolve_audio_ids(test_dir, identifiers, suffixes):\n    test_dir=Path(test_dir).resolve()\n    if not test_dir.is_dir(): raise FileNotFoundError(test_dir)\n    lookup={};duplicates=set()\n    for path in sorted(test_dir.rglob('*')):\n        if not path.is_file() or path.suffix.lower() not in suffixes: continue\n        for key in {path.stem,path.name,path.relative_to(test_dir).as_posix()}:\n            if key in lookup and lookup[key]!=path: duplicates.add(key)\n            lookup[key]=path\n    result=[]\n    for value in identifiers:\n        key=str(value)\n        if key in duplicates: raise ValueError(f'ambiguous audio ID: {key}')\n        if key not in lookup: raise FileNotFoundError(f'no audio for ID: {key}')\n        result.append(lookup[key])\n    if len(set(map(str,result)))!=len(result): raise ValueError('multiple IDs refer to the same audio')\n    return result\n"
exec(compile(INFERENCE_CORE_SOURCE,"runtime_inference.py","exec"), globals())


def load_best_branch(name):
    path=RUN_ROOT/name/'best.pt'
    if not path.is_file(): raise FileNotFoundError(f'학습한 branch checkpoint가 필요합니다: {path}')
    state=torch.load(path,map_location='cpu',weights_only=True)
    if state.get('signature')!=training_signature(MODEL_CONFIGS[name]):
        raise RuntimeError(f'{name}: 설정/manifest/model 버전이 checkpoint와 다릅니다.')
    model=build_branch(name,pretrained=False)
    model.load_state_dict(state['model_state'],strict=True)
    return model.cpu().eval()


CHECKPOINT_PATHS = {name: RUN_ROOT / name / "best.pt" for name in BRANCHES_TO_TRAIN}
CHECKPOINTS_READY = all(path.is_file() for path in CHECKPOINT_PATHS.values())
NEED_TRAINED_MODELS = RUN_TRAINING or RUN_FINAL_VALIDATION or RUN_OOD or BUILD_SUBMIT

if RUN_FINAL_VALIDATION and not CHECKPOINTS_READY:
    missing = [str(path) for path in CHECKPOINT_PATHS.values() if not path.is_file()]
    raise FileNotFoundError(
        "RUN_FINAL_VALIDATION=True이지만 네 branch checkpoint가 모두 준비되지 않았습니다:\n"
        + "\n".join(missing)
    )

if NEED_TRAINED_MODELS:
    trained_models={name:load_best_branch(name) for name in BRANCHES_TO_TRAIN}
    # 모델은 CPU에 보관. 추론할 때 한 모델씩 GPU로 올리고 되돌립니다.
    print('네 branch strict restore: OK')
else:
    print('모델 추론 단계 생략. 저장된 결과는 마지막 대시보드에서 별도로 읽습니다.')


네 branch strict restore: OK


## 18. 최종 validation — 같은 실제 파일을 제출 파이프라인으로 평가
원본은 branch별 4초/16초 예측 CSV를 바로 합쳐 제출의 segmentation/aggregation과 달랐습니다.
수정본은 validation source만으로 고정 파일을 생성하고, **제출에도 사용하는 동일한 함수**로 evidence를 수집해 fusion을 선택합니다.
Branch별 clean/stress 점수는 학습용 진단/체크포인트 선택으로 유지하고, 최종 validation 점수와 분리해 기록합니다.
이 검증 세트도 synthetic/weak-label 분포라는 한계가 있으며 실제 대회 점수를 보장하지 않습니다.

In [25]:
FUSION_PATH=RUN_ROOT/'fusion.json'
VALIDATION_FILE_ROOT=RUN_ROOT/'file_validation'

def prepare_file_validation():
    signature=hashlib.sha256((PIPELINE_VERSION+file_hash(SOURCE_MANIFEST_PATH)+file_hash(RECIPE_MANIFEST_PATH)).encode()).hexdigest()
    root=VALIDATION_FILE_ROOT/signature[:16];root.mkdir(parents=True,exist_ok=True)
    frame_path=root/'manifest.csv'
    if frame_path.is_file():
        frame=pd.read_csv(frame_path,dtype={'ID':str})
        if len(frame)==2*len(recipe_by_split['validation']) and all(Path(p).is_file() for p in frame.path):
            return frame
        raise RuntimeError(f'기존 validation 생성본 불완전: {root}. 자동으로 결과를 섞지 않습니다.')
    frames=[]
    lengths=[64000,96000,128000,192000,256000]
    recipes=recipe_by_split['validation'].reset_index(drop=True)
    for index in tqdm(range(len(recipes)),desc='고정 validation 파일 생성'):
        length=lengths[stable_int(recipes.iloc[index].recipe_id)%len(lengths)]
        base=DynamicMixDataset(recipes.iloc[[index]],source_by_split['validation'],training=False,target_samples=length)
        for view,enabled in [('clean',False),('stress',True)]:
            item=DomainStressView(base,enabled)[0]
            if not bool(item['valid']): raise RuntimeError(item['error'])
            path=root/view/(str(item['id'])+'.flac');path.parent.mkdir(parents=True,exist_ok=True)
            sf.write(path,item['audio'].numpy(),16000,subtype='PCM_16')
            frames.append(dict(ID=str(item['id']),view=view,path=str(path),
                               **dict(zip(DACON_TRUTH_COLUMNS,item['target'].int().tolist()))))
    frame=pd.DataFrame(frames);atomic_csv(frame,frame_path);return frame


def inference_signature():
    hashes={name:file_hash(RUN_ROOT/name/'best.pt') for name in BRANCHES_TO_TRAIN}
    hashes['core']=hashlib.sha256(INFERENCE_CORE_SOURCE.encode()).hexdigest()
    hashes['models']=hashlib.sha256(MODEL_SOURCE.encode()).hexdigest()
    return hashlib.sha256(json.dumps(hashes,sort_keys=True).encode()).hexdigest()


if NEED_TRAINED_MODELS:
    validation_files=prepare_file_validation()
    provisional_fusion={
        'weights':{'voice_specialist':.5,'music_specialist':.5,'file_direct':.5},
        'aggregation':{'top_fraction':.30,'short_maximum_segments':6,'music_maximum_segments':3},
        'inference_batches':INFERENCE_BATCHES,'use_amp':USE_AMP,
    }
    ev_key=hashlib.sha256((inference_signature()+json.dumps(provisional_fusion,sort_keys=True)
        +file_hash(SOURCE_MANIFEST_PATH)+file_hash(RECIPE_MANIFEST_PATH)).encode()).hexdigest()
    evidence_root=RUN_ROOT/'validation_evidence'/ev_key;evidence_root.mkdir(parents=True,exist_ok=True)
    evidence_views={};truth_views={}
    for view in ('clean','stress'):
        table=validation_files[validation_files.view.eq(view)].sort_values('ID').reset_index(drop=True)
        truth_views[view]=table
        evidence_views[view]=[]
        for row in tqdm(table.itertuples(index=False),total=len(table),desc=f'file-level validation {view}'):
            cache=evidence_root/f'{view}_{row.ID}.npz'
            if cache.is_file():
                with np.load(cache,allow_pickle=False) as f: evidence={name:f[name] for name in ('presence','aux','voice','music')}
            else:
                evidence=collect_file_evidence(read_audio(row.path,FFMPEG_EXE),trained_models,provisional_fusion,DEVICE,OFFLOAD_INFERENCE_MODELS)
                temp=cache.with_suffix('.writing')
                with temp.open('wb') as stream: np.savez_compressed(stream,**evidence)
                temp.replace(cache)
            evidence_views[view].append(evidence)
    fusion=copy.deepcopy(provisional_fusion)
    for weight_name,target_index in [('voice_specialist',1),('music_specialist',2),('file_direct',0)]:
        candidates=[]
        for weight in np.linspace(0,1,41):
            trial=copy.deepcopy(fusion);trial['weights'][weight_name]=float(weight)
            quality=[]
            for view in ('clean','stress'):
                truth=truth_views[view]
                prediction=np.array([aggregate_evidence(e,trial) for e in evidence_views[view]])
                valid=np.ones(len(truth),bool) if target_index==0 else truth[DACON_TRUTH_COLUMNS[target_index+2]].eq(1).to_numpy()
                quality.append(1-equal_error_rate(truth.loc[valid,DACON_TRUTH_COLUMNS[target_index]],prediction[valid,target_index]))
            candidates.append((float(np.mean(quality)),float(weight)))
        fusion['weights'][weight_name]=max(candidates)[1]
    fusion.update(version=PIPELINE_VERSION,inference_signature=inference_signature(),
                  selection_data='fixed validation files, identical inference core; no OOD/test labels')
    atomic_json(fusion,FUSION_PATH)
    reports=[]
    for view in ('clean','stress'):
        prediction=pd.DataFrame([aggregate_evidence(e,fusion) for e in evidence_views[view]],columns=DACON_PROBABILITY_COLUMNS)
        metrics = dacon_official_score(truth_views[view][DACON_TRUTH_COLUMNS], prediction)
        reports.append(dict(
            view=view,
            samples=len(truth_views[view]),
            skipped=0,
            invalid=0,
            **metrics,
        ))
        prediction.insert(0,'ID',truth_views[view].ID)
        atomic_csv(prediction,RUN_ROOT/f'validation_file_{view}_predictions.csv')
    validation_reports=pd.DataFrame(reports)
    atomic_csv(validation_reports,RUN_ROOT/'validation_file_metrics.csv')
    display(validation_reports)
    print('file-level validation fusion saved:',FUSION_PATH)
else:
    print('최종 validation 생략. checkpoint가 있으면 RUN_FINAL_VALIDATION=True로 실행할 수 있습니다.')


file-level validation stress: 100%|██████████| 2500/2500 [00:15<00:00, 165.60it/s]


,view,samples,skipped,invalid,file_eer,voice_eer,music_eer,voice_presence_auc,music_presence_auc,ads,cps,score
0,clean,2500,0,0,0.045968,0.058199,0.009605,0.987097,0.999682,0.962494,0.993389,0.965584
1,stress,2500,0,0,0.084364,0.104502,0.087513,0.957141,0.998279,0.910663,0.977710,0.917368


file-level validation fusion saved: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1\fusion.json


In [26]:
FUSION_PATH = RUN_ROOT / "fusion.json"
METRICS_PATH = RUN_ROOT / "validation_file_metrics.csv"

print("fusion.json:", FUSION_PATH.is_file(), FUSION_PATH)
print("validation metrics:", METRICS_PATH.is_file(), METRICS_PATH)

if FUSION_PATH.is_file():
    display(json.loads(FUSION_PATH.read_text(encoding="utf-8")))

if METRICS_PATH.is_file():
    display(pd.read_csv(METRICS_PATH))

fusion.json: True C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1\fusion.json
validation metrics: True C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1\validation_file_metrics.csv


{'weights': {'voice_specialist': 0.15000000000000002,
  'music_specialist': 0.4,
  'file_direct': 0.07500000000000001},
 'aggregation': {'top_fraction': 0.3,
  'short_maximum_segments': 6,
  'music_maximum_segments': 3},
 'inference_batches': {'presence': 4,
  'aasist_aux3': 2,
  'voice_wavlm': 1,
  'music_long': 1},
 'use_amp': False,
 'version': 'v4.1-local-audit-20260909',
 'inference_signature': 'a6e48b54171aaeac26afe23b0cef5207e5724c937ec41d86bf868ae40ca0ac79',
 'selection_data': 'fixed validation files, identical inference core; no OOD/test labels'}

,view,samples,skipped,invalid,file_eer,voice_eer,music_eer,voice_presence_auc,music_presence_auc,ads,cps,score
0,clean,2500,0,0,0.045968,0.058199,0.009605,0.987097,0.999682,0.962494,0.993389,0.965584
1,stress,2500,0,0,0.084364,0.104502,0.087513,0.957141,0.998279,0.910663,0.977710,0.917368


In [27]:
# OOD 평가만 실행
RUN_TRAINING = False
RUN_MODEL_SELF_TEST = False
RUN_FINAL_VALIDATION = False
RUN_OOD = True
BUILD_SUBMIT = False
SHOW_SAVED_RESULTS = True
ALLOW_DOWNLOADS = True

# 필수 결과 확인
missing_checkpoints = [
    str(RUN_ROOT / name / "best.pt")
    for name in BRANCHES_TO_TRAIN
    if not (RUN_ROOT / name / "best.pt").is_file()
]

if missing_checkpoints:
    raise FileNotFoundError(
        "다음 체크포인트가 없습니다:\n" + "\n".join(missing_checkpoints)
    )

FUSION_PATH = RUN_ROOT / "fusion.json"

if not FUSION_PATH.is_file():
    raise FileNotFoundError(
        f"{FUSION_PATH}\n"
        "fusion.json이 없습니다. 먼저 18번 최종 Clean/Stress Validation 셀을 실행하세요."
    )

# 커널에서 모델 또는 fusion 변수가 사라진 경우 복원
if "trained_models" not in globals():
    trained_models = {
        name: load_best_branch(name)
        for name in BRANCHES_TO_TRAIN
    }

if "fusion" not in globals():
    fusion = json.loads(
        FUSION_PATH.read_text(encoding="utf-8")
    )

print("OOD 실행 준비 완료")
print("checkpoint:", list(trained_models))
print("fusion:", FUSION_PATH)

OOD 실행 준비 완료
checkpoint: ['presence', 'aasist_aux3', 'voice_wavlm', 'music_long']
fusion: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1\fusion.json


## 19. 별도 OOD 다운로드/색인 (선택)
`RUN_OOD=True`일 때만 실행합니다. 원본 외부 데이터 URL을 유지했으나 서버 가용성/사용권한을 보증하지 않습니다.
OOD 음악 내부 음성 정답은 caption keyword 기반 약한 라벨이므로 완전한 ground truth나 대회점수의 대체값이 아닙니다.
외부 데이터 추가 다운로드가 필요하며, 결과를 fusion 선택에 사용하지 않습니다.

In [30]:
from pathlib import Path
import re
import shutil
import pandas as pd


if RUN_OOD:
    OOD_PROJECT = RUN_ROOT / "ood2500"
    OOD_ARCHIVES = OOD_PROJECT / "archives"
    OOD_RAW = PROJECT_ROOT / "ood_raw"
    OOD_DATA = RUN_ROOT / "ood_dataset_2500"

    OOD_PROJECT.mkdir(parents=True, exist_ok=True)
    OOD_ARCHIVES.mkdir(parents=True, exist_ok=True)
    OOD_RAW.mkdir(parents=True, exist_ok=True)

    OOD_POOL_SIZE = 625
    OOD_SIZE = 2_500

    # 자동 다운로드를 사용할 경우
    VOICE_OOD_MODE = "in_the_wild_auto"

    # 수동 SpeechFake를 사용할 경우의 경로
    SPEECHFAKE_REAL_DIR = (
        DRIVE_ROOT / "ood_sources" / "speechfake" / "real_test"
    )
    SPEECHFAKE_FAKE_DIR = (
        DRIVE_ROOT / "ood_sources" / "speechfake" / "fake_test"
    )

    OOD_URLS = {
        "in_the_wild":
            "https://huggingface.co/datasets/"
            "mueller91/In-The-Wild/resolve/main/"
            "release_in_the_wild.zip",

        "song_audio":
            "https://zenodo.org/api/records/"
            "10072001/files/audio.zip/content",

        "song_csv":
            "https://zenodo.org/api/records/"
            "10072001/files/song_describer.csv/content",

        "fake_music":
            "https://zenodo.org/api/records/"
            "15063698/files/FakeMusicCaps.zip/content",

        "musiccaps_csv":
            "https://huggingface.co/datasets/"
            "google/MusicCaps/resolve/main/"
            "musiccaps-public.csv",
    }

    OOD_ARCHIVE_PATHS = {
        "in_the_wild":
            OOD_ARCHIVES / "release_in_the_wild.zip",

        "song_audio":
            OOD_ARCHIVES / "song_audio.zip",

        "song_csv":
            OOD_ARCHIVES / "song_describer.csv",

        "fake_music":
            OOD_ARCHIVES / "FakeMusicCaps.zip",

        "musiccaps_csv":
            OOD_ARCHIVES / "musiccaps-public.csv",
    }

    def download_resumable(url, destination):
        return download_http(url, destination)

    def safe_extract(archive_path, destination):
        return safe_extract_zip(archive_path, destination)

    # ---------------------------------------------------------
    # 1. 필요한 파일 다운로드
    # ---------------------------------------------------------

    required = [
        "song_audio",
        "song_csv",
        "fake_music",
        "musiccaps_csv",
    ]

    if VOICE_OOD_MODE == "in_the_wild_auto":
        required.insert(0, "in_the_wild")

    missing_archives = [
        key
        for key in required
        if not OOD_ARCHIVE_PATHS[key].is_file()
    ]

    if missing_archives and not ALLOW_DOWNLOADS:
        raise FileNotFoundError(
            "OOD 다운로드가 비활성화되어 있고 다음 파일이 없습니다:\n"
            + "\n".join(
                str(OOD_ARCHIVE_PATHS[key])
                for key in missing_archives
            )
        )

    # 기존 파일이 모두 있으면 재실행할 때 40GiB 검사를 반복하지 않음
    if missing_archives:
        free_local_gib = (
            shutil.disk_usage(PROJECT_ROOT).free / 2**30
        )

        if free_local_gib < 40:
            raise RuntimeError(
                "아직 다운로드할 OOD 파일이 있으며 "
                "최소 40GiB 여유 공간을 권장합니다.\n"
                f"현재 여유 공간: {free_local_gib:.1f}GiB\n"
                f"누락 파일: {missing_archives}"
            )

        for key in missing_archives:
            download_resumable(
                OOD_URLS[key],
                OOD_ARCHIVE_PATHS[key],
            )
    else:
        print(
            "OOD 압축파일과 메타데이터가 모두 있어 "
            "다운로드를 생략합니다."
        )

    # ---------------------------------------------------------
    # 2. 압축 해제
    # ---------------------------------------------------------

    extraction_targets = {
        "in_the_wild":
            OOD_RAW / "in_the_wild",

        "song_audio":
            OOD_RAW / "song_describer",

        "fake_music":
            OOD_RAW / "fake_music_caps",
    }

    for key, folder in extraction_targets.items():
        if (
            key == "in_the_wild"
            and VOICE_OOD_MODE != "in_the_wild_auto"
        ):
            continue

        marker = folder / ".complete"

        if not marker.is_file():
            safe_extract(
                OOD_ARCHIVE_PATHS[key],
                folder,
            )
            marker.touch()

        audio_count = len(audio_files(folder))

        if audio_count == 0:
            raise RuntimeError(
                f"{key}: 압축 해제 폴더에서 "
                f"오디오 파일을 찾지 못했습니다:\n{folder}"
            )

        print(
            f"{key}:",
            f"오디오 {audio_count:,}개",
            f"경로={folder}",
        )

    # ---------------------------------------------------------
    # 3. Voice 데이터 인덱싱
    # ---------------------------------------------------------

    def normalize_label(value):
        value = (
            str(value)
            .strip()
            .lower()
            .replace("_", "-")
        )

        if value in {
            "real",
            "bonafide",
            "bona-fide",
            "genuine",
            "0",
        }:
            return "real"

        if value in {
            "fake",
            "spoof",
            "deepfake",
            "1",
        }:
            return "fake"

        return None

    def index_voice(root):
        root = Path(root)
        files = audio_files(root)

        by_name = {
            path.name.lower(): path
            for path in files
        }

        by_stem = {
            path.stem.lower(): path
            for path in files
        }

        rows = []

        table_paths = (
            list(root.rglob("*.csv"))
            + list(root.rglob("*.tsv"))
        )

        for table_path in table_paths:
            try:
                separator = (
                    "\t"
                    if table_path.suffix.lower() == ".tsv"
                    else ","
                )

                table = pd.read_csv(
                    table_path,
                    sep=separator,
                )
            except Exception:
                continue

            lower = {
                str(column).lower(): column
                for column in table.columns
            }

            label_col = next(
                (
                    lower[key]
                    for key in (
                        "label",
                        "class",
                        "target",
                    )
                    if key in lower
                ),
                None,
            )

            file_col = next(
                (
                    lower[key]
                    for key in (
                        "file",
                        "path",
                        "filename",
                        "audio",
                    )
                    if key in lower
                ),
                None,
            )

            speaker_col = next(
                (
                    lower[key]
                    for key in (
                        "speaker",
                        "speaker_id",
                        "person",
                    )
                    if key in lower
                ),
                None,
            )

            if label_col is None or file_col is None:
                continue

            for _, row in table.iterrows():
                label = normalize_label(row[label_col])
                raw = Path(str(row[file_col]))

                path = (
                    by_name.get(raw.name.lower())
                    or by_stem.get(raw.stem.lower())
                )

                if label is not None and path is not None:
                    group = (
                        str(row[speaker_col])
                        if speaker_col is not None
                        else path.parent.name
                    )

                    rows.append({
                        "path": str(path),
                        "label": label,
                        "group": group,
                    })

            if rows:
                break

        return (
            pd.DataFrame(
                rows,
                columns=[
                    "path",
                    "label",
                    "group",
                ],
            )
            .drop_duplicates("path")
        )

    # ---------------------------------------------------------
    # 4. 그룹 균형 샘플링 함수
    # ---------------------------------------------------------

    def balanced_sample(
        frame,
        count,
        namespace,
    ):
        frame = frame.copy()

        required_columns = {
            "path",
            "group",
        }

        missing_columns = (
            required_columns
            - set(frame.columns)
        )

        if missing_columns:
            raise ValueError(
                f"{namespace}: 필수 열 누락 "
                f"{sorted(missing_columns)}"
            )

        if frame.empty:
            raise ValueError(
                f"{namespace}: 선택할 파일이 없습니다."
            )

        frame["key"] = frame["path"].map(
            lambda path: stable_int(
                f"ood|{CFG.seed}|{namespace}|{path}"
            )
        )

        groups = [
            group.sort_values("key").to_dict("records")
            for _, group in frame.groupby(
                "group",
                dropna=False,
            )
        ]

        chosen = []
        cursor = 0

        while (
            len(chosen) < count
            and any(
                cursor < len(group)
                for group in groups
            )
        ):
            for group in groups:
                if (
                    cursor < len(group)
                    and len(chosen) < count
                ):
                    chosen.append(group[cursor])

            cursor += 1

        if len(chosen) != count:
            raise ValueError(
                f"{namespace}: "
                f"{len(chosen)}/{count}"
            )

        return (
            pd.DataFrame(chosen)
            .drop(columns="key")
        )

    # ---------------------------------------------------------
    # 5. Real/Fake Voice 각각 625개 선택
    # ---------------------------------------------------------

    if VOICE_OOD_MODE == "speechfake_manual":
        if (
            not SPEECHFAKE_REAL_DIR.is_dir()
            or not SPEECHFAKE_FAKE_DIR.is_dir()
        ):
            raise FileNotFoundError(
                "SpeechFake test 파일을 다음 경로에 준비하세요.\n"
                f"REAL: {SPEECHFAKE_REAL_DIR}\n"
                f"FAKE: {SPEECHFAKE_FAKE_DIR}\n"
                "또는 VOICE_OOD_MODE를 "
                "'in_the_wild_auto'로 변경하세요."
            )

        voice_index = pd.DataFrame(
            [
                {
                    "path": str(path),
                    "label": "real",
                    "group": path.parent.name,
                }
                for path in audio_files(
                    SPEECHFAKE_REAL_DIR
                )
            ]
            + [
                {
                    "path": str(path),
                    "label": "fake",
                    "group": path.parent.name,
                }
                for path in audio_files(
                    SPEECHFAKE_FAKE_DIR
                )
            ]
        )
    else:
        voice_index = index_voice(
            OOD_RAW / "in_the_wild"
        )

    if (
        voice_index.empty
        or "label" not in voice_index.columns
    ):
        raise RuntimeError(
            "OOD voice metadata에서 "
            "real/fake 파일을 찾지 못했습니다."
        )

    voice_counts = (
        voice_index["label"]
        .value_counts(dropna=False)
    )

    print("OOD Voice label counts:")
    display(voice_counts.rename("count"))

    for required_label in ("real", "fake"):
        available = int(
            voice_counts.get(required_label, 0)
        )

        if available < OOD_POOL_SIZE:
            raise RuntimeError(
                f"OOD {required_label} voice 부족: "
                f"{available}/{OOD_POOL_SIZE}"
            )

    ood_real_voice = balanced_sample(
        voice_index[
            voice_index["label"].eq("real")
        ],
        OOD_POOL_SIZE,
        "real_voice",
    )

    ood_fake_voice = balanced_sample(
        voice_index[
            voice_index["label"].eq("fake")
        ],
        OOD_POOL_SIZE,
        "fake_voice",
    )

    # ---------------------------------------------------------
    # 6. Song Describer Real Music 625개 선택
    # ---------------------------------------------------------

    song_raw = pd.read_csv(
        OOD_ARCHIVE_PATHS["song_csv"]
    )

    song_raw["track_id"] = pd.to_numeric(
        song_raw["track_id"],
        errors="coerce",
    )

    song_raw = (
        song_raw
        .dropna(subset=["track_id"])
        .copy()
    )

    song_raw["track_id"] = (
        song_raw["track_id"]
        .astype("int64")
    )

    song = (
        song_raw
        .sort_values("caption_id")
        .groupby(
            "track_id",
            as_index=False,
        )
        .agg({
            "path": "first",
            "artist_id": "first",
            "caption": lambda values: " ".join(
                values.dropna().astype(str)
            ),
        })
    )

    song_files = audio_files(
        OOD_RAW / "song_describer"
    )

    # CSV: 1004034.mp3
    # ZIP: 1004034.2min.mp3
    # 숫자 track_id를 추출해 연결
    def song_track_id_from_audio(path):
        match = re.fullmatch(
            r"(\d+)(?:\.2min)?",
            path.stem,
            flags=re.IGNORECASE,
        )

        return (
            int(match.group(1))
            if match is not None
            else None
        )

    song_lookup = {}
    unrecognized_song_files = []
    duplicate_song_ids = []

    for path in song_files:
        track_id = song_track_id_from_audio(path)

        if track_id is None:
            unrecognized_song_files.append(
                str(path)
            )
        elif track_id in song_lookup:
            duplicate_song_ids.append(
                track_id
            )
        else:
            song_lookup[track_id] = path

    metadata_track_ids = set(
        song["track_id"].astype(int)
    )

    matched_track_ids = (
        metadata_track_ids
        & set(song_lookup)
    )

    missing_track_ids = sorted(
        metadata_track_ids
        - set(song_lookup)
    )

    print(
        "Song Describer audit:",
        f"audio={len(song_files)},",
        f"indexed={len(song_lookup)},",
        f"metadata={len(metadata_track_ids)},",
        f"matched={len(matched_track_ids)}",
    )

    if duplicate_song_ids:
        raise RuntimeError(
            "Song Describer 중복 track_id 예시: "
            f"{duplicate_song_ids[:10]}"
        )

    if len(matched_track_ids) < OOD_POOL_SIZE:
        raise RuntimeError(
            "Song Describer 매칭 수가 부족합니다.\n"
            f"matched={len(matched_track_ids)}/"
            f"{OOD_POOL_SIZE}\n"
            f"missing 예시={missing_track_ids[:10]}\n"
            "인식 불가 파일 예시="
            f"{unrecognized_song_files[:3]}"
        )

    song_rows = []

    for _, row in song.iterrows():
        track_id = int(row["track_id"])
        path = song_lookup.get(track_id)

        if path is not None:
            song_rows.append({
                "path": str(path),
                "group": str(
                    row["artist_id"]
                ),
                "caption": str(
                    row["caption"]
                ),
            })

    real_music_index = (
        pd.DataFrame(
            song_rows,
            columns=[
                "path",
                "group",
                "caption",
            ],
        )
        .drop_duplicates("path")
    )

    ood_real_music = balanced_sample(
        real_music_index,
        OOD_POOL_SIZE,
        "real_music",
    )

    # ---------------------------------------------------------
    # 7. FakeMusicCaps Fake Music 625개 선택
    # ---------------------------------------------------------

    musiccaps = pd.read_csv(
        OOD_ARCHIVE_PATHS["musiccaps_csv"]
    )

    caption_lookup = dict(
        zip(
            musiccaps["ytid"].astype(str),
            musiccaps["caption"]
            .fillna("")
            .astype(str),
        )
    )

    aliases = {
        "musicgen":
            "MusicGen",

        "musicldm":
            "MusicLDM",

        "audioldm2":
            "AudioLDM2",

        "stableaudioopen":
            "StableAudioOpen",

        "mustango":
            "Mustango",
    }

    fake_rows = []

    for path in audio_files(
        OOD_RAW / "fake_music_caps"
    ):
        compact = re.sub(
            r"[^a-z0-9]",
            "",
            path.as_posix().lower(),
        )

        generator = next(
            (
                name
                for token, name in aliases.items()
                if token in compact
            ),
            None,
        )

        if generator is not None:
            fake_rows.append({
                "path": str(path),
                "group": generator,
                "generator": generator,
                "caption": caption_lookup.get(
                    path.stem,
                    "",
                ),
            })

    fake_index = (
        pd.DataFrame(
            fake_rows,
            columns=[
                "path",
                "group",
                "generator",
                "caption",
            ],
        )
        .drop_duplicates("path")
    )

    generator_counts = (
        fake_index["generator"]
        .value_counts()
        .sort_index()
    )

    print("FakeMusicCaps generator counts:")
    display(
        generator_counts.rename("count")
    )

    for generator in sorted(
        set(aliases.values())
    ):
        available = int(
            generator_counts.get(
                generator,
                0,
            )
        )

        if available < 125:
            raise RuntimeError(
                f"FakeMusicCaps {generator} 부족: "
                f"{available}/125"
            )

    fake_music_parts = []

    for generator in sorted(
        set(aliases.values())
    ):
        fake_music_parts.append(
            balanced_sample(
                fake_index[
                    fake_index["generator"]
                    .eq(generator)
                ],
                125,
                generator,
            )
        )

    ood_fake_music = pd.concat(
        fake_music_parts,
        ignore_index=True,
    )

    # ---------------------------------------------------------
    # 8. 네 풀을 하나의 manifest로 저장
    # ---------------------------------------------------------

    source_frames = {
        "real_voice":
            ood_real_voice,

        "fake_voice":
            ood_fake_voice,

        "real_music":
            ood_real_music,

        "fake_music":
            ood_fake_music,
    }

    ood_pools = {}

    for name, frame in source_frames.items():
        frame = frame.copy()
        frame["pool"] = name

        frame["source_id"] = [
            f"ood_{name}_{index:04d}"
            for index in range(len(frame))
        ]

        if len(frame) != OOD_POOL_SIZE:
            raise RuntimeError(
                f"{name}: "
                f"{len(frame)}/{OOD_POOL_SIZE}"
            )

        ood_pools[name] = frame

    ood_source_manifest = pd.concat(
        ood_pools.values(),
        ignore_index=True,
    )

    if len(ood_source_manifest) != OOD_SIZE:
        raise RuntimeError(
            "OOD source manifest 크기 오류: "
            f"{len(ood_source_manifest)}/"
            f"{OOD_SIZE}"
        )

    training_paths = set(
        source_manifest["path"].astype(str)
    )

    ood_paths = set(
        ood_source_manifest["path"]
        .astype(str)
    )

    if training_paths & ood_paths:
        raise RuntimeError(
            "training/OOD source overlap"
        )

    OOD_SOURCE_MANIFEST_PATH = (
        OOD_PROJECT
        / "ood_source_manifest.csv"
    )

    ood_source_manifest.to_csv(
        OOD_SOURCE_MANIFEST_PATH,
        index=False,
        encoding="utf-8",
    )

    display(
        ood_source_manifest
        .groupby(["pool", "group"])
        .size()
        .groupby(level=0)
        .agg([
            "count",
            "min",
            "max",
            "sum",
        ])
    )

    print(
        "OOD source manifest saved:",
        OOD_SOURCE_MANIFEST_PATH,
    )

else:
    print(
        "RUN_OOD=False: "
        "OOD 다운로드/색인을 생략합니다."
    )

OOD 압축파일과 메타데이터가 모두 있어 다운로드를 생략합니다.
in_the_wild: 오디오 31,779개 경로=C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\ood_raw\in_the_wild
song_audio: 오디오 706개 경로=C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\ood_raw\song_describer
fake_music: 오디오 27,605개 경로=C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\ood_raw\fake_music_caps
OOD Voice label counts:


label
real    19963
fake    11816
Name: count, dtype: int64

Song Describer audit: audio=706, indexed=706, metadata=706, matched=706
FakeMusicCaps generator counts:


generator
AudioLDM2          5521
MusicGen           5521
MusicLDM           5521
Mustango           5521
StableAudioOpen    5521
Name: count, dtype: int64

,count,min,max,sum
pool,,,,
fake_music,5,125,125,625
fake_voice,54,4,13,625
real_music,191,1,13,625
real_voice,54,3,13,625


OOD source manifest saved: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1\ood2500\ood_source_manifest.csv


## 20. OOD 고정 파일 생성 (원본 보존)
기존 OOD를 `shutil.rmtree`로 지우지 않습니다. 같은 설정으로 완성된 생성본만 재사용하며,
불완전/다른 설정의 폴더가 있으면 원인과 폴더 위치를 출력하고 중단합니다.

In [31]:
if RUN_OOD:
    OOD_RECIPE_COMPONENTS = {
        "rv": ["real_voice"], "fv": ["fake_voice"], "rm": ["real_music"], "fm": ["fake_music"],
        "rv_rm": ["real_voice", "real_music"], "fv_rm": ["fake_voice", "real_music"],
        "rv_fm": ["real_voice", "fake_music"], "fv_fm": ["fake_voice", "fake_music"],
    }
    VOICE_WORDS = re.compile(r"\b(vocal|voice|singer|singing|sung|lyrics|spoken|speech|whisper|rap|choir|chant)\b", re.I)


    def np_audio(path):
        return decode_audio(path).numpy()


    def crop_tile(audio, samples, rng):
        if len(audio)==0 or samples<=0: raise ValueError("empty OOD source")
        if len(audio) >= samples:
            start = int(rng.integers(0, len(audio) - samples + 1)); return audio[start:start + samples].copy()
        return np.tile(audio, math.ceil(samples / len(audio)))[:samples].copy()


    def np_rms(audio): return float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)) + 1e-9)
    def np_normalize(audio, db): return (audio * (10 ** (db / 20) / np_rms(audio))).astype(np.float32)


    def mix_ood_components(components, rng):
        if len(components) == 1:
            return components[0].copy(), "single"
        layout = rng.choice(["overlap", "partial", "sequential"], p=[.50, .25, .25])
        first, second = components
        if layout == "overlap":
            return (first + second).astype(np.float32), layout
        if layout == "partial":
            length = len(first)
            start = int(rng.integers(0, max(1, int(length * .45))))
            stop = int(rng.integers(max(start + 1, int(length * .60)), length + 1))
            gain = np.zeros(length, np.float32)
            fade = min(int(.05 * 16000), max(1, (stop - start) // 4))
            gain[start:stop] = 1
            gain[start:start + fade] = np.linspace(0, 1, fade, dtype=np.float32)
            gain[stop - fade:stop] = np.linspace(1, 0, fade, dtype=np.float32)
            return (first + second * gain).astype(np.float32), layout
        length = len(first)
        boundary = int(rng.integers(int(length * .35), max(int(length * .35) + 1, int(length * .65))))
        fade = min(int(.10 * 16000), boundary, length - boundary)
        first_gain = np.ones(length, np.float32); second_gain = np.zeros(length, np.float32)
        first_gain[boundary:] = 0; second_gain[boundary:] = 1
        if fade:
            first_gain[boundary - fade:boundary + fade] = np.linspace(1, 0, 2 * fade, dtype=np.float32)
            second_gain[boundary - fade:boundary + fade] = np.linspace(0, 1, 2 * fade, dtype=np.float32)
        return (first * first_gain + second * second_gain).astype(np.float32), layout


    def phone_filter(audio):
        down = librosa.resample(audio, orig_sr=16000, target_sr=8000, res_type="soxr_hq")
        tensor = torch.from_numpy(np.asarray(down, np.float32)).clamp(-1, 1)
        tensor = torchaudio.functional.highpass_biquad(tensor, 8000, 300)
        tensor = torchaudio.functional.lowpass_biquad(tensor, 8000, 3400)
        tensor = torchaudio.functional.mu_law_decoding(torchaudio.functional.mu_law_encoding(tensor, 256), 256)
        return librosa.resample(tensor.numpy(), orig_sr=8000, target_sr=16000, res_type="soxr_hq").astype(np.float32)


    CODECS = [("wav", None), ("flac", None), ("mp3", "64k"), ("mp3", "96k"), ("mp3", "128k"),
              ("aac", "64k"), ("aac", "96k"), ("opus", "48k"), ("opus", "64k")]


    def encode_ood(audio, destination, codec, bitrate):
        temporary = destination.with_suffix(".input.wav"); sf.write(temporary, audio, 16000, subtype="PCM_16")
        args = {"wav": ["-c:a", "pcm_s16le"], "flac": ["-c:a", "flac"],
                "mp3": ["-c:a", "libmp3lame", "-b:a", bitrate], "aac": ["-c:a", "aac", "-b:a", bitrate],
                "opus": ["-c:a", "libopus", "-b:a", bitrate]}[codec]
        subprocess.run([FFMPEG_EXE, "-hide_banner", "-loglevel", "error", "-y", "-i", str(temporary), "-ar", "16000", *args, str(destination)], check=True)
        temporary.unlink()


    OOD_BUILD_STAMP = hashlib.sha256((PIPELINE_VERSION + file_hash(OOD_PROJECT / "ood_source_manifest.csv")).encode()).hexdigest()
    OOD_COMPLETE = OOD_DATA / ".complete.json"
    BUILD_OOD = True
    if OOD_DATA.exists():
        if OOD_COMPLETE.is_file() and json.loads(OOD_COMPLETE.read_text(encoding="utf-8")).get("signature")==OOD_BUILD_STAMP:
            if len(audio_files(OOD_DATA/"data"/"test"))==OOD_SIZE:
                BUILD_OOD=False;print("기존 OOD 파일 재사용")
            else: raise RuntimeError("OOD 완료표시와 파일 개수 불일치")
        else:
            raise RuntimeError(f"기존 OOD 폴더가 불완전하거나 설정이 다릅니다. 별도 보관 후 다시 실행: {OOD_DATA}")
    if BUILD_OOD:

        test_dir = OOD_DATA / "data" / "test"; test_dir.mkdir(parents=True)
        rng = np.random.default_rng(CFG.seed + 9001)
        recipe_types = [name for index, name in enumerate(OOD_RECIPE_COMPONENTS) for _ in range(313 if index < 4 else 312)]
        random.Random(CFG.seed + 9001).shuffle(recipe_types)
        duration_values = rng.choice([4, 5, 6, 8, 10, 12, 15, 20, 30, 45, 60], OOD_SIZE,
                                     p=np.array([8, 8, 10, 12, 14, 12, 10, 8, 5, 2, 1]) / 90)
        records = {name: frame.sample(frac=1, random_state=CFG.seed).to_dict("records") for name, frame in ood_pools.items()}
        cursors = {name: 0 for name in records}; manifest_rows = []; truth_rows = []
        for index, (recipe, seconds) in enumerate(tqdm(zip(recipe_types, duration_values), total=OOD_SIZE)):
            identifier = f"OOD_{index:05d}"; samples = int(seconds * 16000); components = []; selected_rows = []
            for pool in OOD_RECIPE_COMPONENTS[recipe]:
                row = records[pool][cursors[pool] % len(records[pool])]; cursors[pool] += 1
                components.append(np_normalize(crop_tile(np_audio(row["path"]), samples, rng), float(rng.uniform(-25, -19))))
                selected_rows.append(row)
            mixed, layout = mix_ood_components(components, rng)
            telephone = bool(rng.random() < .12)
            if telephone: mixed = crop_tile(phone_filter(mixed), samples, rng)
            mixed = np_normalize(mixed - mixed.mean(), float(rng.uniform(-24, -18)))
            mixed = np.clip(mixed / max(np.max(np.abs(mixed)), 1e-8) * float(rng.uniform(.72, .96)), -1, 1)
            stereo = bool(rng.random() < .30)
            if stereo: mixed = np.stack([mixed, np.roll(mixed, int(rng.integers(2, 32))) * .94], axis=1)
            codec, bitrate = CODECS[index % len(CODECS)]; extension = "m4a" if codec == "aac" else codec
            encode_ood(mixed, test_dir / f"{identifier}.{extension}", codec, bitrate)

            def has_voice(row): return row["pool"].endswith("voice") or bool(VOICE_WORDS.search(str(row.get("caption", ""))))
            vp = int(any(has_voice(row) for row in selected_rows)); mp = int(any(row["pool"].endswith("music") for row in selected_rows))
            vf = int(any(row["pool"] == "fake_voice" or (row["pool"] == "fake_music" and has_voice(row)) for row in selected_rows))
            mf = int(any(row["pool"] == "fake_music" for row in selected_rows)); ff = max(vf, mf)
            manifest_rows.append({"ID": identifier, "recipe_type": recipe, "duration": seconds, "codec": codec,
                                  "bitrate": bitrate or "lossless", "channels": 2 if stereo else 1, "telephone": telephone,
                                  "layout": layout,
                                  "sources": "|".join(row["source_id"] for row in selected_rows)})
            truth_rows.append({"ID": identifier, "FILE_FAKE": ff, "VOICE_FAKE": vf, "MUSIC_FAKE": mf, "VOICE_PRESENT": vp, "MUSIC_PRESENT": mp})
        sample = pd.DataFrame({"ID": [row["ID"] for row in manifest_rows]})
        for column in DACON_PROBABILITY_COLUMNS: sample[column] = 0.0
        sample.to_csv(OOD_DATA / "data" / "sample_submission.csv", index=False)
        pd.DataFrame(manifest_rows).to_csv(OOD_DATA / "manifest.csv", index=False)
        pd.DataFrame(truth_rows).to_csv(OOD_DATA / "ground_truth.csv", index=False)
        print("OOD files:", len(audio_files(test_dir)))

    if BUILD_OOD: atomic_json({"signature":OOD_BUILD_STAMP}, OOD_COMPLETE)

else:
    print('RUN_OOD=False: OOD 합성을 생략합니다.')


100%|██████████| 2500/2500 [09:22<00:00,  4.44it/s]

OOD files: 2500


## 21. OOD 평가
예측 캐시 키에 checkpoint/추론 코드/fusion/정답 manifest hash를 포함합니다. 다른 모델의 기존 prediction을 섞지 않습니다.

In [32]:
if RUN_OOD:
    RUN_OOD_TEST = RUN_OOD
    OOD_SCORE_KEY = hashlib.sha256((inference_signature()+file_hash(FUSION_PATH)+file_hash(OOD_DATA/"ground_truth.csv")).encode()).hexdigest()[:16]
    OOD_PREDICTIONS_PATH = OOD_PROJECT / f"predictions_{OOD_SCORE_KEY}.csv"

    if RUN_OOD_TEST:
        sample = pd.read_csv(OOD_DATA / "data" / "sample_submission.csv")
        existing = pd.read_csv(OOD_PREDICTIONS_PATH) if OOD_PREDICTIONS_PATH.exists() else pd.DataFrame(columns=["ID", *DACON_PROBABILITY_COLUMNS])
        completed = set(existing.ID.astype(str)); rows = existing.to_dict("records")
        lookup = {p.stem: p for p in audio_files(OOD_DATA / "data" / "test")}
        for index, identifier in enumerate(tqdm(sample.ID.astype(str), desc="OOD ensemble"), 1):
            if identifier in completed: continue
            probabilities = predict_file_ensemble(read_audio(lookup[identifier],FFMPEG_EXE), trained_models, fusion, DEVICE, OFFLOAD_INFERENCE_MODELS)
            rows.append({"ID": identifier, **dict(zip(DACON_PROBABILITY_COLUMNS, map(float, probabilities)))})
            if len(rows) % 50 == 0: pd.DataFrame(rows).to_csv(OOD_PREDICTIONS_PATH, index=False)
        prediction = pd.DataFrame(rows).drop_duplicates("ID", keep="last").set_index("ID").loc[sample.ID.astype(str)].reset_index()
        prediction.to_csv(OOD_PREDICTIONS_PATH, index=False)
        truth = pd.read_csv(OOD_DATA / "ground_truth.csv",dtype={"ID":str}).set_index("ID").loc[sample.ID.astype(str)].reset_index()
        report = dacon_official_score(truth[DACON_TRUTH_COLUMNS], prediction[DACON_PROBABILITY_COLUMNS])
        report.update(samples=len(truth), skipped=0, invalid=0)
        (OOD_PROJECT / "official_metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
        display(pd.DataFrame([report]).T.rename(columns={0: "value"}))
        print("주의: music 내부 voice 여부는 caption keyword 기반 보조 label입니다.")

else:
    print('RUN_OOD=False: OOD 평가 생략.')


OOD ensemble: 100%|██████████| 2500/2500 [14:51<00:00,  2.81it/s]


,value
file_eer,0.275171
voice_eer,0.223145
music_eer,0.413020
voice_presence_auc,0.915983
music_presence_auc,0.993176
ads,0.693880
cps,0.954580
score,0.719950
samples,2500.000000
skipped,0.000000


주의: music 내부 voice 여부는 caption keyword 기반 보조 label입니다.


In [38]:
fusion_result = json.loads(
    (RUN_ROOT / "fusion.json").read_text(
        encoding="utf-8"
    )
)

print(fusion_result["weights"])

{'voice_specialist': 0.15000000000000002, 'music_specialist': 0.4, 'file_direct': 0.07500000000000001}


## 22. 추론시간 점검 (프로필을 몰래 변경하지 않음)
OOD 평가 뒤 자동으로 segment 수를 바꾸던 원본 동작을 제거했습니다. 현재 고정된 프로필에 60초 반복 입력을 사용해 비용을 측정합니다. 실제 콘텐츠 성능 검사는 아닙니다.
L4가 아닌 GPU의 시간은 참고용이며, 1,200파일 실제 서버 통과를 보증하지 않습니다.

In [33]:
TIME_GATE_PATH=RUN_ROOT/'inference_time_gate.json'
TIME_GATE_RESULT={'passed':False,'is_official_l4_comparable':False,'skipped':True}
if NEED_TRAINED_MODELS and DEVICE.type=='cuda':
    gpu_name=torch.cuda.get_device_name(0)
    sample_paths=validation_files[validation_files.view.eq('clean')].path.head(24).tolist()
    if sample_paths:
        torch.cuda.synchronize();started=time.perf_counter()
        for path in tqdm(sample_paths,desc='60초 반복 입력으로 현재 프로필 시간 점검'):
            wave = read_audio(path,FFMPEG_EXE)
            # 최대 길이 비용을 보기 위한 반복 입력. 실제 60초 콘텐츠 성능 검사가 아닙니다.
            wave = wave.repeat(math.ceil(60*CFG.sample_rate / wave.numel()))[:60*CFG.sample_rate]
            predict_file_ensemble(wave,trained_models,fusion,DEVICE,OFFLOAD_INFERENCE_MODELS)
        torch.cuda.synchronize();seconds=time.perf_counter()-started
        projected=1.5+seconds/len(sample_paths)*1200*1.15/60
        is_l4='L4' in gpu_name.upper()
        TIME_GATE_RESULT=dict(gpu=gpu_name,skipped=False,is_official_l4_comparable=is_l4,
            measured_files=len(sample_paths),synthetic_seconds=60,projected_minutes=projected,passed=is_l4 and projected<=48,
            inference_signature=inference_signature(),fusion_hash=file_hash(FUSION_PATH))
        print('반복 60초 입력 추정치(실제 파일/서버 변동 미보장):',TIME_GATE_RESULT)
atomic_json(TIME_GATE_RESULT,TIME_GATE_PATH)


60초 반복 입력으로 현재 프로필 시간 점검: 100%|██████████| 24/24 [00:12<00:00,  1.96it/s]


반복 60초 입력 추정치(실제 파일/서버 변동 미보장): {'gpu': 'NVIDIA GeForce RTX 5060 Laptop GPU', 'skipped': False, 'is_official_l4_comparable': False, 'measured_files': 24, 'synthetic_seconds': 60, 'projected_minutes': 13.242074041666152, 'passed': False, 'inference_signature': 'a6e48b54171aaeac26afe23b0cef5207e5724c937ec41d86bf868ae40ca0ac79', 'fusion_hash': '337807471131fdd2795334cf3722972e0e13a1fdd3c7ae845a34e7038ded2aff'}


## 23. 제출 wrapper 및 requirements
`model/runtime_models.py`와 `model/runtime_inference.py`는 노트북에서 사용한 코드와 같습니다.
WavLM config/state도 로컬에 포함합니다. 데이터 루트는 template가 있는 `data/` 또는 `open/`를 확인하고 ID를 정확히 매칭합니다.
대회 기본 설치 패키지는 requirements에서 재설치하지 않고 주석에 버전을 기록합니다. FFmpeg fallback용 `imageio-ffmpeg`만 추가합니다.
참고: https://www.dacon.io/competitions/official/236749/overview/evaluation

In [34]:
INFERENCE_SCRIPT = "from __future__ import annotations\nimport argparse\nimport json\nimport os\nimport shutil\nimport sys\nfrom pathlib import Path\n\n# 모델 추론에서 Hugging Face 네트워크 접근 금지. 모든 state/config를 ZIP에 포함.\nos.environ['HF_HUB_OFFLINE']='1'\nos.environ['TRANSFORMERS_OFFLINE']='1'\nimport numpy as np\nimport pandas as pd\nimport torch\n\nROOT=Path(__file__).resolve().parent\nMODEL_DIR=ROOT/'model'\nsys.path.insert(0,str(MODEL_DIR))\nfrom runtime_models import PresenceLogMel, build_aasist_aux3, VoiceWavLM, MusicSegmentTransformer\nfrom runtime_inference import COLUMNS, read_audio, predict_file_ensemble, resolve_audio_ids\n\nSUFFIXES={'.wav','.flac','.mp3','.m4a','.aac','.ogg','.opus','.wma','.amr'}\n\n\ndef load_models():\n    models={\n        'presence':PresenceLogMel(),\n        'aasist_aux3':build_aasist_aux3(MODEL_DIR/'aasist'),\n        'voice_wavlm':VoiceWavLM(config_path=MODEL_DIR/'wavlm_config',pretrained=False),\n        'music_long':MusicSegmentTransformer(),\n    }\n    for name,model in models.items():\n        state=torch.load(MODEL_DIR/name/'weights.pt',map_location='cpu',weights_only=True)\n        model.load_state_dict(state,strict=True);model.cpu().eval()\n    return models\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument('--data-dir',type=Path,default=None)\n    parser.add_argument('--output',type=Path,default=ROOT/'output'/'submission.csv')\n    parser.add_argument('--cpu',action='store_true')\n    args=parser.parse_args()\n    if args.data_dir is None:\n        roots=[ROOT/name for name in ('data','open') if (ROOT/name/'sample_submission.csv').is_file()]\n        if len(roots)!=1:\n            raise RuntimeError('data/ 또는 open/ 아래 sample_submission.csv 하나가 필요합니다. --data-dir로 명시 가능')\n        data_dir=roots[0]\n    else: data_dir=args.data_dir.resolve()\n    template=pd.read_csv(data_dir/'sample_submission.csv',dtype={'ID':str})\n    required=['ID',*COLUMNS]\n    if list(template.columns)!=required: raise ValueError(f'submission columns: {list(template.columns)}')\n    if template.ID.isna().any() or not template.ID.is_unique or template.empty: raise ValueError('invalid sample IDs')\n    paths=resolve_audio_ids(data_dir/'test',template.ID.tolist(),SUFFIXES)\n    device=torch.device('cuda' if torch.cuda.is_available() and not args.cpu else 'cpu')\n    ffmpeg=shutil.which('ffmpeg')\n    if not ffmpeg:\n        import imageio_ffmpeg\n        ffmpeg=imageio_ffmpeg.get_ffmpeg_exe()\n    models=load_models()\n    fusion=json.loads((MODEL_DIR/'fusion.json').read_text(encoding='utf-8'))\n    rows=[]\n    for i,(identifier,path) in enumerate(zip(template.ID,paths),1):\n        scores=predict_file_ensemble(read_audio(path,ffmpeg),models,fusion,device,offload=True)\n        rows.append([identifier,*map(float,scores)])\n        if i%50==0: print(f'{i}/{len(paths)}',flush=True)\n    frame=pd.DataFrame(rows,columns=required)\n    values=frame[COLUMNS].to_numpy(float)\n    if not np.isfinite(values).all() or not ((values>=0)&(values<=1)).all(): raise ValueError('nonfinite/out-of-range probabilities')\n    if frame.ID.tolist()!=template.ID.tolist(): raise ValueError('ID order mismatch')\n    args.output.parent.mkdir(parents=True,exist_ok=True)\n    temporary=args.output.with_suffix('.csv.writing')\n    frame.to_csv(temporary,index=False,encoding='utf-8');temporary.replace(args.output)\n    print('saved:',args.output,'rows:',len(frame))\n\nif __name__=='__main__': main()\n"
ast.parse(INFERENCE_SCRIPT)
print("제출 wrapper 문법 검사 OK (실제 실행은 다음 셀 smoke test)")


제출 wrapper 문법 검사 OK (실제 실행은 다음 셀 smoke test)


## 24. 학습된 모델 패키징 / 별도 프로세스 smoke test / ZIP
`.env`, Kaggle token, 원본 데이터는 ZIP에 넣지 않습니다. 이 노트북은 사이트에 자동 제출하지 않습니다.
CPU smoke test는 짧은 임의 입력 2개로 실행합니다. 컬럼/ID/출력 범위를 확인하지만 탐지 정확도를 검사하지 않습니다.
이 단계는 실제 학습 체크포인트가 있을 때만 가능합니다. CUDA/L4 1,200개 실행은 별도 확인해야 합니다.

In [35]:
SUBMIT_ZIP=PROJECT_ROOT/'submit_fixed_v4_1.zip'
if BUILD_SUBMIT:
    require_names('trained_models','fusion','FUSION_PATH')
    if fusion.get('inference_signature')!=inference_signature():
        raise RuntimeError('fusion이 현재 모델과 다릅니다. file-level validation/fusion 다시 실행')
    if REQUIRE_L4_TIME_GATE and not TIME_GATE_RESULT.get('passed',False):
        raise RuntimeError('L4 시간 확인 필요. 현재 GPU의 검사로 L4 검증을 대체하지 않습니다.')
    if not TIME_GATE_RESULT.get('passed',False):
        if not ALLOW_UNVERIFIED_SUBMIT: raise RuntimeError('미검증 submit ZIP 생성이 비활성화되어 있습니다.')
        warnings.warn('L4 시간 미검증. ZIP 생성 성공은 리더보드 실행 성공 보장이 아닙니다.')
    stage=BUILD_ROOT/f'submit_{int(time.time())}'
    model_dir=stage/'model';model_dir.mkdir(parents=True,exist_ok=False)
    for name in BRANCHES_TO_TRAIN:
        checkpoint=torch.load(RUN_ROOT/name/'best.pt',map_location='cpu',weights_only=True)
        if checkpoint.get('signature')!=training_signature(MODEL_CONFIGS[name]): raise RuntimeError(name+' signature mismatch')
        target=model_dir/name;target.mkdir()
        torch.save(checkpoint['model_state'],target/'weights.pt')
        del checkpoint
    shutil.copy2(FUSION_PATH,model_dir/'fusion.json')
    shutil.copytree(REPO_ROOT/'aasist'/'models',model_dir/'aasist'/'models',ignore=shutil.ignore_patterns('__pycache__','*.pyc'))
    shutil.copytree(REPO_ROOT/'aasist'/'config',model_dir/'aasist'/'config')
    for filename in ['LICENSE','LICENSE.md','LICENSE.txt']:
        if (REPO_ROOT/'aasist'/filename).is_file(): shutil.copy2(REPO_ROOT/'aasist'/filename,model_dir/'aasist'/filename)
    shutil.copytree(RUN_ROOT/'wavlm_config',model_dir/'wavlm_config')
    (model_dir/'runtime_models.py').write_text(MODEL_SOURCE,encoding='utf-8')
    (model_dir/'runtime_inference.py').write_text(INFERENCE_CORE_SOURCE,encoding='utf-8')
    (stage/'script.py').write_text(INFERENCE_SCRIPT,encoding='utf-8')
    requirements = '# DACON preinstalled (do not reinstall different CUDA binaries):\n# torch==2.7.1+cu128, torchaudio==2.7.1+cu128\n# numpy==1.26.4, pandas==2.0.3, soundfile==0.12.1\n# transformers==4.57.6, huggingface-hub==0.34.4, safetensors==0.6.2\n# transformers dependencies are already present per server documentation.\n# All other imports are Python standard library or bundled model code.\n# FFmpeg fallback for formats SoundFile cannot decode:\nimageio-ffmpeg==0.6.0\n'
    (stage/'requirements.txt').write_text(requirements,encoding='utf-8')
    for source in stage.rglob('*.py'): ast.parse(source.read_text(encoding='utf-8'))
    # 동일한 script를 별도 프로세스/CPU/offline로 실행해 ID 순서를 확인합니다.
    smoke=BUILD_ROOT/f'smoke_{int(time.time())}';test=smoke/'test';test.mkdir(parents=True,exist_ok=False)
    rng=np.random.default_rng(123)
    sf.write(test/'probe_b.wav',rng.normal(0,.01,(66000,2)).astype(np.float32),16000)
    sf.write(test/'probe_a.flac',rng.normal(0,.01,132000).astype(np.float32),32000)
    sample=pd.DataFrame({'ID':['probe_b','probe_a']})
    for name in DACON_PROBABILITY_COLUMNS: sample[name]=0.0
    sample.to_csv(smoke/'sample_submission.csv',index=False)
    smoke_output=smoke/'output'/'submission.csv'
    command=[sys.executable,str(stage/'script.py'),'--data-dir',str(smoke),'--output',str(smoke_output),'--cpu']
    run_to_log(command,smoke/'smoke.log')
    result=pd.read_csv(smoke_output,dtype={'ID':str})
    assert result.ID.tolist()==sample.ID.tolist()
    assert list(result.columns)==['ID',*DACON_PROBABILITY_COLUMNS]
    values=result[DACON_PROBABILITY_COLUMNS].to_numpy(float)
    assert values.shape==(2,5) and np.isfinite(values).all() and ((values>=0)&(values<=1)).all()
    print('실제 제출 script 별도 CPU 프로세스 smoke test: OK')
    temp_zip=SUBMIT_ZIP.with_suffix('.zip.writing')
    with zipfile.ZipFile(temp_zip,'w',zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as archive:
        for path in sorted(stage.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts:
                archive.write(path,path.relative_to(stage).as_posix())
    with zipfile.ZipFile(temp_zip) as archive:
        if archive.testzip() is not None: raise RuntimeError('ZIP CRC failure')
        top={name.split('/')[0] for name in archive.namelist()}
        if top!={'model','script.py','requirements.txt'}: raise RuntimeError(f'ZIP top-level mismatch: {top}')
        # GB/GiB 해석 차이를 피하기 위해 10^9 기준으로 보수적으로 확인
        if sum(x.file_size for x in archive.infolist())>32*10**9 or temp_zip.stat().st_size>10*10**9:
            raise RuntimeError('submission ZIP limit exceeded')
    temp_zip.replace(SUBMIT_ZIP)
    print('ZIP 생성:',SUBMIT_ZIP)
    print('원본 dataset/credential은 포함하지 않았습니다. L4/offline 전체 시간 별도 검증 필요.')
else:
    print('BUILD_SUBMIT=False. 학습 + file-level validation 완료 후 활성화하세요.')


BUILD_SUBMIT=False. 학습 + file-level validation 완료 후 활성화하세요.


## 변경·검증 요약
1. 원본에 기록된 `download_http` NameError와 누락된 `safe_extract_zip`을 복구했습니다.
2. 중복 복구 셀을 정리하고 경로/출처/다운로드/파일 목록의 정의 순서를 고정했습니다.
3. PANNs import 전 라벨 준비, 검증된 가중치 로딩, 실패 스크리닝 mask/CSV 감사 처리를 추가했습니다.
4. 이전 데이터가 남은 MFCC 특징 캐시를 현재 경로/유효한 96차원으로 제한했습니다.
5. 8GB 보수적 batch, 실제 장치 self-test, 동결 WavLM 모드/gradient 누적/FP32 score를 보완했습니다.
6. 모델/데이터 설정이 바뀐 상태에서 이전 checkpoint·prediction을 섞지 않도록 signature를 사용합니다.
7. 최종 validation·OOD·submit에서 같은 모델 정의/segment/fusion 코드를 사용합니다.
8. OOD 결과를 평가한 뒤 시간에 맞춰 추론 정책을 몰래 바꾸던 동작을 제거했습니다.
9. 제출은 plain state_dict + local WavLM config + 모델 코드로 구성하고 별도 프로세스 smoke test 후 ZIP을 만듭니다.

**그대로 남는 연구상의 한계**: FMA vs SONICS 출처 편향, MFCC 화자 추정의 부정확성, FMA prefix/SONICS 메타데이터/캡션의 약한 성분 라벨,
잘라낸 구간의 실제 성분 유무, 제한된 segment 수, 별도 외부 데이터 사용권한, 새로운 생성기 일반화.
이 수정본은 이 문제들을 자동으로 해결했다고 주장하지 않습니다.

### 참고한 1차 자료
- PANNs: https://github.com/qiuqiangkong/panns_inference (config.py / inference.py)
- AASIST: https://github.com/clovaai/aasist (원본 notebook의 commit 유지)
- Kaggle CLI: https://github.com/Kaggle/kaggle-cli/blob/main/skills/references/auth.md
- FMA: https://github.com/mdeff/fma
- torch.load: https://docs.pytorch.org/docs/stable/notes/serialization.html
- 대회: https://www.dacon.io/competitions/official/236749/overview/evaluation

작성 환경의 synthetic test 결과는 별도 `deepvoice_v4_local_audit.txt`에 기록됩니다.

## 25. 저장된 결과 종합 대시보드

이 셀은 모델을 다시 학습하거나 OOD 추론을 다시 실행하지 않습니다. `RUN_ROOT`에 저장된 결과만 읽어 다음을 표시합니다.

- Clean/Stress ensemble의 File·Voice·Music EER, ADS, CPS, Score
- OOD 2,500 ensemble의 동일 지표
- Presence, AASIST, Voice WavLM, Music Long의 best epoch 단독 성능
- 네 모델의 전체 `history.csv`
- 학습/검증/OOD의 skipped·invalid 수와 prediction 파일 무결성

결과가 없다고 표시되면 실제 단계가 실행되지 않은 것입니다. 실제 학습은 `RUN_TRAINING=True`, 학습 완료 후 Clean/Stress 재평가는 `RUN_FINAL_VALIDATION=True`, OOD 생성·평가는 `RUN_OOD=True`로 실행하세요.


In [36]:
RESULT_REPORT_ROOT = RUN_ROOT / "result_reports"
RESULT_REPORT_ROOT.mkdir(parents=True, exist_ok=True)

OFFICIAL_REPORT_COLUMNS = [
    "scope", "file_eer", "voice_eer", "music_eer", "ads",
    "voice_presence_auc", "music_presence_auc", "cps", "score",
    "samples", "skipped", "invalid", "status", "source",
]


def _atomic_report_csv(frame, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".writing")
    frame.to_csv(temporary, index=False, encoding="utf-8")
    temporary.replace(destination)


def _as_float(value):
    try:
        result = float(value)
    except (TypeError, ValueError):
        return np.nan
    return result if np.isfinite(result) else np.nan


def _as_int(value, default=0):
    numeric = _as_float(value)
    return int(default) if not np.isfinite(numeric) else int(numeric)


def _prediction_integrity(path, expected_rows, scope):
    path = Path(path)
    base = {
        "scope": scope,
        "prediction_path": str(path),
        "expected_rows": int(expected_rows),
        "rows": 0,
        "unique_ids": 0,
        "missing_ids": int(expected_rows),
        "duplicate_ids": 0,
        "invalid_probability_rows": 0,
        "status": "missing",
    }
    if not path.is_file():
        return base
    try:
        frame = pd.read_csv(path, dtype={"ID": str})
    except Exception as exc:
        base.update(status=f"read_error: {type(exc).__name__}")
        return base

    base["rows"] = len(frame)
    if "ID" not in frame.columns:
        base.update(status="ID_column_missing", invalid_probability_rows=len(frame))
        return base
    base["unique_ids"] = int(frame["ID"].nunique(dropna=True))
    base["missing_ids"] = max(0, int(expected_rows) - base["unique_ids"])
    base["duplicate_ids"] = int(frame["ID"].duplicated(keep=False).sum())
    missing_columns = [column for column in DACON_PROBABILITY_COLUMNS if column not in frame.columns]
    if missing_columns:
        base.update(status="probability_columns_missing", invalid_probability_rows=len(frame))
        return base
    values = frame[DACON_PROBABILITY_COLUMNS].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    invalid_rows = (~np.isfinite(values) | (values < 0) | (values > 1)).any(axis=1)
    base["invalid_probability_rows"] = int(invalid_rows.sum())
    base["status"] = "ok" if (
        base["rows"] == int(expected_rows)
        and base["unique_ids"] == int(expected_rows)
        and base["duplicate_ids"] == 0
        and base["invalid_probability_rows"] == 0
    ) else "check_required"
    return base


if SHOW_SAVED_RESULTS:
    print("=" * 80)
    print("DeepVoice 저장 결과 종합 보고")
    print("PROJECT_ROOT:", PROJECT_ROOT)
    print("RUN_ROOT:", RUN_ROOT)
    print("=" * 80)

    official_rows = []
    integrity_rows = []
    missing_artifacts = []

    validation_metrics_path = RUN_ROOT / "validation_file_metrics.csv"
    expected_validation = int(CFG.recipe_counts["validation"])
    for view in ("clean", "stress"):
        prediction_path = RUN_ROOT / f"validation_file_{view}_predictions.csv"
        audit = _prediction_integrity(prediction_path, expected_validation, f"{view}_validation")
        integrity_rows.append(audit)

    if validation_metrics_path.is_file():
        validation_metrics = pd.read_csv(validation_metrics_path)
        for view in ("clean", "stress"):
            matched = validation_metrics[validation_metrics["view"].astype(str).str.lower().eq(view)]
            if matched.empty:
                official_rows.append({"scope": f"{view}_validation", "status": "metric_row_missing", "source": str(validation_metrics_path)})
                continue
            row = matched.iloc[-1].to_dict()
            audit = next(item for item in integrity_rows if item["scope"] == f"{view}_validation")
            official_rows.append({
                "scope": f"{view}_validation",
                **{name: _as_float(row.get(name)) for name in (
                    "file_eer", "voice_eer", "music_eer", "ads",
                    "voice_presence_auc", "music_presence_auc", "cps", "score",
                )},
                "samples": _as_int(row.get("samples"), audit["unique_ids"]),
                "skipped": _as_int(row.get("skipped"), audit["missing_ids"]),
                "invalid": _as_int(row.get("invalid"), audit["invalid_probability_rows"]),
                "status": "ok" if audit["status"] == "ok" else f"metrics_exist; predictions_{audit['status']}",
                "source": str(validation_metrics_path),
            })
    else:
        missing_artifacts.append(str(validation_metrics_path))
        for view in ("clean", "stress"):
            official_rows.append({"scope": f"{view}_validation", "status": "not_evaluated", "source": str(validation_metrics_path)})

    saved_ood_project = Path(globals().get("OOD_PROJECT", RUN_ROOT / "ood2500"))
    saved_ood_data = Path(globals().get("OOD_DATA", RUN_ROOT / "ood_dataset_2500"))
    ood_metrics_path = saved_ood_project / "official_metrics.json"
    ood_prediction_candidates = sorted(
        saved_ood_project.glob("predictions_*.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    ood_prediction_path = ood_prediction_candidates[0] if ood_prediction_candidates else saved_ood_project / "predictions_NOT_FOUND.csv"
    ood_expected = 2_500
    ood_truth_path = saved_ood_data / "ground_truth.csv"
    if ood_truth_path.is_file():
        try:
            ood_expected = len(pd.read_csv(ood_truth_path, usecols=["ID"]))
        except Exception:
            pass
    ood_audit = _prediction_integrity(ood_prediction_path, ood_expected, "ood_2500")
    integrity_rows.append(ood_audit)
    if ood_metrics_path.is_file():
        try:
            ood_metrics = json.loads(ood_metrics_path.read_text(encoding="utf-8"))
            official_rows.append({
                "scope": "ood_2500",
                **{name: _as_float(ood_metrics.get(name)) for name in (
                    "file_eer", "voice_eer", "music_eer", "ads",
                    "voice_presence_auc", "music_presence_auc", "cps", "score",
                )},
                "samples": _as_int(ood_metrics.get("samples"), ood_audit["unique_ids"]),
                "skipped": _as_int(ood_metrics.get("skipped"), ood_audit["missing_ids"]),
                "invalid": _as_int(ood_metrics.get("invalid"), ood_audit["invalid_probability_rows"]),
                "status": "ok" if ood_audit["status"] == "ok" else f"metrics_exist; predictions_{ood_audit['status']}",
                "source": str(ood_metrics_path),
            })
        except Exception as exc:
            official_rows.append({"scope": "ood_2500", "status": f"metric_read_error: {type(exc).__name__}", "source": str(ood_metrics_path)})
    else:
        missing_artifacts.append(str(ood_metrics_path))
        official_rows.append({"scope": "ood_2500", "status": "not_evaluated", "source": str(ood_metrics_path)})

    official_report = pd.DataFrame(official_rows).reindex(columns=OFFICIAL_REPORT_COLUMNS)
    print("\n[1-3] Ensemble 공식 지표")
    display(official_report)
    _atomic_report_csv(official_report, RESULT_REPORT_ROOT / "official_metrics_clean_stress_ood.csv")

    standalone_rows = []
    history_index_rows = []
    invalid_rows = []
    all_histories = []
    task_lookup = {
        "presence": "presence",
        "aasist_aux3": "aasist",
        "voice_wavlm": "voice",
        "music_long": "music",
    }
    primary_lookup = {
        "presence": ("clean_cps", "stress_cps"),
        "aasist_aux3": ("clean_ads", "stress_ads"),
        "voice_wavlm": ("clean_voice_quality", "stress_voice_quality"),
        "music_long": ("clean_music_quality", "stress_music_quality"),
    }
    detail_columns = [
        "clean_file_eer", "clean_voice_eer", "clean_music_eer", "clean_ads",
        "clean_voice_presence_auc", "clean_music_presence_auc", "clean_cps",
        "clean_voice_auc", "clean_music_auc",
        "stress_file_eer", "stress_voice_eer", "stress_music_eer", "stress_ads",
        "stress_voice_presence_auc", "stress_music_presence_auc", "stress_cps",
        "stress_voice_auc", "stress_music_auc",
    ]

    for branch in BRANCHES_TO_TRAIN:
        history_path = RUN_ROOT / branch / "history.csv"
        if not history_path.is_file():
            missing_artifacts.append(str(history_path))
            history_index_rows.append({"branch": branch, "status": "missing", "rows": 0, "path": str(history_path)})
            standalone_rows.append({"branch": branch, "task": task_lookup[branch], "status": "not_trained"})
            invalid_rows.append({
                "scope": branch,
                "train_skipped_total": np.nan,
                "clean_skipped_at_best": np.nan,
                "stress_skipped_at_best": np.nan,
                "status": "history_missing",
            })
            continue

        history = pd.read_csv(history_path)
        if history.empty or "robust_score" not in history.columns:
            history_index_rows.append({"branch": branch, "status": "invalid_history", "rows": len(history), "path": str(history_path)})
            standalone_rows.append({"branch": branch, "task": task_lookup[branch], "status": "invalid_history"})
            continue

        robust = pd.to_numeric(history["robust_score"], errors="coerce")
        if not robust.notna().any():
            history_index_rows.append({"branch": branch, "status": "robust_score_missing", "rows": len(history), "path": str(history_path)})
            continue
        best_position = robust.idxmax()
        best = history.loc[best_position]
        clean_primary, stress_primary = primary_lookup[branch]
        standalone_row = {
            "branch": branch,
            "task": task_lookup[branch],
            "best_epoch": _as_int(best["epoch"]),
            "robust_score": _as_float(best.get("robust_score")),
            "clean_primary_metric": clean_primary.removeprefix("clean_"),
            "clean_primary_value": _as_float(best.get(clean_primary)),
            "stress_primary_metric": stress_primary.removeprefix("stress_"),
            "stress_primary_value": _as_float(best.get(stress_primary)),
            "status": "ok",
        }
        standalone_row.update({column: _as_float(best.get(column)) for column in detail_columns})
        standalone_rows.append(standalone_row)
        history_index_rows.append({
            "branch": branch,
            "status": "ok",
            "rows": len(history),
            "first_epoch": _as_int(history["epoch"].min()),
            "last_epoch": _as_int(history["epoch"].max()),
            "best_epoch": _as_int(best["epoch"]),
            "path": str(history_path),
        })

        train_skipped_recorded = "train_skipped" in history.columns
        invalid_rows.append({
            "scope": branch,
            "train_skipped_total": _as_int(pd.to_numeric(history["train_skipped"], errors="coerce").fillna(0).sum()) if train_skipped_recorded else np.nan,
            "clean_skipped_at_best": _as_int(best.get("clean_skipped"), 0),
            "stress_skipped_at_best": _as_int(best.get("stress_skipped"), 0),
            "status": "ok" if train_skipped_recorded else "train_skipped_not_recorded_in_old_history",
        })
        tagged = history.copy()
        tagged.insert(0, "branch", branch)
        all_histories.append(tagged)

    standalone_report = pd.DataFrame(standalone_rows)
    history_index = pd.DataFrame(history_index_rows)
    invalid_report = pd.DataFrame(invalid_rows)
    integrity_report = pd.DataFrame(integrity_rows)

    print("\n[4] 모델별 단독 성능 (각 모델이 담당하는 head 기준)")
    display(standalone_report)
    print("Presence는 CPS, AASIST는 ADS, Voice/Music specialist는 해당 EER quality가 primary metric입니다.")
    print("전 공식 Score는 5개 출력을 모두 만드는 ensemble에만 정의됩니다.")

    print("\n[5] history.csv 색인")
    display(history_index)
    for branch, tagged in ((frame.iloc[0]["branch"], frame) for frame in all_histories):
        print(f"\n--- {branch} history.csv ---")
        display(tagged.reset_index(drop=True))

    print("\n[6] skipped/invalid 요약")
    display(invalid_report)
    display(integrity_report)
    print("기존 history에 train_skipped 컬럼이 없으면 NaN으로 표시됩니다. 수정본으로 이어 학습한 epoch부터 기록됩니다.")

    _atomic_report_csv(standalone_report, RESULT_REPORT_ROOT / "standalone_branch_best_metrics.csv")
    _atomic_report_csv(history_index, RESULT_REPORT_ROOT / "history_index.csv")
    _atomic_report_csv(invalid_report, RESULT_REPORT_ROOT / "loader_skipped_invalid.csv")
    _atomic_report_csv(integrity_report, RESULT_REPORT_ROOT / "prediction_integrity.csv")
    if all_histories:
        _atomic_report_csv(pd.concat(all_histories, ignore_index=True, sort=False), RESULT_REPORT_ROOT / "history_all_models.csv")

    print("\n보고서 CSV 저장 위치:", RESULT_REPORT_ROOT)
    if missing_artifacts:
        print("\n아직 생성되지 않은 결과 파일:")
        for path in dict.fromkeys(missing_artifacts):
            print(" -", path)
        print("\n실제 학습: RUN_TRAINING=True")
        print("checkpoint 완료 후 Clean/Stress 평가만: RUN_FINAL_VALIDATION=True")
        print("OOD 생성·평가: RUN_OOD=True")
else:
    print("SHOW_SAVED_RESULTS=False: 저장 결과 대시보드를 생략합니다.")


DeepVoice 저장 결과 종합 보고
PROJECT_ROOT: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4
RUN_ROOT: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1

[1-3] Ensemble 공식 지표


,scope,file_eer,voice_eer,music_eer,ads,voice_presence_auc,music_presence_auc,cps,score,samples,skipped,invalid,status,source
0,clean_validation,0.045968,0.058199,0.009605,0.962494,0.987097,0.999682,0.993389,0.965584,2500,0,0,ok,C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headsp...
1,stress_validation,0.084364,0.104502,0.087513,0.910663,0.957141,0.998279,0.977710,0.917368,2500,0,0,ok,C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headsp...
2,ood_2500,0.275171,0.223145,0.413020,0.693880,0.915983,0.993176,0.954580,0.719950,2500,0,0,ok,C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headsp...



[4] 모델별 단독 성능 (각 모델이 담당하는 head 기준)


,branch,task,best_epoch,robust_score,clean_primary_metric,clean_primary_value,stress_primary_metric,stress_primary_value,status,clean_file_eer,...,clean_music_auc,stress_file_eer,stress_voice_eer,stress_music_eer,stress_ads,stress_voice_presence_auc,stress_music_presence_auc,stress_cps,stress_voice_auc,stress_music_auc
0,presence,presence,9,0.982025,cps,0.991578,cps,0.972472,ok,NaN,...,NaN,NaN,NaN,NaN,NaN,0.947372,0.997572,0.972472,NaN,NaN
1,aasist_aux3,aasist,13,0.911752,ads,0.952111,ads,0.871393,ok,0.052368,...,NaN,0.118387,0.113344,0.155816,0.871393,NaN,NaN,NaN,NaN,NaN
2,voice_wavlm,voice,8,0.789992,voice_quality,0.816241,voice_quality,0.763743,ok,NaN,...,NaN,NaN,0.236257,NaN,NaN,NaN,NaN,NaN,0.858679,NaN
3,music_long,music,12,0.952508,music_quality,0.990395,music_quality,0.914621,ok,NaN,...,0.999673,NaN,NaN,0.085379,NaN,NaN,NaN,NaN,NaN,0.976172


Presence는 CPS, AASIST는 ADS, Voice/Music specialist는 해당 EER quality가 primary metric입니다.
전 공식 Score는 5개 출력을 모두 만드는 ensemble에만 정의됩니다.

[5] history.csv 색인


...

,branch,epoch,train_loss,train_samples,train_skipped,train_optimizer_updates,robust_score,clean_music_eer,clean_music_auc,clean_music_quality,...,clean_skipped,clean_optimizer_updates,stress_music_eer,stress_music_auc,stress_music_quality,stress_loss,stress_samples,stress_skipped,stress_optimizer_updates,minutes
0,music_long,1,0.404399,22500,0,1317,0.861793,0.019210,0.996541,0.980790,...,0,0,0.257204,0.830382,0.742796,0.498300,2500,0,0,67.613492
1,music_long,2,0.247334,22500,0,1322,0.893276,0.008538,0.998232,0.991462,...,0,0,0.204909,0.889308,0.795091,0.397850,2500,0,0,31.753487
2,music_long,3,0.209903,22500,0,1319,0.910886,0.009605,0.998585,0.990395,...,0,0,0.168623,0.923760,0.831377,0.381652,2500,0,0,31.917518
3,music_long,4,0.182502,22500,0,1316,0.915688,0.010672,0.998632,0.989328,...,0,0,0.157951,0.928725,0.842049,0.329323,2500,0,0,32.398144
4,music_long,5,0.168937,22500,0,1319,0.923693,0.008538,0.998883,0.991462,...,0,0,0.144077,0.943639,0.855923,0.295989,2500,0,0,32.914056
5,music_long,6,0.144464,22500,0,1318,0.932764,0.005336,0.999885,0.994664,...,0,0,0.129136,0.954985,0.870864,0.273848,2500,0,0,31.499736
6,music_long,7,0.137416,22500,0,1319,0.941302,0.008538,0.999236,0.991462,...,0,0,0.108858,0.962483,0.891142,0.243778,2500,0,0,31.733128
7,music_long,8,0.125380,22500,0,1317,0.946638,0.007471,0.999738,0.992529,...,0,0,0.099253,0.965203,0.900747,0.239066,2500,0,0,33.300579
8,music_long,9,0.108198,22500,0,1322,0.946105,0.010672,0.999533,0.989328,...,0,0,0.097118,0.969247,0.902882,0.216721,2500,0,0,35.014342
9,music_long,10,0.098450,22500,0,1321,0.945037,0.016009,0.999465,0.983991,...,0,0,0.093917,0.974315,0.906083,0.206102,2500,0,0,46.549439



[6] skipped/invalid 요약


,scope,train_skipped_total,clean_skipped_at_best,stress_skipped_at_best,status
0,presence,0,0,0,ok
1,aasist_aux3,0,0,0,ok
2,voice_wavlm,0,0,0,ok
3,music_long,0,0,0,ok


,scope,prediction_path,expected_rows,rows,unique_ids,missing_ids,duplicate_ids,invalid_probability_rows,status
0,clean_validation,C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headsp...,2500,2500,2500,0,0,0,ok
1,stress_validation,C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headsp...,2500,2500,2500,0,0,0,ok
2,ood_2500,C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headsp...,2500,2500,2500,0,0,0,ok


기존 history에 train_skipped 컬럼이 없으면 NaN으로 표시됩니다. 수정본으로 이어 학습한 epoch부터 기록됩니다.

보고서 CSV 저장 위치: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1\result_reports


In [37]:
for name in ["presence", "aasist_aux3", "voice_wavlm", "music_long"]:
    run_dir = RUN_ROOT / name
    print(
        name,
        "best.pt:", (run_dir / "best.pt").exists(),
        "history.csv:", (run_dir / "history.csv").exists(),
        "경로:", run_dir,
    )

presence best.pt: True history.csv: True 경로: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1\presence
aasist_aux3 best.pt: True history.csv: True 경로: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1\aasist_aux3
voice_wavlm best.pt: True history.csv: True 경로: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1\voice_wavlm
music_long best.pt: True history.csv: True 경로: C:\Users\shj04\OneDrive\바탕 화면\deepvoice_headspecialist_v4\runs_fixed_v4_1\music_long
